# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

Colab의 GPU 환경에서 개발되었습니다.
- 런타임 - 런타임 유형 변경 - GPU로 변경(T4 GPU 등)



# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작

In [ ]:
!pip -q install git+https://github.com/huggingface/transformers accelerate
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" datasets pillow pandas --upgrade

In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

본 베이스라인은 colab에서 구글 드라이브를 마운트하여 사용합니다.

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

#### 실습 참고 내용

    챕터 2-2 합성 데이터 실습
    - 구글 드라이브 마운트 : drive()

In [ ]:
# 구글드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!7z x -y -p"PASSWORD" -o/content/ "/content/drive/MyDrive/ssafy-16-2-ai.zip"

# 라이브러리, 데이터, 설정

In [3]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 사전 학습 모델 정의
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
IMAGE_SIZE = 384
MAX_NEW_TOKENS = 8
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 데이터셋 로드
train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")

# 학습데이터 200개만 추출
train_df = train_df.sample(n=200, random_state=SEED).reset_index(drop=True)

Device: cuda


FileNotFoundError: [Errno 2] No such file or directory: '/content/train.csv'

# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [ ]:
# 양자화
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# 프로세서
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE*IMAGE_SIZE,
    max_pixels=IMAGE_SIZE*IMAGE_SIZE,
    trust_remote_code=True,
)

# 사전학습 모델
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# 양자화 모델로 로드
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# LoRA 세팅
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [ ]:
# 모델 지시사항
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

# 프롬프트
def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [ ]:
# 커스텀 데이터셋
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}

# 데이터 콜레이터
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]

            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            texts.append(text)
            images.append(img)

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        if self.train:
            enc["labels"] = enc["input_ids"].clone()

        return enc


# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [ ]:
# 검증용 데이터 분리
split = int(len(train_df)*0.9)
train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

# VQAMCDataset 형태로 변환
train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

# 데이터로더
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=DataCollator(processor, True), num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=1, shuffle=False, collate_fn=DataCollator(processor, True), num_workers=0)

# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
from tqdm.auto import tqdm

model = model.to(device)
GRAD_ACCUM = 4

# 옵티마이저, 학습 스케줄러
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
num_training_steps = 1 * math.ceil(len(train_loader)/GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(optimizer, int(num_training_steps*0.03), num_training_steps)

# 스케일러
scaler = torch.cuda.amp.GradScaler(enabled=True)

# 학습 루프
global_step = 0
for epoch in range(1):
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k:v.to(device) for k,v in batch.items()}
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.3f}"})
            running = 0.0

    model.eval()
    val_loss = 0.0
    val_steps = 0
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k:v.to(device) for k,v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    print(f"[Epoch {epoch+1}] valid loss {val_loss/val_steps:.4f}")
    model.train()

# 모델 저장
SAVE_DIR = "/content/qwen2_5_vl_3b_lora"
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Saved:", SAVE_DIR)


# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
# 데이터 파서 : 모델의 응답에서 선지를 추출
def extract_choice(text: str) -> str:
    text = text.strip().lower()

    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last

    tokens = last.split()
    for tok in tokens:
        if tok in ["a", "b", "c", "d"]:
            return tok
    return "a"

# 추론을 위해 모든 레이어 활성화
model.eval()
preds = []

# 추론 루프
for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    img = Image.open(row["path"]).convert("RGB")
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[
            {"type":"image","image":img},
            {"type":"text","text":user_text}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=2, do_sample=False,
                                 eos_token_id=processor.tokenizer.eos_token_id)
    output_text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    # print("output_text:", output_text)
    # print("extract_choice:", extract_choice(output_text))
    preds.append(extract_choice(output_text))

# 제출 파일 생성
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission.csv", index=False)
print("Saved /content/submission.csv")

In [ ]:
# 모델 응답 예시
print(output_text)

# 🚀 추가/교체 영역 — Qwen3.8 + Hybrid VQA

> **기존 baseline 셀은 삭제/수정하지 않습니다.**  
> 아래 셀들은 기존 노트북 뒤에 추가된 **갈아끼우기용 셀**입니다.

이번 통합본에서 가져온 장점:
- `(260325)_baseline_desktop_final.ipynb`: **고해상도(1024), ABCD 로짓 추론, 4회 선지 TTA, dev/test `.npy`, 스마트 라우팅**
- `COLAB_vqa(1).ipynb`: **Google Drive 영구 저장, VM 로컬 데이터 복사, 고정 검증셋, 중간 체크포인트, 확률 파일 기반 앙상블**
- `TEAM_MODEL_TRAINING.ipynb`: **실행별 RUN_DIR, 설정/환경 manifest, assistant-only loss, 재현 가능한 split/결과 보존**

### 실행 원칙
1. 기존 baseline의 **환경/데이터 구조는 최대한 유지**
2. 기존 **모델 로드 셀(10), 학습 셀(18), inference 셀(20)** 대신 아래 대응 셀 실행
3. OCR/Grounding DINO는 **한 번 전처리 후 JSON 캐시**하고 GPU에서 해제
4. LoRA는 **중간 checkpoint를 Drive에 저장**
5. 검증/test 확률은 반드시 `.npy`로 저장하여 앙상블 가능하게 유지

In [4]:
# ============================================================
# [LEAVE-1] 현재 환경/파이프라인 상태 Drive 저장
# ============================================================

import json
import subprocess
from pathlib import Path
from datetime import datetime

STATE_DIR = (
    Path("/content/drive/MyDrive/ssafy_vqa_qwen38")
    / "session_state"
)

STATE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

state = {
    "saved_at": datetime.now().isoformat(),

    "base_model":
        "btbtyler09/Qwen3.8-27B-GPTQ-4bit",

    "loader":
        "GPTQModel.load",

    "backend":
        "AUTO_TRAINABLE -> TritonV2Linear",

    "image_size":
        1024,

    "lora": {
        "r": 16,
        "alpha": 32,
        "dropout": 0.05,
        "modules": 400,
        "trainable_params": 108_789_760,
    },

    "gradient_checkpointing": {
        "decoder": "64/64",
        "vision": "27/27",
    },

    "important_patch":
        "QuantLinearFunction.backward must return 8 gradients",

    "smoke_test": {
        "status": "SUCCESS",
        "samples": 32,
        "optimizer_steps": 2,
        "peak_vram_gb": 36.57,
        "overall_avg_loss": 1.23755,
        "checkpoint":
            "/content/drive/MyDrive/ssafy_vqa_qwen38/"
            "runs/20260921_080822/h3_smoke32/"
            "checkpoint-step-002",
    },

    "next_step":
        "H-4 checkpoint reload validation, then fresh full training",
}

state_path = (
    STATE_DIR
    / "latest_state.json"
)

state_path.write_text(
    json.dumps(
        state,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("✅ saved:", state_path)

✅ saved: /content/drive/MyDrive/ssafy_vqa_qwen38/session_state/latest_state.json


In [5]:
# ===== [RESUME-0] 세션 재시작 후 기본 환경 복구 =====

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime
import json
import torch

PROJECT_NAME = "ssafy_vqa_qwen38"

DRIVE_ROOT = Path("/content/drive/MyDrive") / PROJECT_NAME
DATA_ROOT   = Path("/content/data")
CACHE_ROOT  = DRIVE_ROOT / "cache"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = DRIVE_ROOT / "runs" / RUN_ID

for p in [
    RUN_DIR,
    RUN_DIR / "checkpoints",
    RUN_DIR / "probs",
]:
    p.mkdir(parents=True, exist_ok=True)

print("RUN_DIR:", RUN_DIR)

# 설치 유지 여부 확인
import transformers
import accelerate
import peft
import gptqmodel
import optimum

print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate  :", accelerate.__version__)
print("peft        :", peft.__version__)
print("gptqmodel   :", gptqmodel.__version__)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print(
    "VRAM:",
    round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        2
    ),
    "GB"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RUN_DIR: /content/drive/MyDrive/ssafy_vqa_qwen38/runs/20260922_023634

INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └──────────

In [6]:
# ===== [RESUME-1] 데이터 존재 확인 =====

required = [
    DATA_ROOT / "train.csv",
    DATA_ROOT / "dev.csv",
    DATA_ROOT / "test.csv",
    DATA_ROOT / "sample_submission.csv",
]

for p in required:
    print(p, "->", p.exists())

print(
    "train folder:",
    (DATA_ROOT / "train").exists()
)
print(
    "dev folder  :",
    (DATA_ROOT / "dev").exists()
)
print(
    "test folder :",
    (DATA_ROOT / "test").exists()
)

/content/data/train.csv -> False
/content/data/dev.csv -> False
/content/data/test.csv -> False
/content/data/sample_submission.csv -> False
train folder: False
dev folder  : False
test folder : False


## [교체 A] Colab 저장소 / 데이터 준비


### ✅ ZIP 내부 권장 구조

압축파일 루트에 아래 항목들이 바로 오도록 만드세요.

```text
ssafy-16-2-ai.zip
├── train.csv
├── dev.csv
├── test.csv
├── sample_submission.csv
├── train/
│   ├── train_0001.jpg
│   └── ...
├── dev/
│   ├── dev_0001.jpg
│   └── ...
└── test/
    ├── test_0001.jpg
    └── ...
```

**CSV 4개만 압축하면 안 됩니다.** CSV의 `path`가 실제 이미지 파일을 가리키므로 `train/`, `dev/`, `test/` 이미지 폴더도 같이 들어 있어야 합니다.

`dev.csv`의 `answer1~answer5`는 이 노트북에서 정답으로 사용하지 않습니다. 의미가 확정된 ground truth인지 파일 자체만으로는 알 수 없기 때문에, dev는 모델 확률/예측 저장 및 향후 앙상블 분석 용도로만 사용합니다.


In [ ]:
# ============================================================
# [A] Colab Bootstrap
# - Drive mount
# - 프로젝트 경로 생성
# - GPTQ/PEFT 환경 확인
# ============================================================

# ------------------------------------------------------------
# 0. 필요한 패키지
# IMPORTANT:
# transformers를 import하기 전에 optimum이 설치되어 있어야 함
# ------------------------------------------------------------

%pip -q install -U \
    "transformers==5.17.0" \
    "accelerate==1.15.0" \
    "peft==0.21.0" \
    "optimum==2.3.0"

%pip -q install -U \
    "gptqmodel==7.5.0" \
    --no-build-isolation


# ------------------------------------------------------------
# 1. Google Drive
# ------------------------------------------------------------

from google.colab import drive
drive.mount("/content/drive")


# ------------------------------------------------------------
# 2. 기본 경로
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime

import os
import json
import shutil
import subprocess


PROJECT_NAME = "ssafy_vqa_qwen38"

DRIVE_ROOT = (
    Path("/content/drive/MyDrive")
    / PROJECT_NAME
)

DATA_ROOT = Path("/content/data")

RUN_ID = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

RUN_DIR = (
    DRIVE_ROOT
    / "runs"
    / RUN_ID
)

# OCR / DINO 결과는 실험 run이 바뀌어도 재사용
CACHE_ROOT = (
    DRIVE_ROOT
    / "cache"
)


for p in [
    DRIVE_ROOT,
    RUN_DIR,
    RUN_DIR / "checkpoints",
    RUN_DIR / "probs",
    CACHE_ROOT,
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )


print(
    "DRIVE_ROOT:",
    DRIVE_ROOT
)

print(
    "RUN_DIR   :",
    RUN_DIR
)

print(
    "CACHE_ROOT:",
    CACHE_ROOT
)


# ------------------------------------------------------------
# 3. 환경 확인
# ------------------------------------------------------------

import torch
import transformers
import accelerate
import peft
import gptqmodel
import optimum

from transformers.utils import (
    is_optimum_available
)

from optimum.gptq import (
    GPTQQuantizer
)


print(
    "\n===== ENV ====="
)

print(
    "torch       :",
    torch.__version__
)

print(
    "transformers:",
    transformers.__version__
)

print(
    "accelerate  :",
    accelerate.__version__
)

print(
    "peft        :",
    peft.__version__
)

print(
    "gptqmodel   :",
    gptqmodel.__version__
)

print(
    "optimum     :",
    getattr(
        optimum,
        "__version__",
        "installed"
    )
)

print(
    "HF sees optimum:",
    is_optimum_available()
)

print(
    "GPTQQuantizer:",
    "OK"
)


if torch.cuda.is_available():

    props = torch.cuda.get_device_properties(
        0
    )

    print(
        "GPU         :",
        props.name
    )

    print(
        "VRAM        :",
        round(
            props.total_memory
            / 1024**3,
            2
        ),
        "GB"
    )

else:

    raise RuntimeError(
        "GPU 런타임을 켜세요."
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_ROOT: /content/drive/MyDrive/ssafy_vqa_qwen38
RUN_DIR   : /content/drive/MyDrive/ssafy_vqa_qwen38/runs/20260921_094759
CACHE_ROOT: /content/drive/MyDrive/ssafy_vqa_qwen38/cache

===== ENV =====
torch       : 2.11.0+cu128
transformers: 5.17.0
accelerate  : 1.15.0
peft        : 0.21.0
gptqmodel   : 7.5.0
optimum     : installed
HF sees optimum: True
GPTQQuantizer: OK
GPU         : NVIDIA A100-SXM4-80GB
VRAM        : 79.25 GB


In [ ]:
# 2) GPTQ 의존성 확인
import importlib.util
import importlib.metadata

print("optimum spec:", importlib.util.find_spec("optimum"))
print("optimum version:", importlib.metadata.version("optimum"))

from transformers.utils import is_optimum_available
print("transformers sees optimum:", is_optimum_available())

from optimum.gptq import GPTQQuantizer
print("✅ optimum.gptq.GPTQQuantizer import OK")

optimum spec: ModuleSpec(name='optimum', loader=<_frozen_importlib_external.NamespaceLoader object at 0x7e8b3d7ff2f0>, submodule_search_locations=_NamespacePath(['/usr/local/lib/python3.13/dist-packages/optimum']))
optimum version: 2.3.0
transformers sees optimum: True
✅ optimum.gptq.GPTQQuantizer import OK


In [ ]:
# 권장: Drive의 zip을 Colab VM 로컬로 복사 후 압축해제
ZIP_NAME = "data.zip"
ZIP_PATH = Path("/content/drive/MyDrive") / ZIP_NAME

DATA_ROOT.mkdir(parents=True, exist_ok=True)

if ZIP_PATH.exists():
    local_zip = Path("/content") / ZIP_NAME
    if not local_zip.exists():
        shutil.copy2(ZIP_PATH, local_zip)

    PASSWORD = None  # 비밀번호가 있으면 문자열로 입력, 없으면 None
    cmd = ["7z", "x", "-y", f"-o{DATA_ROOT}", str(local_zip)]
    if PASSWORD:
        cmd.insert(3, f"-p{PASSWORD}")
    subprocess.run(cmd, check=True)
else:
    print("Drive에 zip이 없습니다. 다음 수동 업로드 셀을 사용하세요.")

print("DATA_ROOT contents:", list(DATA_ROOT.glob("*"))[:20])

DATA_ROOT contents: [PosixPath('/content/data/train'), PosixPath('/content/data/dev.csv'), PosixPath('/content/data/train.csv'), PosixPath('/content/data/sample_submission.csv'), PosixPath('/content/data/dev'), PosixPath('/content/data/test'), PosixPath('/content/data/test.csv')]


In [ ]:
# ===== [RESUME-1] 데이터 존재 확인 =====

required = [
    DATA_ROOT / "train.csv",
    DATA_ROOT / "dev.csv",
    DATA_ROOT / "test.csv",
    DATA_ROOT / "sample_submission.csv",
]

for p in required:
    print(p, "->", p.exists())

print(
    "train folder:",
    (DATA_ROOT / "train").exists()
)
print(
    "dev folder  :",
    (DATA_ROOT / "dev").exists()
)
print(
    "test folder :",
    (DATA_ROOT / "test").exists()
)

/content/data/train.csv -> True
/content/data/dev.csv -> True
/content/data/test.csv -> True
/content/data/sample_submission.csv -> True
train folder: True
dev folder  : True
test folder : True


In [ ]:
# (선택) PC → Colab 직접 업로드
# Drive zip을 쓰는 경우 실행하지 마세요.
#
# from google.colab import files
# uploaded = files.upload()
# for name, data in uploaded.items():
#     dst = Path("/content") / name
#     dst.write_bytes(data)
#     print("uploaded:", dst)
#
# !7z x -y -p"PASSWORD" -o/content/data "/content/data.zip"

## [교체 B] 라이브러리 설치

In [ ]:
# ===== [교체 B v2] 충돌 없는 최신 조합 =====

# Colab 기본 torch는 그대로 둠
# 현재 확인된 환경:
# torch 2.11.0+cu128
# A100 80GB

!pip -q install -U \
  "transformers==5.17.0" \
  "accelerate==1.15.0" \
  "peft==0.21.0" \
  albumentations \
  opencv-python-headless \
  easyocr \
  scikit-learn \
  safetensors

# GPTQModel은 HF 공식 문서 권장대로 따로 설치
!pip -q install -U "gptqmodel==7.5.0" --no-build-isolation

print("✅ 패키지 설치 완료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 150.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 125.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 150.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 32.2 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 601.5 kB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━

## [교체 C] 공통 설정 + 실행 manifest

In [ ]:
import os, re, gc, math, json, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

from transformers import (
    AutoProcessor,
    Qwen3_5ForConditionalGeneration,
    get_cosine_schedule_with_warmup,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

Image.MAX_IMAGE_PIXELS = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "GPU 런타임을 켜세요."
device = "cuda"
props = torch.cuda.get_device_properties(0)

# MODEL_ID = "navispace/Qwen3.8-27B-W8A16-GPTQ"
MODEL_ID = "btbtyler09/Qwen3.8-27B-GPTQ-8bit"
# A/B 후보:
# MODEL_ID = "TelperionAI/Qwen3.8-27B-INT4-AWQ-GPTQ"
# MODEL_ID = "bernhardbrieger/Qwen3.8-27B-GPTQ-Int4"

IMAGE_SIZE = 1024
MAX_LENGTH = 4096
TRAIN_BATCH = 1
GRAD_ACCUM = 16
EPOCHS = 1
LR = 5e-5
WARMUP_RATIO = 0.03
SAVE_EVERY_OPT_STEPS = 50

def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError(paths)

TRAIN_CSV = first_existing([DATA_ROOT/"train.csv", Path("/content/train.csv")])
DEV_CSV   = first_existing([DATA_ROOT/"dev.csv", Path("/content/dev.csv")])
TEST_CSV  = first_existing([DATA_ROOT/"test.csv", Path("/content/test.csv")])
SAMPLE_CSV = first_existing([DATA_ROOT/"sample_submission.csv", Path("/content/sample_submission.csv")])

train_df = pd.read_csv(TRAIN_CSV)
dev_df   = pd.read_csv(DEV_CSV)
test_df  = pd.read_csv(TEST_CSV)
sample_submission = pd.read_csv(SAMPLE_CSV)

def normalize_image_path(csv_path, p):
    p = Path(str(p))
    if p.is_absolute() and p.exists():
        return str(p)
    candidates = [csv_path.parent/p, DATA_ROOT/p, Path("/content")/p]
    for c in candidates:
        if c.exists():
            return str(c)
    return str(candidates[0])

train_df["path"] = [normalize_image_path(TRAIN_CSV, p) for p in train_df["path"]]
dev_df["path"]   = [normalize_image_path(DEV_CSV, p) for p in dev_df["path"]]
test_df["path"]  = [normalize_image_path(TEST_CSV, p) for p in test_df["path"]]

CONFIG = {
    "run_id": RUN_ID,
    "model_id": MODEL_ID,
    "image_size": IMAGE_SIZE,
    "seed": SEED,
    "train_batch": TRAIN_BATCH,
    "grad_accum": GRAD_ACCUM,
    "epochs": EPOCHS,
    "lr": LR,
    "save_every_opt_steps": SAVE_EVERY_OPT_STEPS,
    "gpu": props.name,
    "gpu_vram_gb": round(props.total_memory/1024**3, 2),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
}
(RUN_DIR/"config.json").write_text(json.dumps(CONFIG, ensure_ascii=False, indent=2), encoding="utf-8")

print(json.dumps(CONFIG, ensure_ascii=False, indent=2))
assert test_df["id"].tolist() == sample_submission["id"].tolist(), "sample_submission과 test ID/순서가 다릅니다."
print("train:", len(train_df), "dev:", len(dev_df), "test:", len(test_df))
print("sample_submission:", sample_submission.shape)

{
  "run_id": "20260921_071745",
  "model_id": "navispace/Qwen3.8-27B-W8A16-GPTQ",
  "image_size": 1024,
  "seed": 42,
  "train_batch": 1,
  "grad_accum": 16,
  "epochs": 1,
  "lr": 5e-05,
  "save_every_opt_steps": 50,
  "gpu": "NVIDIA A100-SXM4-80GB",
  "gpu_vram_gb": 79.25,
  "torch": "2.11.0+cu128",
  "cuda": "12.8"
}
train: 6714 dev: 2683 test: 6714
sample_submission: (6714, 2)


In [ ]:
# ===== [MODEL PATCH] 실제 사용 모델로 metadata 수정 =====

MODEL_ID = "btbtyler09/Qwen3.8-27B-GPTQ-8bit"

CONFIG["model_id"] = MODEL_ID
CONFIG["loader"] = "GPTQModel.load"
CONFIG["quantization"] = "GPTQ 8bit group_size=32"

(RUN_DIR / "config.json").write_text(
    json.dumps(
        CONFIG,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("MODEL_ID:", MODEL_ID)
print("config updated:", RUN_DIR / "config.json")

MODEL_ID: btbtyler09/Qwen3.8-27B-GPTQ-8bit
config updated: /content/drive/MyDrive/ssafy_vqa_qwen38/runs/20260921_071745/config.json


## [교체 D] 비율 유지 Padding + 학습용 Augmentation

In [ ]:
import albumentations as A

def resize_with_padding(image: Image.Image, target=1024, fill=(127,127,127)):
    image = ImageOps.exif_transpose(image).convert("RGB")
    w, h = image.size
    scale = min(target / w, target / h)
    nw, nh = max(1, round(w*scale)), max(1, round(h*scale))
    resized = image.resize((nw, nh), Image.Resampling.LANCZOS)

    canvas = Image.new("RGB", (target, target), fill)
    x = (target - nw)//2
    y = (target - nh)//2
    canvas.paste(resized, (x, y))
    return canvas

train_aug = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.35),
    A.HueSaturationValue(hue_shift_limit=4, sat_shift_limit=8, val_shift_limit=8, p=0.20),
    A.GaussNoise(std_range=(0.01, 0.04), p=0.15),
    A.ImageCompression(quality_range=(82, 100), p=0.20),
])

def augment_pil(image: Image.Image):
    arr = np.asarray(image)
    aug = train_aug(image=arr)["image"]
    return Image.fromarray(aug)

_tmp = resize_with_padding(Image.open(train_df.iloc[0]["path"]), IMAGE_SIZE)
print("resized:", _tmp.size)

resized: (1024, 1024)


E 진입전 확인 작업

In [ ]:
import torch
import transformers
import accelerate
import peft
import gptqmodel

print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate  :", accelerate.__version__)
print("peft        :", peft.__version__)
print("gptqmodel   :", gptqmodel.__version__)

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")


INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.5.0
Transformers : 5.17.0
Torch        : 2.11.0+cu128
Triton       : 3.6.0
torch       : 2.11.0+cu128
transformers: 5.17.0
accelerate  : 1.15.0
peft        : 0.21.

In [ ]:
# ===== [E-0] OCR / Grounding DINO 환경 smoke test =====

import easyocr
import albumentations as A

from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection
)

print("EasyOCR import OK")
print("Albumentations:", A.__version__)
print("Grounding DINO classes import OK")

SMOKE_PATHS = (
    train_df["path"].head(8).tolist()
    + dev_df["path"].head(6).tolist()
    + test_df["path"].head(6).tolist()
)

print("Smoke images:", len(SMOKE_PATHS))

for p in SMOKE_PATHS[:3]:
    print(p, "->", Path(p).exists())

EasyOCR import OK
Albumentations: 2.0.8
Grounding DINO classes import OK
Smoke images: 20
/content/data/train/train_0001.jpg -> True
/content/data/train/train_0002.jpg -> True
/content/data/train/train_0003.jpg -> True


In [ ]:
# ===== [E-1 SMOKE] EasyOCR 20장 =====

import json
import gc
import torch
from tqdm.auto import tqdm

OCR_CACHE_PATH = CACHE_ROOT / "ocr_cache.json"

if OCR_CACHE_PATH.exists():
    ocr_cache = json.loads(
        OCR_CACHE_PATH.read_text(encoding="utf-8")
    )
else:
    ocr_cache = {}

reader = easyocr.Reader(
    ["ko", "en"],
    gpu=True
)

for path in tqdm(SMOKE_PATHS, desc="EasyOCR smoke"):
    key = str(path)

    if key in ocr_cache:
        continue

    try:
        result = reader.readtext(
            path,
            detail=1,
            paragraph=False
        )

        items = []

        for bbox, text, conf in result:
            if conf >= 0.25 and str(text).strip():

                items.append({
                    "text": str(text).strip(),
                    "conf": float(conf),
                    "bbox": [
                        [float(x), float(y)]
                        for x, y in bbox
                    ]
                })

        ocr_cache[key] = items

    except Exception as e:
        print("OCR ERROR:", path, e)
        ocr_cache[key] = {
            "error": str(e)
        }

OCR_CACHE_PATH.write_text(
    json.dumps(
        ocr_cache,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\nOCR cache:", OCR_CACHE_PATH)

# 하나 출력
for p in SMOKE_PATHS:
    result = ocr_cache.get(str(p))

    if isinstance(result, list) and len(result):
        print("\nExample image:", p)
        print(result[:5])
        break

del reader
gc.collect()
torch.cuda.empty_cache()

print(
    "GPU allocated:",
    round(torch.cuda.memory_allocated()/1024**3, 2),
    "GB"
)

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.1% Complete

EasyOCR smoke:   0%|          | 0/20 [00:00<?, ?it/s]


OCR cache: /content/drive/MyDrive/ssafy_vqa_qwen38/cache/ocr_cache.json

Example image: /content/data/train/train_0001.jpg
[{'text': 'SUNDAY', 'conf': 0.6371622014160425, 'bbox': [[315.0, 382.0], [390.0, 382.0], [390.0, 410.0], [315.0, 410.0]]}, {'text': 'BURGER', 'conf': 0.9890829738460919, 'bbox': [[318.0, 408.0], [392.0, 408.0], [392.0, 434.0], [318.0, 434.0]]}, {'text': 'LoVE', 'conf': 0.45527955889701843, 'bbox': [[328.0, 434.0], [380.0, 434.0], [380.0, 458.0], [328.0, 458.0]]}, {'text': '[DIG', 'conf': 0.5967870950698853, 'bbox': [[167.0, 513.0], [207.0, 513.0], [207.0, 529.0], [167.0, 529.0]]}, {'text': 'Suprem', 'conf': 0.9853723123147459, 'bbox': [[481.03883864861814, 627.6077677297236], [532.6581442576513, 613.8817166634444], [536.9611613513819, 630.3922322702764], [486.3418557423486, 645.1182833365556]]}]
GPU allocated: 0.01 GB


In [ ]:
# ===== [E-2 SMOKE] Grounding DINO 20장 =====

import json
import gc
import torch

from transformers import (
    AutoProcessor as GDProcessor,
    AutoModelForZeroShotObjectDetection
)

DINO_ID = "IDEA-Research/grounding-dino-tiny"

DINO_CACHE_PATH = CACHE_ROOT / "dino_cache.json"

DINO_TEXT = (
    "person . bottle . can . cup . box . bag . "
    "paper . cardboard . plastic . glass . metal . "
    "container . food . phone . table . chair . "
    "sign . carton . package . bin . trash . "
    "recycling bin ."
)

if DINO_CACHE_PATH.exists():
    dino_cache = json.loads(
        DINO_CACHE_PATH.read_text(encoding="utf-8")
    )
else:
    dino_cache = {}

gd_proc = GDProcessor.from_pretrained(DINO_ID)

gd_model = (
    AutoModelForZeroShotObjectDetection
    .from_pretrained(DINO_ID)
    .to("cuda")
    .eval()
)

for path in tqdm(SMOKE_PATHS, desc="DINO smoke"):

    key = str(path)

    if key in dino_cache:
        continue

    try:
        image = Image.open(path).convert("RGB")

        inputs = gd_proc(
            images=image,
            text=DINO_TEXT,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            outputs = gd_model(**inputs)

        result = gd_proc.post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            threshold=0.30,
            text_threshold=0.25,
            target_sizes=[image.size[::-1]],
        )[0]

        labels = result.get(
            "text_labels",
            result.get("labels", [])
        )

        items = []

        for box, score, label in zip(
            result["boxes"],
            result["scores"],
            labels
        ):

            items.append({
                "label": str(label),
                "score": float(score),
                "bbox": [
                    float(x)
                    for x in box.tolist()
                ]
            })

        dino_cache[key] = items

    except Exception as e:

        print("DINO ERROR:", path, e)

        dino_cache[key] = {
            "error": str(e)
        }

DINO_CACHE_PATH.write_text(
    json.dumps(
        dino_cache,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\nDINO cache:", DINO_CACHE_PATH)

for p in SMOKE_PATHS:
    result = dino_cache.get(str(p))

    if isinstance(result, list) and len(result):

        print("\nExample image:", p)
        print(result[:5])
        break

del gd_model, gd_proc
gc.collect()
torch.cuda.empty_cache()

print(
    "GPU allocated:",
    round(torch.cuda.memory_allocated()/1024**3, 2),
    "GB"
)

preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/978 [00:00<?, ?it/s]

DINO smoke:   0%|          | 0/20 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/grounding_dino/processing_grounding_dino.py:94: FutureWarning: The key `labels` is will return integer ids in `GroundingDinoProcessor.post_process_grounded_object_detection` output since v4.51.0. Use `text_labels` instead to retrieve string object names.
  warnings.warn(self.message, FutureWarning)



DINO cache: /content/drive/MyDrive/ssafy_vqa_qwen38/cache/dino_cache.json

Example image: /content/data/train/train_0001.jpg
[{'label': 'chair', 'score': 0.794451892375946, 'bbox': [347.78924560546875, 842.7925415039062, 528.021484375, 958.9924926757812]}, {'label': 'table', 'score': 0.5935085415840149, 'bbox': [0.7328224182128906, 857.1162719726562, 296.5901794433594, 958.96826171875]}, {'label': 'box cart', 'score': 0.5051457285881042, 'bbox': [159.4591522216797, 753.8231811523438, 298.9488220214844, 877.9075927734375]}, {'label': 'bottle', 'score': 0.4995631277561188, 'bbox': [97.57927703857422, 342.62445068359375, 122.70732879638672, 410.3464660644531]}, {'label': 'bottle', 'score': 0.4310401976108551, 'bbox': [362.22515869140625, 185.9775848388672, 396.5240478515625, 268.1221923828125]}]
GPU allocated: 0.07 GB


## [교체 E] Local OCR + Grounding DINO 1회 캐시

In [ ]:
# ===== [E-1 FULL] EasyOCR 전체 캐시 =====

import easyocr
import json
import gc
import torch
from tqdm.auto import tqdm

OCR_CACHE_PATH = CACHE_ROOT / "ocr_cache.json"

if OCR_CACHE_PATH.exists():
    ocr_cache = json.loads(
        OCR_CACHE_PATH.read_text(encoding="utf-8")
    )
else:
    ocr_cache = {}

all_paths = pd.concat([
    train_df["path"],
    dev_df["path"],
    test_df["path"]
]).drop_duplicates().tolist()

print("전체 이미지:", len(all_paths))
print("이미 OCR 완료:", sum(str(p) in ocr_cache for p in all_paths))

reader = easyocr.Reader(
    ["ko", "en"],
    gpu=True
)

SAVE_EVERY = 250

for i, path in enumerate(tqdm(all_paths, desc="EasyOCR FULL"), start=1):

    key = str(path)

    if key in ocr_cache:
        continue

    try:
        result = reader.readtext(
            path,
            detail=1,
            paragraph=False
        )

        items = []

        for bbox, text, conf in result:

            if conf >= 0.25 and str(text).strip():

                items.append({
                    "text": str(text).strip(),
                    "conf": float(conf),
                    "bbox": [
                        [float(x), float(y)]
                        for x, y in bbox
                    ],
                })

        ocr_cache[key] = items

    except Exception as e:

        ocr_cache[key] = {
            "error": str(e)
        }

    if i % SAVE_EVERY == 0:

        OCR_CACHE_PATH.write_text(
            json.dumps(
                ocr_cache,
                ensure_ascii=False
            ),
            encoding="utf-8"
        )

        print(
            f"\n💾 OCR cache saved: "
            f"{sum(str(p) in ocr_cache for p in all_paths)}/{len(all_paths)}"
        )


# 마지막 저장
OCR_CACHE_PATH.write_text(
    json.dumps(
        ocr_cache,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

done = sum(str(p) in ocr_cache for p in all_paths)

print("\n✅ OCR 완료:", done, "/", len(all_paths))
print("저장:", OCR_CACHE_PATH)


del reader

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

전체 이미지: 16111
이미 OCR 완료: 20


EasyOCR FULL:   0%|          | 0/16111 [00:00<?, ?it/s]


💾 OCR cache saved: 262/16111

💾 OCR cache saved: 512/16111

💾 OCR cache saved: 762/16111

💾 OCR cache saved: 1012/16111

💾 OCR cache saved: 1262/16111

💾 OCR cache saved: 1512/16111

💾 OCR cache saved: 1762/16111

💾 OCR cache saved: 2012/16111

💾 OCR cache saved: 2262/16111

💾 OCR cache saved: 2512/16111

💾 OCR cache saved: 2762/16111

💾 OCR cache saved: 3012/16111

💾 OCR cache saved: 3262/16111

💾 OCR cache saved: 3512/16111

💾 OCR cache saved: 3762/16111

💾 OCR cache saved: 4012/16111

💾 OCR cache saved: 4262/16111

💾 OCR cache saved: 4512/16111

💾 OCR cache saved: 4762/16111

💾 OCR cache saved: 5012/16111

💾 OCR cache saved: 5262/16111

💾 OCR cache saved: 5512/16111

💾 OCR cache saved: 5762/16111

💾 OCR cache saved: 6012/16111

💾 OCR cache saved: 6262/16111

💾 OCR cache saved: 6512/16111

💾 OCR cache saved: 6756/16111

💾 OCR cache saved: 7006/16111

💾 OCR cache saved: 7256/16111

💾 OCR cache saved: 7506/16111

💾 OCR cache saved: 7756/16111

💾 OCR cache saved: 8006/16111

💾 OCR cach

In [ ]:
# ===== [E-2 FULL] Grounding DINO 전체 캐시 =====

import json
import gc
import torch

from transformers import (
    AutoProcessor as GDProcessor,
    AutoModelForZeroShotObjectDetection
)

DINO_ID = "IDEA-Research/grounding-dino-tiny"

DINO_CACHE_PATH = CACHE_ROOT / "dino_cache.json"

DINO_TEXT = (
    "person . "
    "bottle . "
    "can . "
    "cup . "
    "box . "
    "bag . "
    "paper . "
    "cardboard . "
    "plastic . "
    "glass . "
    "metal . "
    "container . "
    "food . "
    "phone . "
    "table . "
    "chair . "
    "sign . "
    "carton . "
    "package . "
    "bin . "
    "trash . "
    "recycling bin ."
)

if DINO_CACHE_PATH.exists():

    dino_cache = json.loads(
        DINO_CACHE_PATH.read_text(encoding="utf-8")
    )

else:

    dino_cache = {}


print("전체 이미지:", len(all_paths))
print(
    "이미 DINO 완료:",
    sum(str(p) in dino_cache for p in all_paths)
)


gd_proc = GDProcessor.from_pretrained(
    DINO_ID
)

gd_model = (
    AutoModelForZeroShotObjectDetection
    .from_pretrained(DINO_ID)
    .to("cuda")
    .eval()
)


SAVE_EVERY = 250


for i, path in enumerate(
    tqdm(all_paths, desc="Grounding DINO FULL"),
    start=1
):

    key = str(path)

    if key in dino_cache:
        continue

    try:

        image = Image.open(path).convert("RGB")

        inputs = gd_proc(
            images=image,
            text=DINO_TEXT,
            return_tensors="pt"
        ).to("cuda")


        with torch.no_grad():

            outputs = gd_model(
                **inputs
            )


        result = gd_proc.post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            threshold=0.30,
            text_threshold=0.25,
            target_sizes=[
                image.size[::-1]
            ],
        )[0]


        labels = result.get(
            "text_labels",
            result.get("labels", [])
        )


        items = []

        for box, score, label in zip(
            result["boxes"],
            result["scores"],
            labels
        ):

            items.append({
                "label": str(label),
                "score": float(score),
                "bbox": [
                    float(x)
                    for x in box.tolist()
                ],
            })


        dino_cache[key] = items


    except Exception as e:

        dino_cache[key] = {
            "error": str(e)
        }


    if i % SAVE_EVERY == 0:

        DINO_CACHE_PATH.write_text(
            json.dumps(
                dino_cache,
                ensure_ascii=False
            ),
            encoding="utf-8"
        )

        print(
            f"\n💾 DINO cache saved: "
            f"{sum(str(p) in dino_cache for p in all_paths)}/{len(all_paths)}"
        )


# 마지막 저장
DINO_CACHE_PATH.write_text(
    json.dumps(
        dino_cache,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


done = sum(
    str(p) in dino_cache
    for p in all_paths
)

print(
    "\n✅ DINO 완료:",
    done,
    "/",
    len(all_paths)
)

print(
    "저장:",
    DINO_CACHE_PATH
)


del gd_model, gd_proc

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU allocated:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        2
    ),
    "GB"
)

전체 이미지: 16111
이미 DINO 완료: 20


Loading weights:   0%|          | 0/978 [00:00<?, ?it/s]

Grounding DINO FULL:   0%|          | 0/16111 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/grounding_dino/processing_grounding_dino.py:94: FutureWarning: The key `labels` is will return integer ids in `GroundingDinoProcessor.post_process_grounded_object_detection` output since v4.51.0. Use `text_labels` instead to retrieve string object names.
  warnings.warn(self.message, FutureWarning)



💾 DINO cache saved: 262/16111

💾 DINO cache saved: 512/16111

💾 DINO cache saved: 762/16111

💾 DINO cache saved: 1012/16111

💾 DINO cache saved: 1262/16111

💾 DINO cache saved: 1512/16111

💾 DINO cache saved: 1762/16111

💾 DINO cache saved: 2012/16111

💾 DINO cache saved: 2262/16111

💾 DINO cache saved: 2512/16111

💾 DINO cache saved: 2762/16111

💾 DINO cache saved: 3012/16111

💾 DINO cache saved: 3262/16111

💾 DINO cache saved: 3512/16111

💾 DINO cache saved: 3762/16111

💾 DINO cache saved: 4012/16111

💾 DINO cache saved: 4262/16111

💾 DINO cache saved: 4512/16111

💾 DINO cache saved: 4762/16111

💾 DINO cache saved: 5012/16111

💾 DINO cache saved: 5262/16111

💾 DINO cache saved: 5512/16111

💾 DINO cache saved: 5762/16111

💾 DINO cache saved: 6012/16111

💾 DINO cache saved: 6262/16111

💾 DINO cache saved: 6512/16111

💾 DINO cache saved: 6756/16111

💾 DINO cache saved: 7006/16111

💾 DINO cache saved: 7256/16111

💾 DINO cache saved: 7506/16111

💾 DINO cache saved: 7756/16111

💾 DINO cac

In [ ]:
# ===== [E-3] OCR / DINO 캐시 최종 검증 =====

import json

with open(OCR_CACHE_PATH, "r", encoding="utf-8") as f:
    ocr_cache = json.load(f)

with open(DINO_CACHE_PATH, "r", encoding="utf-8") as f:
    dino_cache = json.load(f)

all_paths = pd.concat([
    train_df["path"],
    dev_df["path"],
    test_df["path"]
]).drop_duplicates().tolist()


def inspect_cache(cache, name):

    missing = [
        str(p) for p in all_paths
        if str(p) not in cache
    ]

    errors = [
        (k, v["error"])
        for k, v in cache.items()
        if isinstance(v, dict)
        and "error" in v
    ]

    empty = [
        k for k, v in cache.items()
        if isinstance(v, list)
        and len(v) == 0
    ]

    print(f"\n===== {name} =====")
    print("cache entries :", len(cache))
    print("missing       :", len(missing))
    print("errors        :", len(errors))
    print("empty results :", len(empty))

    if errors:
        print("\n첫 error 5개:")
        for x in errors[:5]:
            print(x)


inspect_cache(ocr_cache, "OCR")
inspect_cache(dino_cache, "DINO")


===== OCR =====
cache entries : 16111
missing       : 0
errors        : 0
empty results : 764

===== DINO =====
cache entries : 16111
missing       : 0
errors        : 0
empty results : 493


## [교체 F] OCR/Object 참고자료 융합 프롬프트

In [ ]:
# ============================================================
# [E-LOAD] 저장된 OCR / DINO cache 다시 메모리에 로드
# ============================================================

import json

OCR_CACHE_PATH = CACHE_ROOT / "ocr_cache.json"
DINO_CACHE_PATH = CACHE_ROOT / "dino_cache.json"


assert OCR_CACHE_PATH.exists(), (
    f"OCR cache 없음: {OCR_CACHE_PATH}"
)

assert DINO_CACHE_PATH.exists(), (
    f"DINO cache 없음: {DINO_CACHE_PATH}"
)


with open(
    OCR_CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:
    ocr_cache = json.load(f)


with open(
    DINO_CACHE_PATH,
    "r",
    encoding="utf-8"
) as f:
    dino_cache = json.load(f)


print("✅ cache reload 완료")
print("OCR :", len(ocr_cache))
print("DINO:", len(dino_cache))


assert len(ocr_cache) == 16111
assert len(dino_cache) == 16111

print("✅ 16111 / 16111 무결성 확인")

✅ cache reload 완료
OCR : 16111
DINO: 16111
✅ 16111 / 16111 무결성 확인


In [ ]:
# get_reference 정상 작동 확인

_test_ref = get_reference(
    train_df.iloc[0]["path"]
)

print(_test_ref)

{'ocr': ['Jokpreme', 'BURGER', 'Suprem', 'SUNDAY', '[DIG', 'LoVE'], 'objects': ['chair', 'table', 'box cart', 'bottle', 'sign', 'cup', 'box carton', 'carton package', 'box', 'person', 'bin']}


In [ ]:
# ===== [교체 F v2] OCR confidence filtering + object dedup =====
import re

def clean_ocr_text(text):
    text = str(text).strip()

    # 문자/숫자가 하나도 없으면 제거
    if not re.search(r"[A-Za-z0-9가-힣]", text):
        return None

    # 너무 짧은 특수문자성 노이즈 제거
    if len(text) <= 1:
        return None

    return text


def get_reference(path):
    ocr = ocr_cache.get(str(path), [])
    det = dino_cache.get(str(path), [])

    # ---------- OCR ----------
    ocr_items = []

    if isinstance(ocr, list):
        for x in ocr:
            if not isinstance(x, dict):
                continue

            text = clean_ocr_text(x.get("text", ""))
            conf = float(x.get("conf", 0))

            # smoke 결과를 보면 0.25는 다소 noisy하므로
            # prompt에는 0.40 이상만 사용
            if text and conf >= 0.40:
                ocr_items.append((text, conf))

    # confidence 높은 순
    ocr_items.sort(key=lambda x: x[1], reverse=True)

    # 동일 문자열 중복 제거
    seen = set()
    ocr_texts = []

    for text, conf in ocr_items:
        key = text.lower()

        if key in seen:
            continue

        seen.add(key)
        ocr_texts.append(text)

        if len(ocr_texts) >= 15:
            break


    # ---------- Grounding DINO ----------
    obj_items = []

    if isinstance(det, list):
        for x in det:
            if not isinstance(x, dict):
                continue

            label = str(x.get("label", "")).strip()
            score = float(x.get("score", 0))

            if label and score >= 0.30:
                obj_items.append((label, score))

    obj_items.sort(key=lambda x: x[1], reverse=True)

    # 동일 object 반복 제거
    seen = set()
    obj_labels = []

    for label, score in obj_items:
        key = label.lower()

        if key in seen:
            continue

        seen.add(key)
        obj_labels.append(label)

        if len(obj_labels) >= 12:
            break

    return {
        "ocr": ocr_texts,
        "objects": obj_labels,
    }


def make_system_prompt(ref):
    ocr_line = ", ".join(ref["ocr"]) if ref["ocr"] else "없음"
    obj_line = ", ".join(ref["objects"]) if ref["objects"] else "없음"

    return (
        "당신은 이미지 기반 4지선다 VQA 전문가입니다.\n"
        "이미지를 가장 중요한 근거로 사용하고 질문과 선택지를 비교하세요.\n\n"
        "[보조 참고자료]\n"
        f"- OCR 추출 텍스트: {ocr_line}\n"
        f"- 객체 탐지 후보: {obj_line}\n\n"
        "참고자료는 오검출될 수 있으므로 이미지와 충돌하면 무시하세요.\n"
        "최종 답은 반드시 a, b, c, d 중 소문자 한 글자로 선택하세요."
    )


def build_mc_prompt(question, a, b, c, d):
    return (
        f"질문: {question}\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"
        "정답:"
    )

In [ ]:
# ===== [F-CHECK] 실제 Prompt 참고자료 확인 =====

for i in [0, 100, 1000]:

    row = train_df.iloc[i]

    ref = get_reference(row["path"])

    print("=" * 100)
    print("ID:", row["id"])
    print("QUESTION:", row["question"])

    print("\nOCR:")
    print(ref["ocr"][:10])

    print("\nOBJECTS:")
    print(ref["objects"][:10])

    print("\nSYSTEM PROMPT:")
    print(make_system_prompt(ref))

    print()

ID: train_0001.jpg
QUESTION: 사진 속 선반에 적힌 문구는 무엇인가요?

OCR:
['SUNDAY', 'BURGER', 'LoVE', '[DIG', 'Suprem', 'Jokpreme']

OBJECTS:
['chair', 'table', 'box cart', 'bottle', 'bottle', 'bottle', 'sign', 'cup', 'box carton', 'cup']

SYSTEM PROMPT:
당신은 이미지 기반 4지선다 VQA 전문가입니다.
내부 판단 순서: 관찰 -> 추론 -> 결론.

[보조 참고자료]
- Local OCR 추출 텍스트: SUNDAY, BURGER, LoVE, [DIG, Suprem, Jokpreme
- Local Object Detection 후보: chair, table, box cart, bottle, bottle, bottle, sign, cup, box carton, cup, box, cup, cup, bottle, carton package, bottle, person, bin

주의:
- 참고자료에는 오검출이 있을 수 있으므로 실제 이미지와 충돌하면 이미지를 우선하세요.
- 최종 출력은 반드시 a, b, c, d 중 소문자 한 글자만 출력하세요.

ID: train_0101.jpg
QUESTION: 사진 속 병에 적힌 술 이름은 무엇인가요?

OCR:
['창슬']

OBJECTS:
['bottle', 'cup', 'food', 'cup', 'container', 'table', 'metal']

SYSTEM PROMPT:
당신은 이미지 기반 4지선다 VQA 전문가입니다.
내부 판단 순서: 관찰 -> 추론 -> 결론.

[보조 참고자료]
- Local OCR 추출 텍스트: 창슬
- Local Object Detection 후보: bottle, cup, food, cup, container, table, metal

주의:
- 참고자료에는 오검출이 있을 수 있으므로 실제 이미지와 충돌하면 이미지를 

## [교체 G] 고정 train/valid split 저장

In [ ]:
SPLIT_PATH = DRIVE_ROOT/"fixed_split_seed42.csv"

if SPLIT_PATH.exists():
    split_manifest = pd.read_csv(SPLIT_PATH)
    train_idx = split_manifest.loc[split_manifest.split=="train", "row_idx"].astype(int).tolist()
    valid_idx = split_manifest.loc[split_manifest.split=="valid", "row_idx"].astype(int).tolist()
else:
    idx = np.arange(len(train_df))
    stratify = train_df["answer"] if "answer" in train_df.columns else None
    train_idx, valid_idx = train_test_split(
        idx, test_size=0.10, random_state=SEED, shuffle=True, stratify=stratify
    )
    split_manifest = pd.DataFrame({
        "row_idx": np.r_[train_idx, valid_idx],
        "split": ["train"]*len(train_idx) + ["valid"]*len(valid_idx),
    })
    split_manifest.to_csv(SPLIT_PATH, index=False)

train_subset = train_df.iloc[train_idx].reset_index(drop=True)
valid_subset = train_df.iloc[valid_idx].reset_index(drop=True)

print("train:", len(train_subset), "valid:", len(valid_subset))
print("split saved:", SPLIT_PATH)

train: 6042 valid: 672
split saved: /content/drive/MyDrive/ssafy_vqa_qwen38/fixed_split_seed42.csv


## [교체 H] Qwen3.8 W8A16 GPTQ 로드 + LoRA

In [ ]:
# ===== [H-FIX] GPTQ 로딩에 필요한 Optimum 설치 =====

!pip -q install -U "optimum==2.3.0"

import optimum
print("✅ optimum import OK")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 13.5 MB/s eta 0:00:00
✅ optimum import OK


In [ ]:
# ============================================================
# [H-0.5] 이전 세션 CUDA 객체 청소
#
# 현재 새로 로드한 `model`과 `processor`는 유지
# 이전 H-3/H-4 학습 객체들만 제거
# ============================================================

import gc
import sys
import torch


print("===== BEFORE =====")

print(
    "allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "reserved :",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)


# ------------------------------------------------------------
# 이전 학습/체크포인트 테스트에서 남았을 가능성이 있는 객체
# ------------------------------------------------------------

STALE_OBJECTS = [
    # 예전 model reference
    "base_model",
    "core_model",

    # optimizer / scheduler
    "optimizer",
    "scheduler",
    "optimizer_reload",
    "scheduler_reload",

    # parameter lists
    "trainable_params",
    "reload_trainable_params",

    # forward/backward graph
    "outputs",
    "loss",
    "raw_loss",
    "labels",
    "inputs",
    "answer_tensor",

    # H-3/H-4 LoRA test
    "ckpt_adapter",
    "current_adapter",
    "restored_adapter",

    "first_A",
    "first_B",

    "A_before",
    "A_after",
    "B_before",
    "B_after",

    "backup_tensor",

    # trainer state
    "trainer_state",

    # 기타 smoke variables
    "grad_params",
    "bad_grads",
]


deleted = []

for name in STALE_OBJECTS:

    if name in globals():

        try:
            del globals()[name]
            deleted.append(name)

        except Exception:
            pass


print(
    "\ndeleted:",
    deleted
)


# ------------------------------------------------------------
# IPython이 결과 객체를 잡고 있는 경우 제거
# ------------------------------------------------------------

try:

    ip = get_ipython()

    for key in [
        "_",
        "__",
        "___",
    ]:
        ip.user_ns.pop(
            key,
            None
        )

except Exception:
    pass


# ------------------------------------------------------------
# 예전 traceback reference 제거
# ------------------------------------------------------------

try:

    import traceback

    tb = getattr(
        sys,
        "last_traceback",
        None
    )

    if tb is not None:
        traceback.clear_frames(tb)

except Exception:
    pass


for attr in [
    "last_traceback",
    "last_value",
    "last_type",
]:

    if hasattr(sys, attr):

        try:
            setattr(
                sys,
                attr,
                None
            )
        except Exception:
            pass


# ------------------------------------------------------------
# GC
# ------------------------------------------------------------

for _ in range(3):
    gc.collect()


torch.cuda.empty_cache()


try:
    torch.cuda.ipc_collect()
except Exception:
    pass


# ------------------------------------------------------------
# 현재 model은 살아 있어야 함
# ------------------------------------------------------------

assert "model" in globals()
assert "processor" in globals()


triton_count = sum(
    1
    for m in model.modules()
    if m.__class__.__name__
    == "TritonV2Linear"
)


lora_count = sum(
    1
    for m in model.modules()
    if hasattr(m, "lora_A")
)


print("\n===== CURRENT MODEL =====")

print(
    "TritonV2Linear:",
    triton_count
)

print(
    "LoRA modules:",
    lora_count
)


assert triton_count == 400
assert lora_count == 0


print("\n===== AFTER =====")

print(
    "allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "reserved :",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

===== BEFORE =====
allocated: 45.71 GB
reserved : 47.01 GB

deleted: ['core_model', 'optimizer', 'scheduler', 'optimizer_reload', 'scheduler_reload', 'trainable_params', 'reload_trainable_params', 'raw_loss', 'ckpt_adapter', 'current_adapter', 'restored_adapter', 'first_A', 'first_B', 'A_before', 'A_after', 'B_before', 'B_after', 'backup_tensor', 'trainer_state', 'grad_params', 'bad_grads']

===== CURRENT MODEL =====
TritonV2Linear: 400
LoRA modules: 0

===== AFTER =====
allocated: 34.36 GB
reserved : 35.35 GB


In [ ]:
# ============================================================
# [H-0 4BIT TRAIN]
# Qwen3.8-27B GPTQ 4bit + trainable Triton backend
# ============================================================

import gc
import torch
from collections import Counter

from transformers import AutoProcessor
from gptqmodel import GPTQModel, BACKEND


MODEL_ID = "btbtyler09/Qwen3.8-27B-GPTQ-4bit"


gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


print("Loading:", MODEL_ID)
print("Backend: AUTO_TRAINABLE")


processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)


gptq_wrapper = GPTQModel.load(
    MODEL_ID,
    backend=BACKEND.AUTO_TRAINABLE,
    dtype=torch.bfloat16,
    trust_remote_code=True,
)


model = (
    gptq_wrapper.model
    if hasattr(gptq_wrapper, "model")
    else gptq_wrapper
)

model.config.use_cache = False


quant_classes = Counter()

for name, module in model.named_modules():

    if (
        "language_model" in name
        and "Linear" in module.__class__.__name__
    ):
        quant_classes[
            module.__class__.__name__
        ] += 1


print("\n===== QUANT MODULE CLASSES =====")

for cls, count in quant_classes.items():
    print(f"{cls:30s}: {count}")


allocated = (
    torch.cuda.memory_allocated() / 1024**3
)

reserved = (
    torch.cuda.memory_reserved() / 1024**3
)

total = (
    torch.cuda.get_device_properties(0).total_memory
    / 1024**3
)


print("\n===== LOAD RESULT =====")
print("allocated :", round(allocated, 2), "GB")
print("reserved  :", round(reserved, 2), "GB")
print("remaining :", round(total - allocated, 2), "GB")


INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.5.0
Transformers : 5.17.0
Torch        : 2.11.0+cu128
Triton       : 3.6.0
Loading: btbtyler09/Qwen3.8-27B-GPTQ-4bit
Backend: AUTO_TRAINABLE


HF: overriding trust_remote_code=True to False for `btbtyler09/Qwen3.8-27B-GPTQ-4bit` because model_type `qwen3_5` is integrated in installed transformers as `Qwen3_5ForCausalLM`.


from_quantized: adapter: None


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 21 files:   0%|          | 0/21 [00:00<?, ?it/s]

INFO  Estimated Quantization BPW (bits per weight): 4.85 bpw, based on [bits: 4, group_size: 32]
INFO  The layer model.visual.pos_embed is not quantized.                        
INFO  The layer model.visual.blocks.0.attn.qkv is not quantized.                
INFO  The layer model.visual.blocks.0.attn.proj is not quantized.               
INFO  The layer model.visual.blocks.0.mlp.linear_fc1 is not quantized.          
INFO  The layer model.visual.blocks.0.mlp.linear_fc2 is not quantized.          
INFO  The layer model.visual.blocks.1.attn.qkv is not quantized.                
INFO  The layer model.visual.blocks.1.attn.proj is not quantized.               
INFO  The layer model.visual.blocks.1.mlp.linear_fc1 is not quantized.          
INFO  The layer model.visual.blocks.1.mlp.linear_fc2 is not quantized.          
INFO  The layer model.visual.blocks.2.attn.qkv is not quantized.                
INFO  The layer model.visual.blocks.2.attn.proj is not quantized.               
INFO  The la

INFO  Format: Converting `format` from `FORMAT.GPTQ` to internal `FORMAT.GPTQ_V2`.
INFO  Format: Converting GPTQ v1 to v2                                          
INFO  Kernel: Auto-selection: adding candidate `TritonV2Linear`                 
INFO  Kernel: selected -> `TritonV2Linear`.                                     
INFO  Optimize: `TritonV2Linear` compilation triggered.                         
INFO  gc.collect() reclaimed 299 objects in 0.370s                              
INFO  Model: Loaded `generation_config`: GenerationConfig {
  "bos_token_id": 248044,
  "eos_token_id": 248044,
  "output_attentions": false,
  "output_hidden_states": false,
  "pad_token_id": 248044,
  "use_cache": true
}

INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.
INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 248044,
  "do_sample": true,
  "eos_token_id": [
    248046,
    248044
  ],
  "pad_token_id": 248044,
  "temper

In [ ]:
# ============================================================
# [H-1] 4bit GPTQ Qwen3.8 + LoRA attach
# ============================================================

import gc
import torch

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)


# ------------------------------------------------------------
# 1. training 준비
# ------------------------------------------------------------

model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },
)

print("✅ prepare_model_for_kbit_training 완료")


# ------------------------------------------------------------
# 2. LoRA target
# ------------------------------------------------------------

LORA_TARGETS = [
    # full attention
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",

    # linear attention
    "in_proj_qkv",
    "in_proj_z",
    "out_proj",

    # MLP
    "gate_proj",
    "up_proj",
    "down_proj",
]


lora_config = LoraConfig(
    r=16,
    lora_alpha=32,

    target_modules=LORA_TARGETS,

    lora_dropout=0.05,
    bias="none",

    task_type=TaskType.CAUSAL_LM,
)


# ------------------------------------------------------------
# 3. LoRA 부착
# ------------------------------------------------------------

model = get_peft_model(
    model,
    lora_config
)

print("\n✅ LoRA attach 완료")

model.print_trainable_parameters()


# ------------------------------------------------------------
# 4. 실제 주입 개수
# ------------------------------------------------------------

lora_modules = []

for name, module in model.named_modules():

    if hasattr(module, "lora_A"):
        lora_modules.append(name)


print(
    "\nLoRA injected modules:",
    len(lora_modules)
)

print("\n처음 20개:")

for name in lora_modules[:20]:
    print(name)


assert len(lora_modules) == 400, (
    f"❌ LoRA module 수 이상: {len(lora_modules)}"
)


# ------------------------------------------------------------
# 5. trainable parameters
# ------------------------------------------------------------

trainable_n = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_n = sum(
    p.numel()
    for p in model.parameters()
)


print(
    "\nTrainable parameters:",
    f"{trainable_n:,}"
)

print(
    "Trainable ratio:",
    f"{100 * trainable_n / total_n:.4f}%"
)


# ------------------------------------------------------------
# 6. VRAM
# ------------------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

allocated = (
    torch.cuda.memory_allocated()
    / 1024**3
)

reserved = (
    torch.cuda.memory_reserved()
    / 1024**3
)

total_vram = (
    torch.cuda.get_device_properties(0)
    .total_memory
    / 1024**3
)


print(
    "\n===== H-1 RESULT ====="
)

print(
    "allocated :",
    round(allocated, 2),
    "GB"
)

print(
    "reserved  :",
    round(reserved, 2),
    "GB"
)

print(
    "remaining :",
    round(
        total_vram - allocated,
        2
    ),
    "GB"
)

✅ prepare_model_for_kbit_training 완료

✅ LoRA attach 완료
trainable params: 108,789,760 || all params: 3,138,555,120 || trainable%: 3.4662

LoRA injected modules: 400

처음 20개:
base_model.model.model.language_model.layers.0.linear_attn.out_proj
base_model.model.model.language_model.layers.0.linear_attn.in_proj_qkv
base_model.model.model.language_model.layers.0.linear_attn.in_proj_z
base_model.model.model.language_model.layers.0.mlp.gate_proj
base_model.model.model.language_model.layers.0.mlp.up_proj
base_model.model.model.language_model.layers.0.mlp.down_proj
base_model.model.model.language_model.layers.1.linear_attn.out_proj
base_model.model.model.language_model.layers.1.linear_attn.in_proj_qkv
base_model.model.model.language_model.layers.1.linear_attn.in_proj_z
base_model.model.model.language_model.layers.1.mlp.gate_proj
base_model.model.model.language_model.layers.1.mlp.up_proj
base_model.model.model.language_model.layers.1.mlp.down_proj
base_model.model.model.language_model.layers.2.li

In [ ]:
# ============================================================
# [H-1.5] Gradient Checkpointing 강제 활성화
# ============================================================

import torch


# PEFT wrapper 안의 실제 HF 모델
if hasattr(model, "get_base_model"):
    core_model = model.get_base_model()
else:
    core_model = model


print("core type:", type(core_model))

core_model.config.use_cache = False


# ------------------------------------------------------------
# 핵심: PEFT 자동감지에 맡기지 않고 직접 활성화
# ------------------------------------------------------------

core_model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    }
)


# ------------------------------------------------------------
# 실제 decoder layer 상태 검사
# ------------------------------------------------------------

decoder_layers = []

vision_blocks = []

for name, module in core_model.named_modules():

    if module.__class__.__name__ == "Qwen3_5DecoderLayer":
        decoder_layers.append(
            (name, module)
        )

    elif module.__class__.__name__ == "Qwen3_5VisionBlock":
        vision_blocks.append(
            (name, module)
        )


decoder_gc = sum(
    bool(
        getattr(
            module,
            "gradient_checkpointing",
            False
        )
    )
    for _, module in decoder_layers
)


vision_gc = sum(
    bool(
        getattr(
            module,
            "gradient_checkpointing",
            False
        )
    )
    for _, module in vision_blocks
)


print("\n===== GRADIENT CHECKPOINTING CHECK =====")

print(
    "model.is_gradient_checkpointing:",
    getattr(
        core_model,
        "is_gradient_checkpointing",
        "N/A"
    )
)

print(
    "decoder layers:",
    len(decoder_layers)
)

print(
    "decoder GC ON:",
    decoder_gc
)

print(
    "vision blocks:",
    len(vision_blocks)
)

print(
    "vision GC ON:",
    vision_gc
)


assert (
    getattr(
        core_model,
        "is_gradient_checkpointing",
        False
    )
    is True
)

assert len(decoder_layers) == 64

assert decoder_gc == 64


print(
    "\n✅ Gradient Checkpointing REAL ENABLED"
)

core type: <class 'transformers.models.qwen3_5.modeling_qwen3_5.Qwen3_5ForConditionalGeneration'>

===== GRADIENT CHECKPOINTING CHECK =====
model.is_gradient_checkpointing: True
decoder layers: 64
decoder GC ON: 64
vision blocks: 27
vision GC ON: 27

✅ Gradient Checkpointing REAL ENABLED


In [ ]:
# ============================================================
# [H-1.6] GPTQModel Triton backward BUG PATCH
#
# forward inputs = 8개
# 기존 backward return = 7개  -> RuntimeError
# 수정 backward return = 8개
# ============================================================

from torch.amp import custom_bwd
from gptqmodel.nn_modules.triton_utils import dequant as gptq_dequant


QuantLinearFunction = gptq_dequant.QuantLinearFunction
quant_matmul = gptq_dequant.quant_matmul


@custom_bwd(device_type="cuda")
def _fixed_quant_backward(ctx, grad_output):

    qweight, scales, qzeros, g_idx = ctx.saved_tensors

    bits = ctx.bits
    maxq = ctx.maxq
    pack_bits = ctx.pack_bits

    grad_input = None

    if ctx.needs_input_grad[0]:

        grad_input = quant_matmul(
            grad_output,
            qweight,
            scales,
            qzeros,
            g_idx,
            bits,
            pack_bits,
            maxq,
            transpose=True,
        )

    # --------------------------------------------------------
    # forward 입력:
    # input, qweight, scales, qzeros,
    # g_idx, bits, pack_bits, maxq
    #
    # => backward도 반드시 8개 반환
    # --------------------------------------------------------

    return (
        grad_input,  # input
        None,        # qweight
        None,        # scales
        None,        # qzeros
        None,        # g_idx
        None,        # bits
        None,        # pack_bits
        None,        # maxq
    )


QuantLinearFunction.backward = staticmethod(
    _fixed_quant_backward
)


# cache warning도 명시적으로 OFF
core_model.config.use_cache = False

if hasattr(model, "config"):
    model.config.use_cache = False


print("✅ QuantLinearFunction backward patch 적용")
print("forward input gradients expected : 8")
print("patched backward returns         : 8")
print(
    "gradient checkpointing          :",
    core_model.is_gradient_checkpointing
)

✅ QuantLinearFunction backward patch 적용
forward input gradients expected : 8
patched backward returns         : 8
gradient checkpointing          : True


In [ ]:
# ============================================================
# [H-2 PREP-0] 경로 변수 복구
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38"
)

DATA_ROOT = Path(
    "/content/data"
)

CACHE_ROOT = (
    PROJECT_ROOT
    / "cache"
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT   :", DATA_ROOT)
print("CACHE_ROOT  :", CACHE_ROOT)

assert (DATA_ROOT / "train.csv").exists()
assert (CACHE_ROOT / "ocr_cache.json").exists()
assert (CACHE_ROOT / "dino_cache.json").exists()

print("\n✅ H-2 경로 복구 완료")

PROJECT_ROOT: /content/drive/MyDrive/ssafy_vqa_qwen38
DATA_ROOT   : /content/data
CACHE_ROOT  : /content/drive/MyDrive/ssafy_vqa_qwen38/cache

✅ H-2 경로 복구 완료


In [ ]:
# ===== [H-2 PREP] 데이터 / cache / prompt 함수 복구 =====

import json
import re
import pandas as pd
from pathlib import Path

IMAGE_SIZE = 1024

# CSV
if "train_df" not in globals():
    train_df = pd.read_csv(DATA_ROOT / "train.csv")

    def normalize_path(p):
        p = str(p)
        if p.startswith("/"):
            return p
        return str(DATA_ROOT / p)

    train_df["path"] = train_df["path"].map(normalize_path)

# CACHE
if "ocr_cache" not in globals():
    with open(CACHE_ROOT / "ocr_cache.json", "r", encoding="utf-8") as f:
        ocr_cache = json.load(f)

if "dino_cache" not in globals():
    with open(CACHE_ROOT / "dino_cache.json", "r", encoding="utf-8") as f:
        dino_cache = json.load(f)


def clean_ocr_text(text):
    text = str(text).strip()

    if not re.search(r"[A-Za-z0-9가-힣]", text):
        return None

    if len(text) <= 1:
        return None

    return text


def get_reference(path):

    ocr = ocr_cache.get(str(path), [])
    det = dino_cache.get(str(path), [])

    # OCR
    ocr_items = []

    if isinstance(ocr, list):
        for x in ocr:

            if not isinstance(x, dict):
                continue

            text = clean_ocr_text(
                x.get("text", "")
            )

            conf = float(
                x.get("conf", 0)
            )

            if text and conf >= 0.40:
                ocr_items.append(
                    (text, conf)
                )

    ocr_items.sort(
        key=lambda x: x[1],
        reverse=True
    )

    seen = set()
    ocr_texts = []

    for text, conf in ocr_items:

        key = text.lower()

        if key in seen:
            continue

        seen.add(key)
        ocr_texts.append(text)

        if len(ocr_texts) >= 15:
            break

    # DINO
    obj_items = []

    if isinstance(det, list):

        for x in det:

            if not isinstance(x, dict):
                continue

            label = str(
                x.get("label", "")
            ).strip()

            score = float(
                x.get("score", 0)
            )

            if label and score >= 0.30:
                obj_items.append(
                    (label, score)
                )

    obj_items.sort(
        key=lambda x: x[1],
        reverse=True
    )

    seen = set()
    obj_labels = []

    for label, score in obj_items:

        key = label.lower()

        if key in seen:
            continue

        seen.add(key)
        obj_labels.append(label)

        if len(obj_labels) >= 12:
            break

    return {
        "ocr": ocr_texts,
        "objects": obj_labels,
    }


def make_system_prompt(ref):

    ocr_line = (
        ", ".join(ref["ocr"])
        if ref["ocr"]
        else "없음"
    )

    obj_line = (
        ", ".join(ref["objects"])
        if ref["objects"]
        else "없음"
    )

    return (
        "당신은 이미지 기반 4지선다 VQA 전문가입니다.\n"
        "이미지를 가장 중요한 근거로 사용하고 질문과 선택지를 비교하세요.\n\n"
        "[보조 참고자료]\n"
        f"- OCR 추출 텍스트: {ocr_line}\n"
        f"- 객체 탐지 후보: {obj_line}\n\n"
        "참고자료는 오검출될 수 있으므로 이미지와 충돌하면 무시하세요.\n"
        "최종 답은 반드시 a, b, c, d 중 소문자 한 글자로 선택하세요."
    )


def build_mc_prompt(question, a, b, c, d):

    return (
        f"질문: {question}\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"
        "정답:"
    )


print("train:", len(train_df))
print("OCR :", len(ocr_cache))
print("DINO:", len(dino_cache))

print(
    get_reference(
        train_df.iloc[0]["path"]
    )
)

train: 6714
OCR : 16111
DINO: 16111
{'ocr': ['Jokpreme', 'BURGER', 'Suprem', 'SUNDAY', '[DIG', 'LoVE'], 'objects': ['chair', 'table', 'box cart', 'bottle', 'sign', 'cup', 'box carton', 'carton package', 'box', 'person', 'bin']}


In [ ]:
import torch

print(
    "allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

allocated: 25.06 GB
reserved: 25.21 GB


In [ ]:
# ============================================================
# [H-2B SMOKE FINAL v2]
# Qwen3.8 4bit + LoRA
# 1 image REAL forward / backward smoke test
#
# 핵심:
# - model 전체는 train
# - LoRA는 train
# - TritonV2Linear base 400개만 eval
# - thinking OFF
# - 정답 a/b/c/d 1 token만 loss
# - OOM 시 cleanup까지 자동 수행
# ============================================================

import gc
import torch
from PIL import Image


# ------------------------------------------------------------
# 0. 이미지 resize
# ------------------------------------------------------------

def smoke_letterbox(image, size=1024):

    image = image.convert("RGB")

    w, h = image.size

    scale = min(
        size / w,
        size / h
    )

    nw = max(1, round(w * scale))
    nh = max(1, round(h * scale))

    image = image.resize(
        (nw, nh),
        Image.Resampling.LANCZOS
    )

    canvas = Image.new(
        "RGB",
        (size, size),
        (0, 0, 0)
    )

    canvas.paste(
        image,
        (
            (size - nw) // 2,
            (size - nh) // 2
        )
    )

    return canvas


# ------------------------------------------------------------
# 1. 실제 train sample
# ------------------------------------------------------------

row = train_df.iloc[0]

image = smoke_letterbox(
    Image.open(row["path"]),
    IMAGE_SIZE
)

ref = get_reference(
    row["path"]
)

system_text = make_system_prompt(
    ref
)

user_text = build_mc_prompt(
    row["question"],
    row["a"],
    row["b"],
    row["c"],
    row["d"],
)

answer = str(
    row["answer"]
).strip().lower()

assert answer in {
    "a", "b", "c", "d"
}


print("ID     :", row["id"])
print("Answer :", answer)
print("Image  :", image.size)


# ------------------------------------------------------------
# 2. ABCD token 확인
# ------------------------------------------------------------

tokenizer = processor.tokenizer

CHOICE_IDS = {}

print("\n===== ABCD TOKEN CHECK =====")

for letter in "abcd":

    ids = tokenizer.encode(
        letter,
        add_special_tokens=False
    )

    print(
        letter,
        "->",
        ids,
        repr(tokenizer.decode(ids))
    )

    assert len(ids) == 1

    CHOICE_IDS[letter] = ids[0]


answer_id = CHOICE_IDS[answer]


# ------------------------------------------------------------
# 3. multimodal prompt
# ------------------------------------------------------------

messages = [
    {
        "role": "system",
        "content": [
            {
                "type": "text",
                "text": system_text
            }
        ]
    },

    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": image
            },
            {
                "type": "text",
                "text": user_text
            }
        ]
    }
]


inputs = processor.apply_chat_template(
    messages,

    tokenize=True,

    add_generation_prompt=True,

    # ★ reasoning 없이 바로 분류
    enable_thinking=False,

    return_dict=True,
    return_tensors="pt",
)


prompt_len = inputs[
    "input_ids"
].shape[1]


print(
    "\nProcessor keys:",
    list(inputs.keys())
)

print(
    "prompt tokens:",
    prompt_len
)


tail = tokenizer.decode(
    inputs["input_ids"][0, -50:],
    skip_special_tokens=False
)


print("\n===== PROMPT TAIL =====")
print(tail)


assert "</think>" in tail, (
    "❌ enable_thinking=False 적용 실패"
)


# ------------------------------------------------------------
# 4. 정답 1 token append
# ------------------------------------------------------------

answer_tensor = torch.tensor(
    [[answer_id]],
    dtype=inputs["input_ids"].dtype
)


inputs["input_ids"] = torch.cat(
    [
        inputs["input_ids"],
        answer_tensor
    ],
    dim=1
)


# attention_mask +1
inputs["attention_mask"] = torch.cat(
    [
        inputs["attention_mask"],

        torch.ones(
            (1, 1),
            dtype=inputs["attention_mask"].dtype
        )
    ],
    dim=1
)


# Qwen multimodal text token = 0
if "mm_token_type_ids" in inputs:

    inputs["mm_token_type_ids"] = torch.cat(
        [
            inputs["mm_token_type_ids"],

            torch.zeros(
                (1, 1),
                dtype=inputs[
                    "mm_token_type_ids"
                ].dtype
            )
        ],
        dim=1
    )


# ------------------------------------------------------------
# 5. shape 검증
# ------------------------------------------------------------

seq_len = inputs[
    "input_ids"
].shape[1]


print(
    "\nFinal sequence:",
    seq_len
)

print(
    "attention_mask:",
    inputs[
        "attention_mask"
    ].shape
)


assert (
    inputs[
        "attention_mask"
    ].shape[1]
    == seq_len
)


if "mm_token_type_ids" in inputs:

    print(
        "mm_token_type_ids:",
        inputs[
            "mm_token_type_ids"
        ].shape
    )

    assert (
        inputs[
            "mm_token_type_ids"
        ].shape[1]
        == seq_len
    )


print(
    "Final tail:",
    repr(
        tokenizer.decode(
            inputs[
                "input_ids"
            ][0, -30:]
        )
    )
)


# ------------------------------------------------------------
# 6. answer token 하나만 loss
# ------------------------------------------------------------

labels = torch.full_like(
    inputs["input_ids"],
    -100
)

labels[
    0,
    -1
] = answer_id


print(
    "\nLoss target:",
    tokenizer.decode(
        labels[
            labels != -100
        ].tolist()
    )
)


assert (
    labels != -100
).sum().item() == 1


# ------------------------------------------------------------
# 7. GPU 이동
# ------------------------------------------------------------

inputs = inputs.to(
    "cuda"
)

labels = labels.to(
    "cuda"
)


# ------------------------------------------------------------
# 8. ★ 핵심 FIX
#
# 전체 모델 + LoRA = train
# GPTQ Triton base 400개만 eval
# ------------------------------------------------------------

model.train()


quant_eval_names = []

for name, module in model.named_modules():

    if (
        module.__class__.__name__
        == "TritonV2Linear"
    ):

        module.eval()

        quant_eval_names.append(
            name
        )


quant_train_count = sum(

    1

    for _, module
    in model.named_modules()

    if (
        module.__class__.__name__
        == "TritonV2Linear"
        and module.training
    )
)


print("\n===== MIXED TRAIN MODE =====")

print(
    "model.training:",
    model.training
)

print(
    "TritonV2Linear forced eval:",
    len(quant_eval_names)
)

print(
    "quant layers still training:",
    quant_train_count
)

print(
    "gradient checkpointing:",
    getattr(
        core_model,
        "is_gradient_checkpointing",
        False
    )
)

assert (
    core_model.is_gradient_checkpointing
    is True
)


assert model.training is True

assert len(
    quant_eval_names
) == 400

assert quant_train_count == 0


# LoRA가 train 상태인지도 확인
lora_train = 0
lora_total = 0

for name, module in model.named_modules():

    if hasattr(
        module,
        "lora_A"
    ):

        lora_total += 1

        if module.training:
            lora_train += 1


print(
    "LoRA modules training:",
    lora_train,
    "/",
    lora_total
)


assert lora_total == 400
assert lora_train == 400


print(
    "\n✅ mixed train/eval mode ready"
)


# ------------------------------------------------------------
# 9. forward / backward 준비
# ------------------------------------------------------------

model.zero_grad(
    set_to_none=True
)

torch.cuda.empty_cache()

torch.cuda.reset_peak_memory_stats()


print(
    "\nVRAM before forward:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        2
    ),
    "GB"
)


outputs = None
loss = None


# ------------------------------------------------------------
# 10. REAL forward + backward
# ------------------------------------------------------------

try:

    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16
    ):

        outputs = model(
            **inputs,
            labels=labels
        )

        loss = outputs.loss


    print("\n===== FORWARD =====")

    print(
        "loss:",
        float(
            loss.detach().cpu()
        )
    )

    print(
        "allocated:",
        round(
            torch.cuda.memory_allocated()
            / 1024**3,
            2
        ),
        "GB"
    )

    print(
        "peak:",
        round(
            torch.cuda.max_memory_allocated()
            / 1024**3,
            2
        ),
        "GB"
    )


    assert torch.isfinite(loss)


    # --------------------------------------------------------
    # backward
    # --------------------------------------------------------

    loss.backward()


    print("\n===== BACKWARD =====")

    print(
        "allocated:",
        round(
            torch.cuda.memory_allocated()
            / 1024**3,
            2
        ),
        "GB"
    )

    print(
        "peak VRAM:",
        round(
            torch.cuda.max_memory_allocated()
            / 1024**3,
            2
        ),
        "GB"
    )


    # --------------------------------------------------------
    # gradient 검사
    # --------------------------------------------------------

    grad_params = []
    bad_grads = []


    for name, param in model.named_parameters():

        if not param.requires_grad:
            continue

        if param.grad is None:
            continue

        grad_params.append(
            name
        )

        if not torch.isfinite(
            param.grad
        ).all():

            bad_grads.append(
                name
            )


    print(
        "\nLoRA params with grad:",
        len(grad_params)
    )

    print(
        "non-finite grad params:",
        len(bad_grads)
    )


    print(
        "\nGradient sample:"
    )


    shown = 0

    for name, param in model.named_parameters():

        if (
            param.requires_grad
            and param.grad is not None
        ):

            print(
                name,
                "mean_abs_grad =",
                float(
                    param.grad
                    .abs()
                    .mean()
                    .detach()
                    .cpu()
                )
            )

            shown += 1

            if shown >= 5:
                break


    assert len(
        grad_params
    ) > 0

    assert len(
        bad_grads
    ) == 0


    print(
        "\n✅ H-2B SMOKE SUCCESS"
    )


finally:

    # --------------------------------------------------------
    # OOM이 나도 최대한 메모리 정상 복귀
    # --------------------------------------------------------

    model.zero_grad(
        set_to_none=True
    )


    if outputs is not None:
        del outputs

    if loss is not None:
        del loss


    gc.collect()
    torch.cuda.empty_cache()


    print(
        "\n===== CLEANUP ====="
    )

    print(
        "allocated:",
        round(
            torch.cuda.memory_allocated()
            / 1024**3,
            2
        ),
        "GB"
    )

    print(
        "reserved:",
        round(
            torch.cuda.memory_reserved()
            / 1024**3,
            2
        ),
        "GB"
    )

ID     : train_0001.jpg
Answer : a
Image  : (1024, 1024)

===== ABCD TOKEN CHECK =====
a -> [64] 'a'
b -> [65] 'b'
c -> [66] 'c'
d -> [67] 'd'

Processor keys: ['input_ids', 'attention_mask', 'mm_token_type_ids', 'pixel_values', 'image_grid_thw']
prompt tokens: 1237

===== PROMPT TAIL =====
?
(a) SUNDAY BURGER LOVE
(b) MONDAY SALAD TIME
(c) FRIDAY PIZZA PARTY
(d) WEDNESDAY SOUP DAY

정답:<|im_end|>
<|im_start|>assistant
<think>

</think>



Final sequence: 1238
attention_mask: torch.Size([1, 1238])
mm_token_type_ids: torch.Size([1, 1238])
Final tail: ' FRIDAY PIZZA PARTY\n(d) WEDNESDAY SOUP DAY\n\n정답:<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\na'

Loss target: a

===== MIXED TRAIN MODE =====
model.training: True
TritonV2Linear forced eval: 400
quant layers still training: 0
gradient checkpointing: True
LoRA modules training: 400 / 400

✅ mixed train/eval mode ready

VRAM before forward: 25.07 GB


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.



===== FORWARD =====
loss: 1.0893259048461914
allocated: 30.73 GB
peak: 32.1 GB

===== BACKWARD =====
allocated: 26.15 GB
peak VRAM: 33.02 GB

LoRA params with grad: 800
non-finite grad params: 0

Gradient sample:
base_model.model.model.language_model.layers.0.linear_attn.out_proj.lora_A.default.weight mean_abs_grad = 0.0
base_model.model.model.language_model.layers.0.linear_attn.out_proj.lora_B.default.weight mean_abs_grad = 0.014277445152401924
base_model.model.model.language_model.layers.0.linear_attn.in_proj_qkv.lora_A.default.weight mean_abs_grad = 0.0
base_model.model.model.language_model.layers.0.linear_attn.in_proj_qkv.lora_B.default.weight mean_abs_grad = 0.0015818264801055193
base_model.model.model.language_model.layers.0.linear_attn.in_proj_z.lora_A.default.weight mean_abs_grad = 0.0

✅ H-2B SMOKE SUCCESS

===== CLEANUP =====
allocated: 25.09 GB
reserved: 25.21 GB


In [ ]:
# ============================================================
# [H-3] 32-sample MINI TRAIN
#
# 목적:
# - 실제 optimizer.step() 검증
# - grad accumulation = 16
# - 총 32 samples = optimizer 2 steps
# - LoRA A/B gradient 확인
# - step별 adapter + optimizer/scheduler 저장
#
# 현재 성공한 H-2 상태 그대로 이어서 실행
# ============================================================

import gc
import json
import math
import time
from pathlib import Path

import torch
from PIL import Image

from transformers import get_cosine_schedule_with_warmup


# ============================================================
# 0. 설정
# ============================================================

NUM_SAMPLES = 32

GRAD_ACCUM = 16

LR = 1e-4

WEIGHT_DECAY = 0.01

MAX_GRAD_NORM = 1.0


SMOKE_DIR = (
    Path(RUN_DIR)
    / "h3_smoke32"
)

SMOKE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "SAVE DIR:",
    SMOKE_DIR
)


# ============================================================
# 1. GPTQModel Triton backward patch
#
# 세션을 재시작하지 않았다면 이미 적용되어 있지만
# H-3 자체가 독립적으로 안전하도록 다시 적용
# ============================================================

from torch.amp import custom_bwd

from gptqmodel.nn_modules.triton_utils import (
    dequant as gptq_dequant
)


QuantLinearFunction = (
    gptq_dequant.QuantLinearFunction
)

quant_matmul = (
    gptq_dequant.quant_matmul
)


@custom_bwd(
    device_type="cuda"
)
def _fixed_quant_backward(
    ctx,
    grad_output
):

    (
        qweight,
        scales,
        qzeros,
        g_idx,
    ) = ctx.saved_tensors


    bits = ctx.bits
    maxq = ctx.maxq
    pack_bits = ctx.pack_bits


    grad_input = None


    if ctx.needs_input_grad[0]:

        grad_input = quant_matmul(
            grad_output,
            qweight,
            scales,
            qzeros,
            g_idx,
            bits,
            pack_bits,
            maxq,
            transpose=True,
        )


    # forward input 8개에 맞춰
    # backward도 8개 반환
    return (
        grad_input,
        None,
        None,
        None,
        None,
        None,
        None,
        None,
    )


QuantLinearFunction.backward = staticmethod(
    _fixed_quant_backward
)


print(
    "✅ Triton backward patch ready"
)


# ============================================================
# 2. Gradient Checkpointing 재확인
# ============================================================

if hasattr(
    model,
    "get_base_model"
):
    core_model = (
        model.get_base_model()
    )
else:
    core_model = model


core_model.config.use_cache = False

if hasattr(
    model,
    "config"
):
    model.config.use_cache = False


# 혹시라도 꺼졌다면 다시 ON
if not getattr(
    core_model,
    "is_gradient_checkpointing",
    False
):

    core_model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={
            "use_reentrant": False
        }
    )


print(
    "gradient checkpointing:",
    core_model.is_gradient_checkpointing
)


assert (
    core_model.is_gradient_checkpointing
    is True
)


# ============================================================
# 3. mixed training mode
#
# LoRA / 전체 wrapper = train
# TritonV2Linear base = eval
# ============================================================

def set_training_mode():

    model.train()


    quant_count = 0

    for _, module in (
        model.named_modules()
    ):

        if (
            module.__class__.__name__
            == "TritonV2Linear"
        ):

            module.eval()

            quant_count += 1


    assert quant_count == 400


    still_train = sum(
        1

        for _, module
        in model.named_modules()

        if (
            module.__class__.__name__
            == "TritonV2Linear"
            and module.training
        )
    )


    assert still_train == 0


    return quant_count


quant_count = set_training_mode()


print(
    "Triton base eval:",
    quant_count
)


# ============================================================
# 4. 이미지 처리
# ============================================================

def train_letterbox(
    image,
    size=1024
):

    image = image.convert(
        "RGB"
    )


    w, h = image.size

    scale = min(
        size / w,
        size / h
    )


    nw = max(
        1,
        round(w * scale)
    )

    nh = max(
        1,
        round(h * scale)
    )


    image = image.resize(
        (nw, nh),
        Image.Resampling.LANCZOS
    )


    canvas = Image.new(
        "RGB",
        (size, size),
        (0, 0, 0)
    )


    canvas.paste(
        image,
        (
            (size - nw) // 2,
            (size - nh) // 2,
        )
    )


    return canvas


# ============================================================
# 5. 1 sample → model input
# ============================================================

tokenizer = processor.tokenizer


CHOICE_IDS = {}

for letter in "abcd":

    ids = tokenizer.encode(
        letter,
        add_special_tokens=False
    )

    assert len(ids) == 1

    CHOICE_IDS[
        letter
    ] = ids[0]


def make_train_input(row):

    # -------------------------
    # image
    # -------------------------

    image = train_letterbox(
        Image.open(
            row["path"]
        ),
        IMAGE_SIZE
    )


    # -------------------------
    # OCR / DINO reference
    # -------------------------

    ref = get_reference(
        row["path"]
    )


    system_text = (
        make_system_prompt(
            ref
        )
    )


    user_text = (
        build_mc_prompt(
            row["question"],
            row["a"],
            row["b"],
            row["c"],
            row["d"],
        )
    )


    answer = str(
        row["answer"]
    ).strip().lower()


    assert answer in (
        "a",
        "b",
        "c",
        "d",
    )


    answer_id = (
        CHOICE_IDS[
            answer
        ]
    )


    # -------------------------
    # conversation
    # -------------------------

    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": system_text,
                }
            ],
        },

        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image,
                },

                {
                    "type": "text",
                    "text": user_text,
                },
            ],
        },
    ]


    inputs = (
        processor.apply_chat_template(
            messages,

            tokenize=True,

            add_generation_prompt=True,

            enable_thinking=False,

            return_dict=True,

            return_tensors="pt",
        )
    )


    prompt_len = (
        inputs[
            "input_ids"
        ].shape[1]
    )


    # -------------------------
    # answer token append
    # -------------------------

    answer_tensor = torch.tensor(
        [[answer_id]],
        dtype=inputs[
            "input_ids"
        ].dtype,
    )


    inputs[
        "input_ids"
    ] = torch.cat(
        [
            inputs[
                "input_ids"
            ],
            answer_tensor,
        ],
        dim=1,
    )


    inputs[
        "attention_mask"
    ] = torch.cat(
        [
            inputs[
                "attention_mask"
            ],

            torch.ones(
                (1, 1),

                dtype=inputs[
                    "attention_mask"
                ].dtype,
            ),
        ],
        dim=1,
    )


    if (
        "mm_token_type_ids"
        in inputs
    ):

        inputs[
            "mm_token_type_ids"
        ] = torch.cat(
            [
                inputs[
                    "mm_token_type_ids"
                ],

                torch.zeros(
                    (1, 1),

                    dtype=inputs[
                        "mm_token_type_ids"
                    ].dtype,
                ),
            ],
            dim=1,
        )


    # -------------------------
    # label = 마지막 1 token
    # -------------------------

    labels = torch.full_like(
        inputs[
            "input_ids"
        ],
        -100,
    )


    labels[
        0,
        -1
    ] = answer_id


    return (
        inputs,
        labels,
        answer,
        prompt_len,
    )


# ============================================================
# 6. optimizer / scheduler
# ============================================================

trainable_params = [
    p

    for p in model.parameters()

    if p.requires_grad
]


optimizer = torch.optim.AdamW(
    trainable_params,

    lr=LR,

    betas=(
        0.9,
        0.999,
    ),

    eps=1e-8,

    weight_decay=WEIGHT_DECAY,
)


TOTAL_OPT_STEPS = math.ceil(
    NUM_SAMPLES
    / GRAD_ACCUM
)


WARMUP_STEPS = max(
    1,
    int(
        TOTAL_OPT_STEPS
        * 0.1
    ),
)


scheduler = (
    get_cosine_schedule_with_warmup(
        optimizer,

        num_warmup_steps=
        WARMUP_STEPS,

        num_training_steps=
        TOTAL_OPT_STEPS,
    )
)


print(
    "\n===== OPTIMIZER ====="
)

print(
    "samples:",
    NUM_SAMPLES
)

print(
    "grad accumulation:",
    GRAD_ACCUM
)

print(
    "optimizer steps:",
    TOTAL_OPT_STEPS
)

print(
    "LR:",
    LR
)


# ============================================================
# 7. 첫 LoRA A/B parameter 찾기
# ============================================================

first_A = None
first_B = None


for name, param in (
    model.named_parameters()
):

    if (
        first_A is None
        and ".lora_A."
        in name
    ):

        first_A = (
            name,
            param
        )


    if (
        first_B is None
        and ".lora_B."
        in name
    ):

        first_B = (
            name,
            param
        )


assert first_A is not None
assert first_B is not None


print(
    "\nA:",
    first_A[0]
)

print(
    "B:",
    first_B[0]
)


# optimizer step 전 값 기록
A_before = (
    first_A[1]
    .detach()
    .float()
    .clone()
    .cpu()
)

B_before = (
    first_B[1]
    .detach()
    .float()
    .clone()
    .cpu()
)


# ============================================================
# 8. checkpoint 함수
# ============================================================

def save_smoke_checkpoint(
    opt_step,
    micro_step,
    avg_loss,
):

    ckpt_dir = (
        SMOKE_DIR
        / f"checkpoint-step-{opt_step:03d}"
    )


    ckpt_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    # LoRA adapter
    model.save_pretrained(
        ckpt_dir,
        safe_serialization=True,
    )


    # optimizer + scheduler
    torch.save(
        {
            "optimizer":
                optimizer.state_dict(),

            "scheduler":
                scheduler.state_dict(),

            "optimizer_step":
                opt_step,

            "micro_step":
                micro_step,

            "avg_loss":
                avg_loss,
        },

        ckpt_dir
        / "trainer_state.pt",
    )


    # metadata
    metadata = {
        "base_model":
            MODEL_ID,

        "image_size":
            IMAGE_SIZE,

        "num_samples":
            NUM_SAMPLES,

        "grad_accum":
            GRAD_ACCUM,

        "lr":
            LR,

        "optimizer_step":
            opt_step,

        "micro_step":
            micro_step,

        "avg_loss":
            avg_loss,
    }


    (
        ckpt_dir
        / "run_meta.json"
    ).write_text(
        json.dumps(
            metadata,
            ensure_ascii=False,
            indent=2,
        ),

        encoding="utf-8",
    )


    print(
        "\n💾 checkpoint saved:"
    )

    print(
        ckpt_dir
    )


# ============================================================
# 9. MINI TRAIN
# ============================================================

set_training_mode()

model.zero_grad(
    set_to_none=True
)


optimizer.zero_grad(
    set_to_none=True
)


torch.cuda.reset_peak_memory_stats()


running_loss = 0.0

block_loss = 0.0

optimizer_step = 0


start_time = time.time()


print(
    "\n================================"
)
print(
    "H-3 MINI TRAIN START"
)
print(
    "================================\n"
)


for micro_idx in range(
    NUM_SAMPLES
):

    row = train_df.iloc[
        micro_idx
    ]


    inputs = None
    labels = None
    outputs = None
    loss = None


    try:

        (
            inputs,
            labels,
            answer,
            prompt_len,
        ) = make_train_input(
            row
        )


        inputs = inputs.to(
            "cuda"
        )

        labels = labels.to(
            "cuda"
        )


        # --------------------------------
        # forward
        # --------------------------------

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):

            outputs = model(
                **inputs,
                labels=labels,
            )

            raw_loss = (
                outputs.loss
            )


            # gradient accumulation
            loss = (
                raw_loss
                / GRAD_ACCUM
            )


        raw_loss_value = float(
            raw_loss
            .detach()
            .cpu()
        )


        running_loss += (
            raw_loss_value
        )

        block_loss += (
            raw_loss_value
        )


        # --------------------------------
        # backward
        # --------------------------------

        loss.backward()


        # --------------------------------
        # optimizer step
        # --------------------------------

        do_step = (
            (
                micro_idx + 1
            )
            % GRAD_ACCUM
            == 0
        ) or (
            micro_idx + 1
            == NUM_SAMPLES
        )


        # 첫 optimizer step 이후
        # 다음 accumulation에서
        # A gradient가 생기는지 확인
        if (
            micro_idx
            == GRAD_ACCUM
        ):

            A_grad = (
                first_A[1].grad
            )

            B_grad = (
                first_B[1].grad
            )


            print(
                "\n===== POST-STEP GRAD CHECK ====="
            )

            print(
                "A grad mean:",
                None
                if A_grad is None
                else float(
                    A_grad
                    .abs()
                    .mean()
                    .detach()
                    .cpu()
                )
            )

            print(
                "B grad mean:",
                None
                if B_grad is None
                else float(
                    B_grad
                    .abs()
                    .mean()
                    .detach()
                    .cpu()
                )
            )


        if do_step:

            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                MAX_GRAD_NORM,
            )


            optimizer.step()

            scheduler.step()


            optimizer.zero_grad(
                set_to_none=True
            )

            model.zero_grad(
                set_to_none=True
            )


            optimizer_step += 1


            current_block_count = (
                GRAD_ACCUM

                if (
                    micro_idx + 1
                    >= GRAD_ACCUM
                )

                else (
                    micro_idx + 1
                )
            )


            avg_block_loss = (
                block_loss
                / current_block_count
            )


            print(
                "\n================================"
            )

            print(
                f"OPT STEP {optimizer_step}"
            )

            print(
                f"micro:"
                f" {micro_idx + 1}"
                f"/{NUM_SAMPLES}"
            )

            print(
                "block avg loss:",
                round(
                    avg_block_loss,
                    6
                )
            )

            print(
                "lr:",
                scheduler.get_last_lr()[0]
            )

            print(
                "VRAM allocated:",
                round(
                    torch.cuda
                    .memory_allocated()
                    / 1024**3,
                    2
                ),
                "GB"
            )

            print(
                "VRAM peak:",
                round(
                    torch.cuda
                    .max_memory_allocated()
                    / 1024**3,
                    2
                ),
                "GB"
            )

            print(
                "================================"
            )


            # smoke test에서는
            # optimizer step마다 저장
            save_smoke_checkpoint(
                optimizer_step,
                micro_idx + 1,
                avg_block_loss,
            )


            block_loss = 0.0


        else:

            print(
                f"[{micro_idx + 1:02d}"
                f"/{NUM_SAMPLES}]",
                f"id={row['id']}",
                f"loss="
                f"{raw_loss_value:.4f}"
            )


    finally:

        if outputs is not None:
            del outputs

        if loss is not None:
            del loss

        if labels is not None:
            del labels

        if inputs is not None:
            del inputs


        gc.collect()
        torch.cuda.empty_cache()


# ============================================================
# 10. 학습 후 LoRA weight 변화 확인
# ============================================================

A_after = (
    first_A[1]
    .detach()
    .float()
    .cpu()
)

B_after = (
    first_B[1]
    .detach()
    .float()
    .cpu()
)


A_delta = (
    A_after
    - A_before
).abs().mean().item()


B_delta = (
    B_after
    - B_before
).abs().mean().item()


elapsed = (
    time.time()
    - start_time
)


print(
    "\n================================"
)
print(
    "H-3 MINI TRAIN FINISHED"
)
print(
    "================================"
)

print(
    "optimizer steps:",
    optimizer_step
)

print(
    "overall avg loss:",
    round(
        running_loss
        / NUM_SAMPLES,
        6
    )
)

print(
    "LoRA A mean delta:",
    A_delta
)

print(
    "LoRA B mean delta:",
    B_delta
)

print(
    "peak VRAM:",
    round(
        torch.cuda
        .max_memory_allocated()
        / 1024**3,
        2
    ),
    "GB"
)

print(
    "elapsed:",
    round(
        elapsed / 60,
        2
    ),
    "min"
)

print(
    "checkpoint dir:",
    SMOKE_DIR
)

print(
    "\n✅ H-3 MINI TRAIN SUCCESS"
)

SAVE DIR: /content/drive/MyDrive/ssafy_vqa_qwen38/runs/20260921_080822/h3_smoke32
✅ Triton backward patch ready
gradient checkpointing: True
Triton base eval: 400

===== OPTIMIZER =====
samples: 32
grad accumulation: 16
optimizer steps: 2
LR: 0.0001

A: base_model.model.model.language_model.layers.0.linear_attn.out_proj.lora_A.default.weight
B: base_model.model.model.language_model.layers.0.linear_attn.out_proj.lora_B.default.weight

H-3 MINI TRAIN START

[01/32] id=train_0001.jpg loss=1.0893
[02/32] id=train_0002.jpg loss=1.2949
[03/32] id=train_0003.jpg loss=2.6558
[04/32] id=train_0004.jpg loss=1.0941
[05/32] id=train_0005.jpg loss=0.5607
[06/32] id=train_0006.jpg loss=3.1897
[07/32] id=train_0007.jpg loss=1.4133
[08/32] id=train_0008.jpg loss=0.6451
[09/32] id=train_0009.jpg loss=0.9514
[10/32] id=train_0010.jpg loss=1.0212
[11/32] id=train_0011.jpg loss=1.3547
[12/32] id=train_0012.jpg loss=0.3882
[13/32] id=train_0013.jpg loss=0.9867
[14/32] id=train_0014.jpg loss=4.0389
[15/32] 

In [ ]:
# ============================================================
# [H-4] CHECKPOINT RELOAD TEST
#
# 검증 대상:
# 1. checkpoint-step-002 파일 존재
# 2. LoRA adapter checkpoint 읽기
# 3. 현재 LoRA와 checkpoint가 동일한지 확인
# 4. 일부 LoRA weight를 일부러 망가뜨린 뒤 checkpoint로 복원
# 5. optimizer state 복원
# 6. scheduler state 복원
# 7. resume metadata 검증
#
# 주의:
# - 본 학습 재개가 아니라 "저장/복원 기능 검증"
# - H-3 scheduler는 이미 2/2 step 끝나서 LR=0
# ============================================================

import json
import torch
from pathlib import Path

from peft.utils.save_and_load import (
    load_peft_weights,
    get_peft_model_state_dict,
    set_peft_model_state_dict,
)

from transformers import get_cosine_schedule_with_warmup


# ============================================================
# 0. checkpoint 위치
# ============================================================

CKPT_DIR = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38/"
    "runs/20260921_080822/"
    "h3_smoke32/"
    "checkpoint-step-002"
)

print("CHECKPOINT:", CKPT_DIR)

assert CKPT_DIR.exists(), (
    f"❌ checkpoint 없음: {CKPT_DIR}"
)


required_files = [
    "adapter_model.safetensors",
    "adapter_config.json",
    "trainer_state.pt",
    "run_meta.json",
]


print("\n===== FILE CHECK =====")

for filename in required_files:

    path = CKPT_DIR / filename

    print(
        filename,
        "->",
        path.exists(),
        (
            f"{path.stat().st_size / 1024**2:.2f} MB"
            if path.exists()
            else ""
        ),
    )

    assert path.exists()


print("\n✅ checkpoint files OK")


# ============================================================
# 1. checkpoint metadata 확인
# ============================================================

with open(
    CKPT_DIR / "run_meta.json",
    "r",
    encoding="utf-8",
) as f:

    run_meta = json.load(f)


print("\n===== RUN META =====")

print(
    json.dumps(
        run_meta,
        ensure_ascii=False,
        indent=2,
    )
)


assert (
    run_meta["optimizer_step"]
    == 2
)

assert (
    run_meta["micro_step"]
    == 32
)


# ============================================================
# 2. 저장된 LoRA adapter 읽기
# ============================================================

print("\n===== LOAD ADAPTER WEIGHTS =====")


ckpt_adapter = load_peft_weights(
    str(CKPT_DIR),
    device="cpu",
)


print(
    "checkpoint adapter tensors:",
    len(ckpt_adapter)
)


assert len(
    ckpt_adapter
) > 0


# 현재 모델 LoRA state
current_adapter = (
    get_peft_model_state_dict(
        model
    )
)


current_adapter = {
    k:
    v.detach()
    .cpu()
    .clone()

    for k, v
    in current_adapter.items()
}


print(
    "current adapter tensors:",
    len(current_adapter)
)


# ============================================================
# 3. key 완전 일치 확인
# ============================================================

ckpt_keys = set(
    ckpt_adapter.keys()
)

current_keys = set(
    current_adapter.keys()
)


missing_in_current = (
    ckpt_keys
    - current_keys
)

extra_in_current = (
    current_keys
    - ckpt_keys
)


print("\n===== KEY CHECK =====")

print(
    "missing in current:",
    len(missing_in_current)
)

print(
    "extra in current:",
    len(extra_in_current)
)


if missing_in_current:

    print(
        "missing sample:",
        list(
            missing_in_current
        )[:5]
    )


if extra_in_current:

    print(
        "extra sample:",
        list(
            extra_in_current
        )[:5]
    )


assert (
    ckpt_keys
    == current_keys
)


print(
    "✅ adapter keys identical"
)


# ============================================================
# 4. 현재 모델 vs checkpoint 값 비교
# ============================================================

max_diff_before = 0.0

worst_key_before = None


for key in ckpt_keys:

    a = (
        current_adapter[key]
        .float()
    )

    b = (
        ckpt_adapter[key]
        .float()
    )


    diff = (
        a - b
    ).abs().max().item()


    if (
        diff
        > max_diff_before
    ):

        max_diff_before = diff

        worst_key_before = key


print("\n===== BEFORE RELOAD =====")

print(
    "max abs diff:",
    max_diff_before
)

print(
    "worst key:",
    worst_key_before
)


# H-3 직후니까 보통 정확히 0
assert (
    max_diff_before
    < 1e-6
)


print(
    "✅ current model == checkpoint"
)


# ============================================================
# 5. 실제 reload 기능 검증
#
# LoRA tensor 하나를 일부러 변경한 뒤
# checkpoint를 다시 적용해서 복원되는지 확인
# ============================================================

print("\n===== DESTRUCTIVE RESTORE TEST =====")


target_name = None
target_param = None


for name, param in (
    model.named_parameters()
):

    if (
        param.requires_grad
        and ".lora_B."
        in name
    ):

        target_name = name
        target_param = param

        break


assert (
    target_param
    is not None
)


print(
    "test parameter:",
    target_name
)


# 원본 backup
backup_tensor = (
    target_param
    .detach()
    .clone()
)


try:

    # ----------------------------------------
    # 일부러 checkpoint와 다른 값으로 변경
    # ----------------------------------------

    with torch.no_grad():

        target_param.add_(
            1.0
        )


    corrupted_delta = (
        target_param.detach()
        - backup_tensor
    ).abs().mean().item()


    print(
        "corrupted mean delta:",
        corrupted_delta
    )


    assert (
        corrupted_delta
        > 0.9
    )


    # ----------------------------------------
    # checkpoint adapter 실제 reload
    # ----------------------------------------

    load_result = (
        set_peft_model_state_dict(
            model,
            ckpt_adapter,
            adapter_name="default",
        )
    )


    unexpected = getattr(
        load_result,
        "unexpected_keys",
        []
    )


    print(
        "unexpected keys:",
        len(unexpected)
    )


    if unexpected:

        print(
            unexpected[:10]
        )


    assert (
        len(unexpected)
        == 0
    )


    # ----------------------------------------
    # 복원 후 전체 LoRA 다시 비교
    # ----------------------------------------

    restored_adapter = (
        get_peft_model_state_dict(
            model
        )
    )


    max_diff_after = 0.0
    worst_key_after = None


    for key in ckpt_keys:

        a = (
            restored_adapter[key]
            .detach()
            .cpu()
            .float()
        )

        b = (
            ckpt_adapter[key]
            .float()
        )


        diff = (
            a - b
        ).abs().max().item()


        if (
            diff
            > max_diff_after
        ):

            max_diff_after = diff

            worst_key_after = key


    print(
        "\nrestored max abs diff:",
        max_diff_after
    )

    print(
        "worst restored key:",
        worst_key_after
    )


    assert (
        max_diff_after
        < 1e-6
    )


finally:

    # 혹시 중간 assert/error가 나더라도
    # checkpoint state를 다시 적용
    set_peft_model_state_dict(
        model,
        ckpt_adapter,
        adapter_name="default",
    )


print(
    "\n✅ LoRA destructive reload SUCCESS"
)


# ============================================================
# 6. trainer_state 읽기
# ============================================================

trainer_state = torch.load(
    CKPT_DIR
    / "trainer_state.pt",

    map_location="cpu",

    weights_only=False,
)


print("\n===== TRAINER STATE =====")

print(
    "optimizer_step:",
    trainer_state[
        "optimizer_step"
    ]
)

print(
    "micro_step:",
    trainer_state[
        "micro_step"
    ]
)

print(
    "avg_loss:",
    trainer_state[
        "avg_loss"
    ]
)


assert (
    trainer_state[
        "optimizer_step"
    ]
    == 2
)

assert (
    trainer_state[
        "micro_step"
    ]
    == 32
)


# ============================================================
# 7. optimizer 새로 만든 뒤 state reload
# ============================================================

reload_trainable_params = [
    p

    for p
    in model.parameters()

    if p.requires_grad
]


optimizer_reload = (
    torch.optim.AdamW(
        reload_trainable_params,

        lr=1e-4,

        betas=(
            0.9,
            0.999,
        ),

        eps=1e-8,

        weight_decay=0.01,
    )
)


optimizer_reload.load_state_dict(
    trainer_state[
        "optimizer"
    ]
)


print("\n===== OPTIMIZER RELOAD =====")

print(
    "optimizer state entries:",
    len(
        optimizer_reload.state
    )
)

print(
    "loaded LR:",
    optimizer_reload
    .param_groups[0]["lr"]
)


assert (
    len(
        optimizer_reload.state
    )
    > 0
)


print(
    "✅ optimizer state loaded"
)


# ============================================================
# 8. scheduler 새로 만든 뒤 state reload
# ============================================================

scheduler_reload = (
    get_cosine_schedule_with_warmup(
        optimizer_reload,

        num_warmup_steps=1,

        num_training_steps=2,
    )
)


scheduler_reload.load_state_dict(
    trainer_state[
        "scheduler"
    ]
)


print("\n===== SCHEDULER RELOAD =====")

print(
    "last_epoch:",
    scheduler_reload.last_epoch
)

print(
    "last LR:",
    scheduler_reload
    .get_last_lr()
)


# smoke scheduler는 2/2 종료
# 그래서 LR 0이 정상
assert (
    abs(
        scheduler_reload
        .get_last_lr()[0]
    )
    < 1e-12
)


print(
    "✅ scheduler state loaded"
)


# ============================================================
# 9. 결과를 Drive에 기록
# ============================================================

validation = {

    "checkpoint":
        str(CKPT_DIR),

    "adapter_tensor_count":
        len(
            ckpt_adapter
        ),

    "adapter_max_diff_before":
        max_diff_before,

    "adapter_max_diff_after":
        max_diff_after,

    "optimizer_state_entries":
        len(
            optimizer_reload.state
        ),

    "optimizer_step":
        trainer_state[
            "optimizer_step"
        ],

    "micro_step":
        trainer_state[
            "micro_step"
        ],

    "scheduler_last_epoch":
        scheduler_reload.last_epoch,

    "scheduler_last_lr":
        scheduler_reload
        .get_last_lr()[0],

    "status":
        "SUCCESS",
}


validation_path = (
    CKPT_DIR
    / "reload_validation.json"
)


validation_path.write_text(
    json.dumps(
        validation,
        ensure_ascii=False,
        indent=2,
    ),

    encoding="utf-8",
)


print("\n================================")
print("H-4 CHECKPOINT RELOAD SUCCESS")
print("================================")

print(
    "adapter max diff after:",
    max_diff_after
)

print(
    "optimizer state entries:",
    len(
        optimizer_reload.state
    )
)

print(
    "scheduler last epoch:",
    scheduler_reload.last_epoch
)

print(
    "scheduler last lr:",
    scheduler_reload
    .get_last_lr()[0]
)

print(
    "validation saved:",
    validation_path
)

print(
    "\n✅ 이제 checkpoint 자체는 복구 가능하다고 검증됨"
)

CHECKPOINT: /content/drive/MyDrive/ssafy_vqa_qwen38/runs/20260921_080822/h3_smoke32/checkpoint-step-002

===== FILE CHECK =====
adapter_model.safetensors -> True 415.12 MB
adapter_config.json -> True 0.00 MB
trainer_state.pt -> True 830.66 MB
run_meta.json -> True 0.00 MB

✅ checkpoint files OK

===== RUN META =====
{
  "base_model": "btbtyler09/Qwen3.8-27B-GPTQ-4bit",
  "image_size": 1024,
  "num_samples": 32,
  "grad_accum": 16,
  "lr": 0.0001,
  "optimizer_step": 2,
  "micro_step": 32,
  "avg_loss": 1.0064098434522748
}

===== LOAD ADAPTER WEIGHTS =====
checkpoint adapter tensors: 800
current adapter tensors: 800

===== KEY CHECK =====
missing in current: 0
extra in current: 0
✅ adapter keys identical

===== BEFORE RELOAD =====
max abs diff: 0.0
worst key: None
✅ current model == checkpoint

===== DESTRUCTIVE RESTORE TEST =====
test parameter: base_model.model.model.language_model.layers.0.linear_attn.out_proj.lora_B.default.weight
corrupted mean delta: 1.0
unexpected keys: 0

resto

In [ ]:
# ============================================================
# [H-4B-0] FRESH RUNTIME BOOTSTRAP
#
# 새 Colab 세션에서 제일 먼저 실행
#
# 하는 일:
# 1. Google Drive mount
# 2. 성공했던 패키지 버전 그대로 설치
# 3. data.zip -> /content/data 복구
# 4. 프로젝트 / cache / checkpoint 존재 확인
# 5. GPU / package 버전 확인
#
# 아직 27B 모델은 로드하지 않음
# ============================================================

# ------------------------------------------------------------
# 0. Drive mount
# ------------------------------------------------------------

from google.colab import drive

drive.mount(
    "/content/drive"
)


# ------------------------------------------------------------
# 1. 패키지 설치
#
# 중요:
# torch는 Colab 기본 CUDA 빌드 그대로 사용
# ------------------------------------------------------------

%pip install -q \
    "transformers==5.17.0" \
    "accelerate==1.15.0" \
    "peft==0.21.0" \
    "gptqmodel==7.5.0" \
    "optimum==2.3.0" \
    "einops"


# FLA는 dependency를 건드리지 않도록 따로 설치
%pip install -q --no-deps \
    "fla-core==0.5.2" \
    "flash-linear-attention==0.5.2"


# ------------------------------------------------------------
# 2. imports
# ------------------------------------------------------------

import os
import gc
import json
import zipfile
from pathlib import Path

import torch
import transformers
import accelerate
import peft
import gptqmodel


# ------------------------------------------------------------
# 3. 경로 복구
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38"
)

CACHE_ROOT = (
    PROJECT_ROOT
    / "cache"
)

DATA_ZIP = Path(
    "/content/drive/MyDrive/data.zip"
)

DATA_ROOT = Path(
    "/content/data"
)


CKPT_DIR = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38/"
    "runs/20260921_080822/"
    "h3_smoke32/"
    "checkpoint-step-002"
)


MODEL_ID = (
    "btbtyler09/"
    "Qwen3.8-27B-GPTQ-4bit"
)

IMAGE_SIZE = 1024


# ------------------------------------------------------------
# 4. data 복구
# ------------------------------------------------------------

if not (
    DATA_ROOT
    / "train.csv"
).exists():

    print(
        "📦 /content/data 없음 -> data.zip 압축 해제"
    )

    assert DATA_ZIP.exists(), (
        f"❌ data.zip 없음: {DATA_ZIP}"
    )

    DATA_ROOT.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(
        DATA_ZIP,
        "r"
    ) as zf:

        zf.extractall(
            DATA_ROOT
        )

    print(
        "✅ data unzip 완료"
    )

else:

    print(
        "✅ /content/data 이미 존재"
    )


# ------------------------------------------------------------
# 5. 필수 파일 확인
# ------------------------------------------------------------

required_paths = {

    "train.csv":
        DATA_ROOT
        / "train.csv",

    "dev.csv":
        DATA_ROOT
        / "dev.csv",

    "test.csv":
        DATA_ROOT
        / "test.csv",

    "OCR cache":
        CACHE_ROOT
        / "ocr_cache.json",

    "DINO cache":
        CACHE_ROOT
        / "dino_cache.json",

    "fixed split":
        PROJECT_ROOT
        / "fixed_split_seed42.csv",

    "adapter":
        CKPT_DIR
        / "adapter_model.safetensors",

    "adapter config":
        CKPT_DIR
        / "adapter_config.json",

    "trainer state":
        CKPT_DIR
        / "trainer_state.pt",

    "reload validation":
        CKPT_DIR
        / "reload_validation.json",
}


print(
    "\n===== FILE CHECK ====="
)


all_ok = True


for name, path in (
    required_paths.items()
):

    exists = path.exists()

    print(
        f"{name:20s}",
        "✅"
        if exists
        else "❌",
        path
    )

    if not exists:
        all_ok = False


assert all_ok, (
    "❌ 필수 파일 중 누락된 것이 있음"
)


# ------------------------------------------------------------
# 6. 이전 H-4 검증 결과 확인
# ------------------------------------------------------------

with open(
    CKPT_DIR
    / "reload_validation.json",
    "r",
    encoding="utf-8"
) as f:

    reload_validation = (
        json.load(f)
    )


print(
    "\n===== PREVIOUS H-4 ====="
)

print(
    json.dumps(
        reload_validation,
        ensure_ascii=False,
        indent=2
    )
)


assert (
    reload_validation[
        "status"
    ]
    == "SUCCESS"
)


# ------------------------------------------------------------
# 7. GPU 확인
# ------------------------------------------------------------

assert (
    torch.cuda.is_available()
), "❌ GPU Runtime 아님"


gpu_name = (
    torch.cuda
    .get_device_name(0)
)

gpu_total = (
    torch.cuda
    .get_device_properties(0)
    .total_memory
    / 1024**3
)


print(
    "\n===== GPU ====="
)

print(
    "GPU:",
    gpu_name
)

print(
    "VRAM:",
    round(
        gpu_total,
        2
    ),
    "GB"
)


# ------------------------------------------------------------
# 8. package versions
# ------------------------------------------------------------

print(
    "\n===== VERSIONS ====="
)

print(
    "torch       :",
    torch.__version__
)

print(
    "CUDA        :",
    torch.version.cuda
)

print(
    "transformers:",
    transformers.__version__
)

print(
    "accelerate  :",
    accelerate.__version__
)

print(
    "peft        :",
    peft.__version__
)

print(
    "gptqmodel   :",
    getattr(
        gptqmodel,
        "__version__",
        "unknown"
    )
)


# ------------------------------------------------------------
# 9. FLA import 확인
# ------------------------------------------------------------

try:

    import fla

    print(
        "FLA         :",
        getattr(
            fla,
            "__version__",
            "installed"
        )
    )

    fla_ok = True

except Exception as e:

    print(
        "FLA import ERROR:",
        repr(e)
    )

    fla_ok = False


assert fla_ok


# ------------------------------------------------------------
# 10. 현재 CUDA 상태
# ------------------------------------------------------------

gc.collect()

torch.cuda.empty_cache()


print(
    "\n===== CUDA MEMORY ====="
)

print(
    "allocated:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        2
    ),
    "GB"
)

print(
    "reserved :",
    round(
        torch.cuda.memory_reserved()
        / 1024**3,
        2
    ),
    "GB"
)


print(
    "\n================================"
)

print(
    "✅ H-4B-0 FRESH BOOTSTRAP SUCCESS"
)

print(
    "================================"
)

print(
    "MODEL_ID:",
    MODEL_ID
)

print(
    "CKPT_DIR:",
    CKPT_DIR
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ /content/data 이미 존재

===== FILE CHECK =====
train.csv            ✅ /content/data/train.csv
dev.csv              ✅ /content/data/dev.csv
test.csv             ✅ /content/data/test.csv
OCR cache            ✅ /content/drive/MyDrive/ssafy_vqa_qwen38/cache/ocr_cache.json
DINO cache           ✅ /content/drive/MyDrive/ssafy_vqa_qwen38/cache/dino_cache.json
fixed split          ✅ /content/drive/MyDrive/ssafy_vqa_qwen38/fixed_split_seed42.csv
adapter              ✅ /content/drive/MyDrive/ssafy_vqa_qwen38/runs/20260921_080822/h3_smoke32/checkpoint-step-002/adapter_model.safetensors
adapter config       ✅ /content/drive/MyDrive/ssafy_vqa_qwen38/runs/20260921_080822/h3_smoke32/checkpoint-step-002/adapter_config.json
trainer state        ✅ /content/drive/MyDrive/ssafy_vqa_qwen38/runs/20260921_080822/h3_smoke32/checkpoint-step-002/trainer_state.pt
reload validation    ✅ /

In [ ]:
# ============================================================
# [H-5] FULL TRAINING
# Qwen3.8-27B GPTQ 4bit + LoRA
#
# fixed split:
#   train = 6042
#   valid = 672
#
# micro batch       = 1
# grad accumulation = 16
# epoch             = 1
# LR                = 1e-4
# checkpoint        = every 50 optimizer steps
#
# IMPORTANT:
# - H-0 / H-1 / H-1.5 / H-1.6 / H-2 PREP / H-2B 성공 후 실행
# - TritonV2Linear는 항상 eval
# - LoRA는 train
# - Gradient Checkpointing 유지
# ============================================================

import gc
import os
import json
import math
import time
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from PIL import Image

from transformers import (
    get_cosine_schedule_with_warmup,
)

from peft.utils.save_and_load import (
    load_peft_weights,
    set_peft_model_state_dict,
)


# ============================================================
# 0. CONFIG
# ============================================================

SEED = 42

EPOCHS = 1
GRAD_ACCUM = 16

LR = 1e-4
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

WARMUP_RATIO = 0.05

CHECKPOINT_EVERY = 50

IMAGE_SIZE = 1024


# ------------------------------------------------------------
# 처음 학습할 때는 None
#
# 나중에 중단된 checkpoint에서 재개하려면:
#
# RESUME_CKPT = Path(
#     ".../checkpoint-step-000150"
# )
# ------------------------------------------------------------

# RESUME_CKPT = None
# RESUME_CKPT = Path(
#     "/content/drive/MyDrive/ssafy_vqa_qwen38/"
#     "runs/full_20260921_100843/"
#     "emergency_checkpoint"
# )
RESUME_CKPT = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38/"
    "runs/full_20260921_100843/"
    "checkpoint-step-000350"
)

# ============================================================
# 1. RANDOM SEED
# ============================================================

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# ============================================================
# 2. PROJECT PATH
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38"
)

SPLIT_PATH = (
    PROJECT_ROOT
    / "fixed_split_seed42.csv"
)


assert SPLIT_PATH.exists(), (
    f"❌ split 파일 없음: {SPLIT_PATH}"
)


# ============================================================
# 3. RUN DIRECTORY
# ============================================================

if RESUME_CKPT is None:

    RUN_ID = (
        datetime.now()
        .strftime("%Y%m%d_%H%M%S")
    )

    FULL_RUN_DIR = (
        PROJECT_ROOT
        / "runs"
        / f"full_{RUN_ID}"
    )

else:

    RESUME_CKPT = Path(
        RESUME_CKPT
    )

    assert RESUME_CKPT.exists()

    # checkpoint-step-XXXX의 부모
    FULL_RUN_DIR = (
        RESUME_CKPT.parent
    )


FULL_RUN_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "FULL RUN DIR:",
    FULL_RUN_DIR
)


# ============================================================
# 4. FIXED SPLIT LOAD
#
# fixed_split_seed42.csv 구조:
#   row_idx | split
#
# row_idx = 원본 train.csv의 0-based row index
# ============================================================

split_df = pd.read_csv(
    SPLIT_PATH
)


print(
    "\n===== SPLIT FILE ====="
)

print(
    "columns:",
    split_df.columns.tolist()
)

print(
    "rows:",
    len(split_df)
)


# ------------------------------------------------------------
# 구조 검증
# ------------------------------------------------------------

assert "row_idx" in split_df.columns
assert "split" in split_df.columns

assert len(split_df) == len(train_df)


split_df = split_df[
    [
        "row_idx",
        "split",
    ]
].copy()


split_df["row_idx"] = (
    split_df["row_idx"]
    .astype(int)
)


split_df["split"] = (
    split_df["split"]
    .astype(str)
    .str.strip()
    .str.lower()
)


# row_idx 자체도 정상인지 확인
assert split_df["row_idx"].is_unique

assert (
    split_df["row_idx"].min()
    == 0
)

assert (
    split_df["row_idx"].max()
    == len(train_df) - 1
)


print(
    "split values:",
    split_df["split"]
    .value_counts()
    .to_dict()
)


# ------------------------------------------------------------
# train.csv 원본 행 번호 복구
# ------------------------------------------------------------

work_df = (
    train_df
    .reset_index(drop=True)
    .copy()
)


work_df["row_idx"] = np.arange(
    len(work_df),
    dtype=np.int64,
)


# ------------------------------------------------------------
# row_idx 기준 merge
# ------------------------------------------------------------

merged = work_df.merge(
    split_df,
    on="row_idx",
    how="inner",
    validate="one_to_one",
)


assert len(merged) == 6714


# ------------------------------------------------------------
# train / valid
# ------------------------------------------------------------

train_split = (
    merged[
        merged["split"]
        == "train"
    ]
    .reset_index(drop=True)
)


valid_split = (
    merged[
        merged["split"]
        .isin(
            [
                "valid",
                "val",
                "validation",
            ]
        )
    ]
    .reset_index(drop=True)
)


print(
    "\n===== FIXED SPLIT ====="
)

print(
    "train:",
    len(train_split)
)

print(
    "valid:",
    len(valid_split)
)


assert len(train_split) == 6042
assert len(valid_split) == 672


# ID 중복도 마지막 확인
assert train_split["id"].is_unique
assert valid_split["id"].is_unique


train_ids = set(
    train_split["id"]
)

valid_ids = set(
    valid_split["id"]
)


assert train_ids.isdisjoint(
    valid_ids
)


print(
    "overlap:",
    len(
        train_ids
        & valid_ids
    )
)


print(
    "\n✅ fixed split 정상 복구"
)


# ============================================================
# 5. TRAIN ORDER
#
# 반드시 고정 순서 저장
# resume 시 정확히 같은 sample 순서 사용
# ============================================================

ORDER_PATH = (
    FULL_RUN_DIR
    / "train_order_ids.json"
)


if ORDER_PATH.exists():

    train_order_ids = json.loads(
        ORDER_PATH.read_text(
            encoding="utf-8"
        )
    )

else:

    rng = np.random.RandomState(
        SEED
    )

    permutation = rng.permutation(
        len(train_split)
    )

    train_order_ids = (
        train_split
        .iloc[permutation]["id"]
        .tolist()
    )


    ORDER_PATH.write_text(
        json.dumps(
            train_order_ids,
            ensure_ascii=False,
            indent=2,
        ),

        encoding="utf-8",
    )


assert len(train_order_ids) == 6042
assert len(set(train_order_ids)) == 6042


train_lookup = (
    train_split
    .set_index("id")
)


print(
    "train order:",
    len(train_order_ids)
)


# ============================================================
# 6. ALBUMENTATIONS
#
# geometry / flip / crop 없음
#
# VQA에서 의미를 바꾸지 않도록
# 아주 약한 밝기/노이즈만 사용
# ============================================================

try:

    import albumentations as A

except ImportError:

    raise RuntimeError(
        "\nAlbumentations가 없습니다.\n"
        "새 셀에서 먼저 실행:\n"
        "%pip install -q albumentations\n"
        "그 후 H-5를 다시 실행하세요."
    )


train_aug = A.Compose(
    [
        A.RandomBrightnessContrast(
            brightness_limit=0.05,
            contrast_limit=0.05,
            p=0.20,
        ),

        A.GaussNoise(
            std_range=(0.01, 0.03),
            p=0.10,
        ),
    ]
)


# ============================================================
# 7. LETTERBOX
# ============================================================

def letterbox_image(
    image,
    size=1024,
):

    image = image.convert(
        "RGB"
    )

    w, h = image.size


    scale = min(
        size / w,
        size / h,
    )


    nw = max(
        1,
        round(
            w * scale
        )
    )

    nh = max(
        1,
        round(
            h * scale
        )
    )


    image = image.resize(
        (nw, nh),
        Image.Resampling.LANCZOS,
    )


    canvas = Image.new(
        "RGB",
        (size, size),
        (0, 0, 0),
    )


    canvas.paste(
        image,
        (
            (size - nw) // 2,
            (size - nh) // 2,
        ),
    )


    return canvas


# ============================================================
# 8. TRAIN IMAGE AUGMENTATION
# ============================================================

def load_train_image(path):

    image = Image.open(
        path
    ).convert("RGB")


    arr = np.array(
        image
    )


    arr = train_aug(
        image=arr
    )["image"]


    image = Image.fromarray(
        arr
    )


    return letterbox_image(
        image,
        IMAGE_SIZE,
    )


def load_eval_image(path):

    image = Image.open(
        path
    )

    return letterbox_image(
        image,
        IMAGE_SIZE,
    )


# ============================================================
# 9. ABCD TOKEN
# ============================================================

tokenizer = (
    processor.tokenizer
)


CHOICE_IDS = {}


for letter in "abcd":

    ids = tokenizer.encode(
        letter,
        add_special_tokens=False,
    )

    assert len(ids) == 1

    CHOICE_IDS[
        letter
    ] = ids[0]


print(
    "\nCHOICE IDS:",
    CHOICE_IDS
)


# ============================================================
# 10. SAMPLE -> INPUT
# ============================================================

def build_sample_inputs(
    row,
    training=True,
    include_label=True,
):

    if training:

        image = load_train_image(
            row["path"]
        )

    else:

        image = load_eval_image(
            row["path"]
        )


    ref = get_reference(
        row["path"]
    )


    system_text = (
        make_system_prompt(
            ref
        )
    )


    user_text = (
        build_mc_prompt(
            row["question"],
            row["a"],
            row["b"],
            row["c"],
            row["d"],
        )
    )


    messages = [
        {
            "role": "system",

            "content": [
                {
                    "type": "text",
                    "text": system_text,
                }
            ],
        },

        {
            "role": "user",

            "content": [
                {
                    "type": "image",
                    "image": image,
                },

                {
                    "type": "text",
                    "text": user_text,
                },
            ],
        },
    ]


    inputs = (
        processor.apply_chat_template(
            messages,

            tokenize=True,

            add_generation_prompt=True,

            enable_thinking=False,

            return_dict=True,

            return_tensors="pt",
        )
    )


    # --------------------------------------------------------
    # inference면 answer token append 안 함
    # --------------------------------------------------------

    if not include_label:

        return inputs


    answer = str(
        row["answer"]
    ).strip().lower()


    assert answer in {
        "a",
        "b",
        "c",
        "d",
    }


    answer_id = (
        CHOICE_IDS[
            answer
        ]
    )


    answer_tensor = torch.tensor(
        [[answer_id]],

        dtype=inputs[
            "input_ids"
        ].dtype,
    )


    # input_ids
    inputs[
        "input_ids"
    ] = torch.cat(
        [
            inputs[
                "input_ids"
            ],

            answer_tensor,
        ],

        dim=1,
    )


    # attention mask
    inputs[
        "attention_mask"
    ] = torch.cat(
        [
            inputs[
                "attention_mask"
            ],

            torch.ones(
                (1, 1),

                dtype=inputs[
                    "attention_mask"
                ].dtype,
            ),
        ],

        dim=1,
    )


    # multimodal token type
    if (
        "mm_token_type_ids"
        in inputs
    ):

        inputs[
            "mm_token_type_ids"
        ] = torch.cat(
            [
                inputs[
                    "mm_token_type_ids"
                ],

                torch.zeros(
                    (1, 1),

                    dtype=inputs[
                        "mm_token_type_ids"
                    ].dtype,
                ),
            ],

            dim=1,
        )


    labels = torch.full_like(
        inputs[
            "input_ids"
        ],

        -100,
    )


    labels[
        0,
        -1
    ] = answer_id


    return (
        inputs,
        labels,
    )


# ============================================================
# 11. CORE MODEL
# ============================================================

if hasattr(
    model,
    "get_base_model"
):

    core_model = (
        model.get_base_model()
    )

else:

    core_model = model


core_model.config.use_cache = False

if hasattr(
    model,
    "config"
):

    model.config.use_cache = False


assert (
    core_model.is_gradient_checkpointing
    is True
)


# ============================================================
# 12. TRAIN MODE HELPER
#
# 절대 raw model.train()만 사용하지 않기
# ============================================================

def set_gptq_lora_train_mode():

    model.train()


    quant_count = 0


    for module in (
        model.modules()
    ):

        if (
            module.__class__.__name__
            == "TritonV2Linear"
        ):

            module.eval()

            quant_count += 1


    still_training = sum(
        1

        for module in model.modules()

        if (
            module.__class__.__name__
            == "TritonV2Linear"
            and module.training
        )
    )


    lora_total = sum(
        1

        for module in model.modules()

        if hasattr(
            module,
            "lora_A"
        )
    )


    lora_train = sum(
        1

        for module in model.modules()

        if (
            hasattr(
                module,
                "lora_A"
            )
            and module.training
        )
    )


    assert quant_count == 400
    assert still_training == 0

    assert lora_total == 400
    assert lora_train == 400

    assert (
        core_model
        .is_gradient_checkpointing
        is True
    )


    return (
        quant_count,
        lora_total,
    )


quant_count, lora_count = (
    set_gptq_lora_train_mode()
)


print(
    "\n===== TRAIN MODE ====="
)

print(
    "Triton eval:",
    quant_count
)

print(
    "LoRA train:",
    lora_count
)

print(
    "GC:",
    core_model
    .is_gradient_checkpointing
)


# ============================================================
# 13. TRAINABLE PARAMS
# ============================================================

trainable_params = [
    p

    for p in model.parameters()

    if p.requires_grad
]


trainable_num = sum(
    p.numel()

    for p in trainable_params
)


print(
    "trainable params:",
    f"{trainable_num:,}"
)


assert (
    trainable_num
    == 108_789_760
)


# ============================================================
# 14. OPTIMIZER / SCHEDULER
# ============================================================

TOTAL_MICRO_STEPS = (
    len(train_order_ids)
    * EPOCHS
)


TOTAL_OPT_STEPS = math.ceil(
    TOTAL_MICRO_STEPS
    / GRAD_ACCUM
)


WARMUP_STEPS = max(
    1,
    round(
        TOTAL_OPT_STEPS
        * WARMUP_RATIO
    ),
)


optimizer = torch.optim.AdamW(
    trainable_params,

    lr=LR,

    betas=(
        0.9,
        0.999,
    ),

    eps=1e-8,

    weight_decay=
        WEIGHT_DECAY,
)


scheduler = (
    get_cosine_schedule_with_warmup(
        optimizer,

        num_warmup_steps=
            WARMUP_STEPS,

        num_training_steps=
            TOTAL_OPT_STEPS,
    )
)


print(
    "\n===== TRAIN CONFIG ====="
)

print(
    "train samples:",
    len(train_order_ids)
)

print(
    "valid samples:",
    len(valid_split)
)

print(
    "epochs:",
    EPOCHS
)

print(
    "grad accum:",
    GRAD_ACCUM
)

print(
    "optimizer steps:",
    TOTAL_OPT_STEPS
)

print(
    "warmup steps:",
    WARMUP_STEPS
)

print(
    "LR:",
    LR
)


# ============================================================
# 15. RUN CONFIG SAVE
# ============================================================

RUN_CONFIG = {

    "model":
        MODEL_ID,

    "image_size":
        IMAGE_SIZE,

    "seed":
        SEED,

    "train_samples":
        len(train_order_ids),

    "valid_samples":
        len(valid_split),

    "epochs":
        EPOCHS,

    "grad_accum":
        GRAD_ACCUM,

    "lr":
        LR,

    "weight_decay":
        WEIGHT_DECAY,

    "warmup_ratio":
        WARMUP_RATIO,

    "warmup_steps":
        WARMUP_STEPS,

    "optimizer_steps":
        TOTAL_OPT_STEPS,

    "checkpoint_every":
        CHECKPOINT_EVERY,

    "lora_trainable_params":
        trainable_num,

    "gradient_checkpointing":
        True,

    "triton_backward_patch":
        "8 gradient return fix",

    "augmentation": {
        "RandomBrightnessContrast":
            {
                "limit": 0.05,
                "p": 0.20,
            },

        "GaussNoise":
            {
                "std_range":
                    [0.01, 0.03],

                "p":
                    0.10,
            },
    },
}


(
    FULL_RUN_DIR
    / "run_config.json"
).write_text(
    json.dumps(
        RUN_CONFIG,
        ensure_ascii=False,
        indent=2,
    ),

    encoding="utf-8",
)


# ============================================================
# 16. CHECKPOINT SAVE
# ============================================================

def save_checkpoint(
    opt_step,
    micro_step,
    epoch,
    running_loss,
):

    ckpt_dir = (
        FULL_RUN_DIR
        / f"checkpoint-step-{opt_step:06d}"
    )


    ckpt_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    print(
        "\n💾 SAVING:",
        ckpt_dir
    )


    # --------------------------------------------------------
    # LoRA adapter
    # --------------------------------------------------------

    model.save_pretrained(
        ckpt_dir,

        safe_serialization=True,
    )


    # --------------------------------------------------------
    # full training state
    # --------------------------------------------------------

    state = {

        "optimizer":
            optimizer.state_dict(),

        "scheduler":
            scheduler.state_dict(),

        "optimizer_step":
            opt_step,

        "micro_step":
            micro_step,

        "epoch":
            epoch,

        "running_loss":
            running_loss,

        "python_rng":
            random.getstate(),

        "numpy_rng":
            np.random.get_state(),

        "torch_rng":
            torch.get_rng_state(),

        "cuda_rng":
            torch.cuda.get_rng_state_all(),
    }


    torch.save(
        state,

        ckpt_dir
        / "trainer_state.pt",
    )


    meta = {

        "optimizer_step":
            opt_step,

        "micro_step":
            micro_step,

        "epoch":
            epoch,

        "avg_loss":
            (
                running_loss
                / max(
                    micro_step,
                    1,
                )
            ),

        "lr":
            scheduler
            .get_last_lr()[0],

        "train_total":
            len(train_order_ids),

        "status":
            "TRAINING",
    }


    (
        ckpt_dir
        / "checkpoint_meta.json"
    ).write_text(
        json.dumps(
            meta,
            ensure_ascii=False,
            indent=2,
        ),

        encoding="utf-8",
    )


    print(
        "✅ checkpoint saved"
    )


    return ckpt_dir


# ============================================================
# 17. RESUME
# ============================================================

start_micro_step = 0
optimizer_step = 0
running_loss = 0.0


if RESUME_CKPT is not None:

    print(
        "\n===== RESUME ====="
    )


    saved_adapter = (
        load_peft_weights(
            str(
                RESUME_CKPT
            ),

            device="cpu",
        )
    )


    load_result = (
        set_peft_model_state_dict(
            model,

            saved_adapter,

            adapter_name="default",
        )
    )


    unexpected = getattr(
        load_result,
        "unexpected_keys",
        [],
    )


    assert (
        len(unexpected)
        == 0
    )


    state = torch.load(
        RESUME_CKPT
        / "trainer_state.pt",

        map_location="cpu",

        weights_only=False,
    )


    optimizer.load_state_dict(
        state[
            "optimizer"
        ]
    )


    scheduler.load_state_dict(
        state[
            "scheduler"
        ]
    )


    start_micro_step = int(
        state[
            "micro_step"
        ]
    )


    optimizer_step = int(
        state[
            "optimizer_step"
        ]
    )


    running_loss = float(
        state[
            "running_loss"
        ]
    )


    # RNG restore
    random.setstate(
        state[
            "python_rng"
        ]
    )


    np.random.set_state(
        state[
            "numpy_rng"
        ]
    )


    torch.set_rng_state(
        state[
            "torch_rng"
        ]
    )


    torch.cuda.set_rng_state_all(
        state[
            "cuda_rng"
        ]
    )


    print(
        "resume micro:",
        start_micro_step
    )

    print(
        "resume optimizer:",
        optimizer_step
    )

    print(
        "resume LR:",
        scheduler
        .get_last_lr()[0]
    )


    set_gptq_lora_train_mode()


# ============================================================
# 18. LOG
# ============================================================

LOG_PATH = (
    FULL_RUN_DIR
    / "train_log.csv"
)


if (
    not LOG_PATH.exists()
    or start_micro_step == 0
):

    pd.DataFrame(
        columns=[
            "optimizer_step",
            "micro_step",
            "block_loss",
            "running_avg_loss",
            "lr",
            "allocated_gb",
            "peak_gb",
            "elapsed_min",
        ]
    ).to_csv(
        LOG_PATH,
        index=False,
    )


# ============================================================
# 19. TRAIN START
# ============================================================

set_gptq_lora_train_mode()


optimizer.zero_grad(
    set_to_none=True
)

model.zero_grad(
    set_to_none=True
)


gc.collect()

torch.cuda.empty_cache()

torch.cuda.reset_peak_memory_stats()


print(
    "\n========================================"
)

print(
    "H-5 FULL TRAIN START"
)

print(
    "========================================"
)

print(
    "starting micro:",
    start_micro_step
)

print(
    "starting opt:",
    optimizer_step
)

print(
    "VRAM:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        2,
    ),
    "GB"
)


train_start_time = (
    time.time()
)


block_loss_sum = 0.0
block_count = 0


# ============================================================
# 20. TRAIN LOOP
# ============================================================

try:

    for micro_idx in range(
        start_micro_step,
        TOTAL_MICRO_STEPS,
    ):

        # 현재는 1 epoch
        epoch = (
            micro_idx
            // len(train_order_ids)
        )


        local_idx = (
            micro_idx
            % len(train_order_ids)
        )


        sample_id = (
            train_order_ids[
                local_idx
            ]
        )


        row = (
            train_lookup
            .loc[
                sample_id
            ]
        )


        inputs = None
        labels = None
        outputs = None
        loss = None
        raw_loss = None


        try:

            inputs, labels = (
                build_sample_inputs(
                    row,

                    training=True,

                    include_label=True,
                )
            )


            inputs = inputs.to(
                "cuda"
            )


            labels = labels.to(
                "cuda"
            )


            # ----------------------------------------------
            # 현재 accumulation block 크기
            #
            # 마지막 block은 10개라
            # /16하지 않고 /10 처리
            # ----------------------------------------------

            block_start = (
                local_idx
                // GRAD_ACCUM
            ) * GRAD_ACCUM


            block_end = min(
                block_start
                + GRAD_ACCUM,

                len(
                    train_order_ids
                ),
            )


            accum_divisor = (
                block_end
                - block_start
            )


            # ----------------------------------------------
            # forward
            # ----------------------------------------------

            with torch.autocast(
                device_type="cuda",

                dtype=
                    torch.bfloat16,
            ):

                outputs = model(
                    **inputs,

                    labels=labels,
                )


                raw_loss = (
                    outputs.loss
                )


                loss = (
                    raw_loss
                    / accum_divisor
                )


            raw_loss_value = float(
                raw_loss
                .detach()
                .cpu()
            )


            running_loss += (
                raw_loss_value
            )


            block_loss_sum += (
                raw_loss_value
            )


            block_count += 1


            # ----------------------------------------------
            # backward
            # ----------------------------------------------

            loss.backward()


            # ----------------------------------------------
            # optimizer step?
            # ----------------------------------------------

            is_block_end = (
                (
                    local_idx + 1
                )
                % GRAD_ACCUM
                == 0
            )


            is_epoch_end = (
                local_idx + 1
                == len(
                    train_order_ids
                )
            )


            if (
                is_block_end
                or is_epoch_end
            ):

                # gradient clipping
                grad_norm = (
                    torch.nn.utils
                    .clip_grad_norm_(
                        trainable_params,

                        MAX_GRAD_NORM,
                    )
                )


                optimizer.step()

                scheduler.step()


                optimizer.zero_grad(
                    set_to_none=True
                )

                model.zero_grad(
                    set_to_none=True
                )


                optimizer_step += 1


                block_avg_loss = (
                    block_loss_sum
                    / block_count
                )


                total_done = (
                    micro_idx + 1
                )


                running_avg = (
                    running_loss
                    / total_done
                )


                elapsed = (
                    time.time()
                    - train_start_time
                )


                allocated_gb = (
                    torch.cuda
                    .memory_allocated()
                    / 1024**3
                )


                peak_gb = (
                    torch.cuda
                    .max_memory_allocated()
                    / 1024**3
                )


                current_lr = (
                    scheduler
                    .get_last_lr()[0]
                )


                print(
                    f"\n[OPT "
                    f"{optimizer_step:03d}"
                    f"/{TOTAL_OPT_STEPS}]"
                )

                print(
                    f"micro      : "
                    f"{total_done}"
                    f"/{TOTAL_MICRO_STEPS}"
                )

                print(
                    "block loss :",
                    round(
                        block_avg_loss,
                        5,
                    )
                )

                print(
                    "avg loss   :",
                    round(
                        running_avg,
                        5,
                    )
                )

                print(
                    "grad norm  :",
                    round(
                        float(
                            grad_norm
                            .detach()
                            .cpu()
                        ),
                        4,
                    )
                )

                print(
                    "lr         :",
                    f"{current_lr:.8f}"
                )

                print(
                    "VRAM       :",
                    round(
                        allocated_gb,
                        2,
                    ),
                    "GB"
                )

                print(
                    "peak       :",
                    round(
                        peak_gb,
                        2,
                    ),
                    "GB"
                )

                print(
                    "elapsed    :",
                    round(
                        elapsed
                        / 60,
                        1,
                    ),
                    "min"
                )


                # ------------------------------------------
                # CSV log
                # ------------------------------------------

                pd.DataFrame(
                    [
                        {
                            "optimizer_step":
                                optimizer_step,

                            "micro_step":
                                total_done,

                            "block_loss":
                                block_avg_loss,

                            "running_avg_loss":
                                running_avg,

                            "lr":
                                current_lr,

                            "allocated_gb":
                                allocated_gb,

                            "peak_gb":
                                peak_gb,

                            "elapsed_min":
                                elapsed / 60,
                        }
                    ]
                ).to_csv(
                    LOG_PATH,

                    mode="a",

                    header=False,

                    index=False,
                )


                # ------------------------------------------
                # checkpoint every 50 optimizer steps
                # ------------------------------------------

                if (
                    optimizer_step
                    % CHECKPOINT_EVERY
                    == 0
                ):

                    save_checkpoint(
                        opt_step=
                            optimizer_step,

                        micro_step=
                            total_done,

                        epoch=
                            epoch,

                        running_loss=
                            running_loss,
                    )



                block_loss_sum = 0.0
                block_count = 0


                # optimizer block 끝에서만
                # cache 정리
                gc.collect()
                torch.cuda.empty_cache()


        finally:

            if outputs is not None:
                del outputs

            if loss is not None:
                del loss

            if raw_loss is not None:
                del raw_loss

            if labels is not None:
                del labels

            if inputs is not None:
                del inputs


# ============================================================
# 21. ERROR -> EMERGENCY CHECKPOINT
# ============================================================

except Exception as e:

    print(
        "\n❌ TRAIN ERROR:"
    )

    print(
        repr(e)
    )


    model.zero_grad(
        set_to_none=True
    )

    optimizer.zero_grad(
        set_to_none=True
    )


    gc.collect()
    torch.cuda.empty_cache()


    emergency_dir = (
        FULL_RUN_DIR
        / "emergency_checkpoint"
    )


    emergency_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    model.save_pretrained(
        emergency_dir,

        safe_serialization=True,
    )


    torch.save(
        {
            "optimizer":
                optimizer.state_dict(),

            "scheduler":
                scheduler.state_dict(),

            "optimizer_step":
                optimizer_step,

            "micro_step":
                micro_idx,

            "running_loss":
                running_loss,

            "python_rng":
                random.getstate(),

            "numpy_rng":
                np.random.get_state(),

            "torch_rng":
                torch.get_rng_state(),

            "cuda_rng":
                torch.cuda.get_rng_state_all(),
        },

        emergency_dir
        / "trainer_state.pt",
    )


    print(
        "💾 emergency checkpoint:",
        emergency_dir
    )


    raise


# ============================================================
# 22. FINAL CHECKPOINT
# ============================================================

FINAL_CKPT = (
    FULL_RUN_DIR
    / "checkpoint-final"
)


FINAL_CKPT.mkdir(
    parents=True,
    exist_ok=True,
)


model.save_pretrained(
    FINAL_CKPT,

    safe_serialization=True,
)


torch.save(
    {
        "optimizer":
            optimizer.state_dict(),

        "scheduler":
            scheduler.state_dict(),

        "optimizer_step":
            optimizer_step,

        "micro_step":
            TOTAL_MICRO_STEPS,

        "epoch":
            EPOCHS,

        "running_loss":
            running_loss,

        "python_rng":
            random.getstate(),

        "numpy_rng":
            np.random.get_state(),

        "torch_rng":
            torch.get_rng_state(),

        "cuda_rng":
            torch.cuda.get_rng_state_all(),
    },

    FINAL_CKPT
    / "trainer_state.pt",
)


print(
    "\n✅ FINAL CHECKPOINT:",
    FINAL_CKPT
)


# ============================================================
# 23. FULL VALIDATION
#
# logits based MCQ
# no generate()
# augmentation 없음
# ============================================================

print(
    "\n========================================"
)

print(
    "FULL VALIDATION START"
)

print(
    "========================================"
)


model.eval()


choice_id_tensor = torch.tensor(
    [
        CHOICE_IDS["a"],
        CHOICE_IDS["b"],
        CHOICE_IDS["c"],
        CHOICE_IDS["d"],
    ],

    device="cuda",

    dtype=torch.long,
)


valid_logits = []
valid_ids = []

correct = 0


validation_start = (
    time.time()
)


with torch.inference_mode():

    for i in range(
        len(valid_split)
    ):

        row = (
            valid_split
            .iloc[i]
        )


        inputs = (
            build_sample_inputs(
                row,

                training=False,

                include_label=False,
            )
        )


        inputs = inputs.to(
            "cuda"
        )


        with torch.autocast(
            device_type="cuda",

            dtype=
                torch.bfloat16,
        ):

            outputs = model(
                **inputs
            )


        # next-token logits
        next_logits = (
            outputs.logits[
                0,
                -1
            ]
        )


        mc_logits = (
            next_logits[
                choice_id_tensor
            ]
            .float()
            .detach()
            .cpu()
            .numpy()
        )


        valid_logits.append(
            mc_logits
        )


        valid_ids.append(
            str(
                row["id"]
            )
        )


        pred_idx = int(
            np.argmax(
                mc_logits
            )
        )


        pred = "abcd"[
            pred_idx
        ]


        answer = str(
            row["answer"]
        ).strip().lower()


        if pred == answer:
            correct += 1


        if (
            (i + 1) % 50
            == 0
            or i + 1
            == len(valid_split)
        ):

            acc = (
                correct
                / (i + 1)
            )


            elapsed = (
                time.time()
                - validation_start
            )


            print(
                f"[VALID "
                f"{i + 1:03d}"
                f"/{len(valid_split)}] "
                f"acc={acc:.4f} "
                f"time={elapsed/60:.1f}m"
            )


        del outputs
        del inputs


        if (
            (i + 1)
            % 50
            == 0
        ):

            gc.collect()
            torch.cuda.empty_cache()


# ============================================================
# 24. SAVE VALID LOGITS + IDS
# ============================================================

valid_logits = np.stack(
    valid_logits,
    axis=0,
)


assert valid_logits.shape == (
    672,
    4,
)


VALID_LOGITS_PATH = (
    FULL_RUN_DIR
    / "valid_logits.npy"
)


np.save(
    VALID_LOGITS_PATH,
    valid_logits,
)


VALID_IDS_PATH = (
    FULL_RUN_DIR
    / "valid_ids.json"
)


VALID_IDS_PATH.write_text(
    json.dumps(
        valid_ids,
        ensure_ascii=False,
        indent=2,
    ),

    encoding="utf-8",
)


valid_accuracy = (
    correct
    / len(valid_split)
)


manifest = {

    "model":
        MODEL_ID,

    "checkpoint":
        str(
            FINAL_CKPT
        ),

    "split":
        "fixed_valid_seed42",

    "n":
        len(valid_ids),

    "logits_shape":
        list(
            valid_logits.shape
        ),

    "choice_order":
        [
            "a",
            "b",
            "c",
            "d",
        ],

    "choice_token_ids":
        CHOICE_IDS,

    "thinking":
        False,

    "tta":
        False,

    "image_size":
        IMAGE_SIZE,

    "accuracy":
        valid_accuracy,
}


(
    FULL_RUN_DIR
    / "valid_manifest.json"
).write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
    ),

    encoding="utf-8",
)


print(
    "\n========================================"
)

print(
    "✅ H-5 FULL TRAIN SUCCESS"
)

print(
    "========================================"
)

print(
    "optimizer steps:",
    optimizer_step,
    "/",
    TOTAL_OPT_STEPS,
)

print(
    "train avg loss:",
    round(
        running_loss
        / TOTAL_MICRO_STEPS,
        6,
    )
)

print(
    "valid accuracy:",
    round(
        valid_accuracy,
        6,
    )
)

print(
    "peak VRAM:",
    round(
        torch.cuda
        .max_memory_allocated()
        / 1024**3,
        2,
    ),
    "GB"
)

print(
    "RUN DIR:",
    FULL_RUN_DIR
)

print(
    "FINAL ADAPTER:",
    FINAL_CKPT
)

print(
    "VALID LOGITS:",
    VALID_LOGITS_PATH
)

print(
    "VALID IDs:",
    VALID_IDS_PATH
)

FULL RUN DIR: /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843

===== SPLIT FILE =====
columns: ['row_idx', 'split']
rows: 6714
split values: {'train': 6042, 'valid': 672}

===== FIXED SPLIT =====
train: 6042
valid: 672
overlap: 0

✅ fixed split 정상 복구
train order: 6042

CHOICE IDS: {'a': 64, 'b': 65, 'c': 66, 'd': 67}

===== TRAIN MODE =====
Triton eval: 400
LoRA train: 400
GC: True
trainable params: 108,789,760

===== TRAIN CONFIG =====
train samples: 6042
valid samples: 672
epochs: 1
grad accum: 16
optimizer steps: 378
warmup steps: 19
LR: 0.0001

===== RESUME =====
resume micro: 800
resume optimizer: 50
resume LR: 9.817144004871128e-05

H-5 FULL TRAIN START
starting micro: 800
starting opt: 50
VRAM: 26.51 GB

[OPT 051/378]
micro      : 816/6042
block loss : 0.02247
avg loss   : 0.1975
grad norm  : 3.1713
lr         : 0.00009805
VRAM       : 27.13 GB
peak       : 35.26 GB
elapsed    : 0.6 min

[OPT 052/378]
micro      : 832/6042
block loss : 0.07473
avg loss   : 0.19

KeyboardInterrupt: 

In [ ]:
# ============================================================
# [H-5 RECOVERY-1]
# emergency checkpoint micro_step off-by-one 수정
# ============================================================

import torch
from pathlib import Path

EMERGENCY_CKPT = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38/"
    "runs/full_20260921_100843/"
    "emergency_checkpoint"
)

STATE_PATH = (
    EMERGENCY_CKPT
    / "trainer_state.pt"
)

state = torch.load(
    STATE_PATH,
    map_location="cpu",
    weights_only=False,
)

print("BEFORE")
print("optimizer_step:", state["optimizer_step"])
print("micro_step    :", state["micro_step"])
print("running_loss  :", state["running_loss"])

assert state["optimizer_step"] == 50
assert state["micro_step"] == 799

# 0-based micro_idx 799
# = 800 samples까지 이미 처리 완료
state["micro_step"] = 800

torch.save(
    state,
    STATE_PATH,
)

# 재확인
check = torch.load(
    STATE_PATH,
    map_location="cpu",
    weights_only=False,
)

print("\nAFTER")
print("optimizer_step:", check["optimizer_step"])
print("micro_step    :", check["micro_step"])

assert check["optimizer_step"] == 50
assert check["micro_step"] == 800

print("\n✅ emergency checkpoint resume position fixed")

BEFORE
optimizer_step: 50
micro_step    : 799
running_loss  : 160.79696450131473

AFTER
optimizer_step: 50
micro_step    : 800

✅ emergency checkpoint resume position fixed


In [ ]:
# ============================================================
# [H-5 STATUS CHECK]
# 학습 / final / validation 저장 상태 확인
# ============================================================

import json
import torch
from pathlib import Path

RUN_DIR = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38/"
    "runs/full_20260921_100843"
)

print("RUN:", RUN_DIR)


# ------------------------------------------------------------
# 1. checkpoints
# ------------------------------------------------------------

print("\n===== CHECKPOINTS =====")

for p in sorted(RUN_DIR.glob("checkpoint-step-*")):
    print(p.name)


# ------------------------------------------------------------
# 2. FINAL checkpoint
# ------------------------------------------------------------

FINAL = RUN_DIR / "checkpoint-final"

print("\n===== FINAL =====")

print("folder:", FINAL.exists())

for name in [
    "adapter_model.safetensors",
    "adapter_config.json",
    "trainer_state.pt",
]:
    path = FINAL / name
    print(name, "✅" if path.exists() else "❌")


# ------------------------------------------------------------
# 3. final trainer state
# ------------------------------------------------------------

state_path = FINAL / "trainer_state.pt"

if state_path.exists():

    state = torch.load(
        state_path,
        map_location="cpu",
        weights_only=False,
    )

    print("\n===== FINAL STATE =====")

    print(
        "optimizer_step:",
        state.get("optimizer_step")
    )

    print(
        "micro_step:",
        state.get("micro_step")
    )

    print(
        "epoch:",
        state.get("epoch")
    )


# ------------------------------------------------------------
# 4. validation output
# ------------------------------------------------------------

print("\n===== VALIDATION FILES =====")

for name in [
    "valid_logits.npy",
    "valid_ids.json",
    "valid_manifest.json",
]:

    path = RUN_DIR / name

    print(
        name,
        "✅" if path.exists() else "❌"
    )


# ------------------------------------------------------------
# 5. validation manifest
# ------------------------------------------------------------

manifest_path = (
    RUN_DIR / "valid_manifest.json"
)

if manifest_path.exists():

    manifest = json.loads(
        manifest_path.read_text(
            encoding="utf-8"
        )
    )

    print("\n===== VALID RESULT =====")

    print(
        "n:",
        manifest.get("n")
    )

    print(
        "shape:",
        manifest.get("logits_shape")
    )

    print(
        "accuracy:",
        manifest.get("accuracy")
    )

    print(
        "thinking:",
        manifest.get("thinking")
    )

    print(
        "TTA:",
        manifest.get("tta")
    )

RUN: /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843

===== CHECKPOINTS =====
checkpoint-step-000100
checkpoint-step-000150
checkpoint-step-000200
checkpoint-step-000250
checkpoint-step-000300
checkpoint-step-000350

===== FINAL =====
folder: True
adapter_model.safetensors ✅
adapter_config.json ✅
trainer_state.pt ✅

===== FINAL STATE =====
optimizer_step: 378
micro_step: 6042
epoch: 1

===== VALIDATION FILES =====
valid_logits.npy ✅
valid_ids.json ✅
valid_manifest.json ✅

===== VALID RESULT =====
n: 672
shape: [672, 4]
accuracy: 0.9434523809523809
thinking: False
TTA: False


In [ ]:
# ============================================================
# [H-5 VERIFY] 저장된 실험 설정 확인
# ============================================================

import json
from pathlib import Path

RUN_DIR = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38/"
    "runs/full_20260921_100843"
)

for filename in [
    "run_config.json",
    "valid_manifest.json",
    "train_order_ids.json",
]:

    path = RUN_DIR / filename

    print("\n" + "=" * 60)
    print(filename)
    print("=" * 60)

    data = json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )

    if filename == "train_order_ids.json":

        print("num ids:", len(data))
        print("first 5:", data[:5])
        print("last 5 :", data[-5:])

    else:

        print(
            json.dumps(
                data,
                ensure_ascii=False,
                indent=2
            )
        )


run_config.json
{
  "model": "btbtyler09/Qwen3.8-27B-GPTQ-4bit",
  "image_size": 1024,
  "seed": 42,
  "train_samples": 6042,
  "valid_samples": 672,
  "epochs": 1,
  "grad_accum": 16,
  "lr": 0.0001,
  "weight_decay": 0.01,
  "warmup_ratio": 0.05,
  "warmup_steps": 19,
  "optimizer_steps": 378,
  "checkpoint_every": 50,
  "lora_trainable_params": 108789760,
  "gradient_checkpointing": true,
  "triton_backward_patch": "8 gradient return fix",
  "augmentation": {
    "RandomBrightnessContrast": {
      "limit": 0.05,
      "p": 0.2
    },
    "GaussNoise": {
      "std_range": [
        0.01,
        0.03
      ],
      "p": 0.1
    }
  }
}

valid_manifest.json
{
  "model": "btbtyler09/Qwen3.8-27B-GPTQ-4bit",
  "checkpoint": "/content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/checkpoint-final",
  "split": "fixed_valid_seed42",
  "n": 672,
  "logits_shape": [
    672,
    4
  ],
  "choice_order": [
    "a",
    "b",
    "c",
    "d"
  ],
  "choice_token_ids": {
    "a": 6

In [ ]:
# ============================================================
# [H-6A] VALIDATION ERROR / CONFIDENCE ANALYSIS
#
# - inference 재실행 없음
# - 기존 valid_logits.npy 사용
# - 오답 38개 분석
# - confidence / margin / entropy
# - answer별 accuracy
# - confusion matrix
# - CSV + JSON 저장
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path


# ============================================================
# 0. PATH
# ============================================================

RUN_DIR = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38/"
    "runs/full_20260921_100843"
)

LOGITS_PATH = RUN_DIR / "valid_logits.npy"
IDS_PATH = RUN_DIR / "valid_ids.json"
MANIFEST_PATH = RUN_DIR / "valid_manifest.json"

ANALYSIS_DIR = RUN_DIR / "analysis"
ANALYSIS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 1. LOAD
# ============================================================

logits = np.load(
    LOGITS_PATH
)

valid_ids = json.loads(
    IDS_PATH.read_text(
        encoding="utf-8"
    )
)

manifest = json.loads(
    MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)


print("logits:", logits.shape)
print("ids   :", len(valid_ids))
print("saved accuracy:", manifest["accuracy"])

assert logits.shape == (672, 4)
assert len(valid_ids) == 672


# ============================================================
# 2. VALID SPLIT 다시 구성
# ============================================================

split_df = pd.read_csv(
    PROJECT_ROOT / "fixed_split_seed42.csv"
)

work_df = (
    train_df
    .reset_index(drop=True)
    .copy()
)

work_df["row_idx"] = np.arange(
    len(work_df)
)

merged = work_df.merge(
    split_df,
    on="row_idx",
    how="inner",
    validate="one_to_one"
)

valid_df = (
    merged[
        merged["split"]
        .astype(str)
        .str.lower()
        .isin([
            "valid",
            "val",
            "validation"
        ])
    ]
    .copy()
)


# ------------------------------------------------------------
# 중요:
# logits 저장 순서는 valid_ids 기준
# ------------------------------------------------------------

valid_lookup = (
    valid_df
    .set_index("id")
)

ordered_valid = (
    valid_lookup
    .loc[valid_ids]
    .reset_index()
)


assert (
    ordered_valid["id"]
    .astype(str)
    .tolist()
    == valid_ids
)


# ============================================================
# 3. LOGITS -> PROBABILITY
# ============================================================

x = (
    logits
    - logits.max(
        axis=1,
        keepdims=True
    )
)

exp_x = np.exp(x)

probs = (
    exp_x
    / exp_x.sum(
        axis=1,
        keepdims=True
    )
)


CHOICES = np.array(
    list("abcd")
)


pred_idx = np.argmax(
    probs,
    axis=1
)

pred = CHOICES[
    pred_idx
]


answer = (
    ordered_valid["answer"]
    .astype(str)
    .str.strip()
    .str.lower()
    .to_numpy()
)


correct = (
    pred == answer
)


# ============================================================
# 4. CONFIDENCE / MARGIN
# ============================================================

sorted_probs = np.sort(
    probs,
    axis=1
)


confidence = (
    sorted_probs[:, -1]
)

second_conf = (
    sorted_probs[:, -2]
)

margin = (
    confidence
    - second_conf
)


# entropy
entropy = -np.sum(
    probs
    * np.log(
        probs + 1e-12
    ),
    axis=1
)


# 정답 probability
answer_idx = np.array([
    "abcd".index(a)
    for a in answer
])

answer_prob = probs[
    np.arange(
        len(probs)
    ),
    answer_idx
]


# ============================================================
# 5. ANALYSIS DATAFRAME
# ============================================================

analysis_df = (
    ordered_valid.copy()
)


analysis_df["pred"] = pred
analysis_df["correct"] = correct

analysis_df["confidence"] = confidence
analysis_df["second_conf"] = second_conf
analysis_df["margin"] = margin
analysis_df["entropy"] = entropy
analysis_df["answer_prob"] = answer_prob


for i, c in enumerate("abcd"):

    analysis_df[
        f"prob_{c}"
    ] = probs[:, i]


# ============================================================
# 6. BASIC RESULT
# ============================================================

accuracy = (
    correct.mean()
)

wrong_df = (
    analysis_df[
        ~analysis_df["correct"]
    ]
    .copy()
)


print("\n================================")
print("H-6A VALID ANALYSIS")
print("================================")

print(
    "accuracy:",
    round(
        float(accuracy),
        6
    )
)

print(
    "correct:",
    int(correct.sum())
)

print(
    "wrong  :",
    int((~correct).sum())
)


assert (
    abs(
        accuracy
        - manifest["accuracy"]
    )
    < 1e-12
)


# ============================================================
# 7. CORRECT vs WRONG confidence
# ============================================================

print(
    "\n===== CONFIDENCE ====="
)

print(
    "correct mean conf:",
    round(
        analysis_df.loc[
            analysis_df["correct"],
            "confidence"
        ].mean(),
        4
    )
)

print(
    "wrong mean conf  :",
    round(
        wrong_df[
            "confidence"
        ].mean(),
        4
    )
)

print(
    "correct mean margin:",
    round(
        analysis_df.loc[
            analysis_df["correct"],
            "margin"
        ].mean(),
        4
    )
)

print(
    "wrong mean margin  :",
    round(
        wrong_df[
            "margin"
        ].mean(),
        4
    )
)


# ============================================================
# 8. ANSWER별 성능
# ============================================================

per_answer = (
    analysis_df
    .groupby("answer")
    .agg(
        n=(
            "correct",
            "size"
        ),

        correct=(
            "correct",
            "sum"
        ),

        accuracy=(
            "correct",
            "mean"
        ),

        avg_confidence=(
            "confidence",
            "mean"
        ),

        avg_margin=(
            "margin",
            "mean"
        ),
    )
)


print(
    "\n===== PER ANSWER ====="
)

print(
    per_answer
)


# ============================================================
# 9. CONFUSION MATRIX
# ============================================================

confusion = pd.crosstab(
    pd.Series(
        answer,
        name="answer"
    ),

    pd.Series(
        pred,
        name="pred"
    ),

    dropna=False
)


confusion = confusion.reindex(
    index=list("abcd"),
    columns=list("abcd"),
    fill_value=0
)


print(
    "\n===== CONFUSION ====="
)

print(
    confusion
)


# ============================================================
# 10. 가장 애매했던 문제
# ============================================================

uncertain = (
    analysis_df
    .sort_values(
        "margin",
        ascending=True
    )
)


print(
    "\n===== LOWEST MARGIN TOP 15 ====="
)

print(
    uncertain[
        [
            "id",
            "answer",
            "pred",
            "correct",
            "confidence",
            "margin",
            "question",
        ]
    ]
    .head(15)
    .to_string(
        index=False
    )
)


# ============================================================
# 11. 가장 자신있게 틀린 문제
# ============================================================

confident_wrong = (
    wrong_df
    .sort_values(
        "confidence",
        ascending=False
    )
)


print(
    "\n===== CONFIDENT WRONG TOP 15 ====="
)

print(
    confident_wrong[
        [
            "id",
            "answer",
            "pred",
            "confidence",
            "margin",
            "answer_prob",
            "question",
        ]
    ]
    .head(15)
    .to_string(
        index=False
    )
)


# ============================================================
# 12. SAVE
# ============================================================

ANALYSIS_CSV = (
    ANALYSIS_DIR
    / "valid_analysis.csv"
)

WRONG_CSV = (
    ANALYSIS_DIR
    / "valid_wrong.csv"
)

CONFUSION_CSV = (
    ANALYSIS_DIR
    / "confusion_matrix.csv"
)

PER_ANSWER_CSV = (
    ANALYSIS_DIR
    / "per_answer.csv"
)


analysis_df.to_csv(
    ANALYSIS_CSV,
    index=False,
    encoding="utf-8-sig"
)

wrong_df.to_csv(
    WRONG_CSV,
    index=False,
    encoding="utf-8-sig"
)

confusion.to_csv(
    CONFUSION_CSV,
    encoding="utf-8-sig"
)

per_answer.to_csv(
    PER_ANSWER_CSV,
    encoding="utf-8-sig"
)


summary = {

    "accuracy":
        float(accuracy),

    "correct":
        int(correct.sum()),

    "wrong":
        int((~correct).sum()),

    "correct_avg_confidence":
        float(
            analysis_df.loc[
                analysis_df["correct"],
                "confidence"
            ].mean()
        ),

    "wrong_avg_confidence":
        float(
            wrong_df[
                "confidence"
            ].mean()
        ),

    "correct_avg_margin":
        float(
            analysis_df.loc[
                analysis_df["correct"],
                "margin"
            ].mean()
        ),

    "wrong_avg_margin":
        float(
            wrong_df[
                "margin"
            ].mean()
        ),
}


SUMMARY_PATH = (
    ANALYSIS_DIR
    / "analysis_summary.json"
)


SUMMARY_PATH.write_text(
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    "\n================================"
)

print(
    "✅ H-6A ANALYSIS SUCCESS"
)

print(
    "================================"
)

print(
    "analysis:",
    ANALYSIS_CSV
)

print(
    "wrong:",
    WRONG_CSV
)

print(
    "summary:",
    SUMMARY_PATH
)

logits: (672, 4)
ids   : 672
saved accuracy: 0.9434523809523809

H-6A VALID ANALYSIS
accuracy: 0.943452
correct: 634
wrong  : 38

===== CONFIDENCE =====
correct mean conf: 0.9839
wrong mean conf  : 0.7673
correct mean margin: 0.9703
wrong mean margin  : 0.5797

===== PER ANSWER =====
          n  correct  accuracy  avg_confidence  avg_margin
answer                                                    
a       170      160  0.941176        0.977317    0.958545
b       164      161  0.981707        0.980503    0.964610
c       172      160  0.930233        0.967797    0.940232
d       166      153  0.921687        0.961215    0.929663

===== CONFUSION =====
pred      a    b    c    d
answer                    
a       160    1    3    6
b         2  161    0    1
c         3    3  160    6
d         5    8    0  153

===== LOWEST MARGIN TOP 15 =====
            id answer pred  correct  confidence   margin                                                question
train_5089.jpg      d    b   

In [ ]:
# ============================================================
# [H-6B] 4-WAY CYCLIC CHOICE TTA - VALIDATION
#
# 목적:
# - FINAL adapter 강제 복원
# - 기존 shift0 logits 재사용
# - shift1,2,3만 추가 inference
# - 각 permutation 결과를 원래 a/b/c/d 의미로 역매핑
# - probability 평균
# - baseline vs TTA accuracy / rescue / harm 분석
#
# permutation:
# shift0: [a,b,c,d]  ← 기존 valid_logits.npy 재사용
# shift1: [b,c,d,a]
# shift2: [c,d,a,b]
# shift3: [d,a,b,c]
# ============================================================

import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from peft.utils.save_and_load import (
    load_peft_weights,
    set_peft_model_state_dict,
)


# ============================================================
# 0. PATH
# ============================================================

RUN_DIR = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38/"
    "runs/full_20260921_100843"
)

FINAL_CKPT = (
    RUN_DIR
    / "checkpoint-final"
)

BASE_LOGITS_PATH = (
    RUN_DIR
    / "valid_logits.npy"
)

VALID_IDS_PATH = (
    RUN_DIR
    / "valid_ids.json"
)

TTA_DIR = (
    RUN_DIR
    / "tta_valid"
)

TTA_DIR.mkdir(
    parents=True,
    exist_ok=True
)


assert FINAL_CKPT.exists()
assert BASE_LOGITS_PATH.exists()
assert VALID_IDS_PATH.exists()

assert "model" in globals()
assert "processor" in globals()


# ============================================================
# 1. ★ FINAL ADAPTER 강제 복원
#
# 현재 RAM의 LoRA가 실수로 재학습된 상태여도
# checkpoint-final로 덮어씀
# ============================================================

print("===== FINAL ADAPTER RESTORE =====")

final_adapter = load_peft_weights(
    str(FINAL_CKPT),
    device="cpu",
)

print(
    "adapter tensors:",
    len(final_adapter)
)

assert len(final_adapter) == 800


load_result = set_peft_model_state_dict(
    model,
    final_adapter,
    adapter_name="default",
)


unexpected = getattr(
    load_result,
    "unexpected_keys",
    [],
)

print(
    "unexpected keys:",
    len(unexpected)
)

assert len(unexpected) == 0


model.eval()

gc.collect()
torch.cuda.empty_cache()


print(
    "VRAM:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        2
    ),
    "GB"
)

print(
    "✅ checkpoint-final restored"
)


# ============================================================
# 2. VALID DATA 정렬 복구
# ============================================================

valid_ids = json.loads(
    VALID_IDS_PATH.read_text(
        encoding="utf-8"
    )
)

assert len(valid_ids) == 672


split_df = pd.read_csv(
    PROJECT_ROOT
    / "fixed_split_seed42.csv"
)


work_df = (
    train_df
    .reset_index(drop=True)
    .copy()
)

work_df["row_idx"] = np.arange(
    len(work_df),
    dtype=np.int64,
)


merged = work_df.merge(
    split_df,
    on="row_idx",
    how="inner",
    validate="one_to_one",
)


valid_df = (
    merged[
        merged["split"]
        .astype(str)
        .str.lower()
        .isin([
            "valid",
            "val",
            "validation",
        ])
    ]
    .copy()
)


valid_lookup = (
    valid_df
    .set_index("id")
)


ordered_valid = (
    valid_lookup
    .loc[valid_ids]
    .reset_index()
)


assert (
    ordered_valid["id"]
    .astype(str)
    .tolist()
    == valid_ids
)


answers = (
    ordered_valid["answer"]
    .astype(str)
    .str.strip()
    .str.lower()
    .to_numpy()
)


# ============================================================
# 3. ABCD token IDs
# ============================================================

tokenizer = processor.tokenizer


CHOICE_IDS = {}

for letter in "abcd":

    ids = tokenizer.encode(
        letter,
        add_special_tokens=False,
    )

    assert len(ids) == 1

    CHOICE_IDS[letter] = ids[0]


choice_id_tensor = torch.tensor(
    [
        CHOICE_IDS["a"],
        CHOICE_IDS["b"],
        CHOICE_IDS["c"],
        CHOICE_IDS["d"],
    ],
    device="cuda",
    dtype=torch.long,
)


# ============================================================
# 4. SOFTMAX
# ============================================================

def np_softmax(x):

    x = (
        x
        - np.max(
            x,
            axis=-1,
            keepdims=True,
        )
    )

    e = np.exp(x)

    return (
        e
        / e.sum(
            axis=-1,
            keepdims=True,
        )
    )


# ============================================================
# 5. SHIFT 정의
#
# perm[j] =
#   현재 prompt의 j번째 letter가
#   원래 어떤 선택지인지
# ============================================================

PERMS = {

    0: [0, 1, 2, 3],

    1: [1, 2, 3, 0],

    2: [2, 3, 0, 1],

    3: [3, 0, 1, 2],
}


# ============================================================
# 6. TTA INPUT
# ============================================================

def build_tta_input(
    row,
    shift,
):

    image = load_eval_image(
        row["path"]
    )


    ref = get_reference(
        row["path"]
    )


    system_text = (
        make_system_prompt(
            ref
        )
    )


    original_options = [
        row["a"],
        row["b"],
        row["c"],
        row["d"],
    ]


    perm = PERMS[shift]


    shown_options = [
        original_options[i]
        for i in perm
    ]


    user_text = build_mc_prompt(
        row["question"],
        shown_options[0],
        shown_options[1],
        shown_options[2],
        shown_options[3],
    )


    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": system_text,
                }
            ],
        },

        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image,
                },

                {
                    "type": "text",
                    "text": user_text,
                },
            ],
        },
    ]


    inputs = processor.apply_chat_template(
        messages,

        tokenize=True,
        add_generation_prompt=True,

        enable_thinking=False,

        return_dict=True,
        return_tensors="pt",
    )


    return inputs


# ============================================================
# 7. 기존 SHIFT 0 재사용
# ============================================================

base_logits = np.load(
    BASE_LOGITS_PATH
).astype(np.float32)


assert base_logits.shape == (
    672,
    4,
)


shift_probs = {}


shift_probs[0] = np_softmax(
    base_logits
)


np.save(
    TTA_DIR
    / "shift0_mapped_probs.npy",

    shift_probs[0]
)


print(
    "\nSHIFT 0 loaded:",
    shift_probs[0].shape
)


# ============================================================
# 8. SHIFT 1 / 2 / 3 INFERENCE
# ============================================================

for shift in [
    1,
    2,
    3,
]:

    print(
        "\n========================================"
    )

    print(
        f"TTA SHIFT {shift} START"
    )

    print(
        "perm:",
        PERMS[shift]
    )

    print(
        "========================================"
    )


    raw_logits_all = []

    mapped_probs_all = []


    start_time = time.time()


    with torch.inference_mode():

        for i in range(
            len(ordered_valid)
        ):

            row = (
                ordered_valid
                .iloc[i]
            )


            inputs = build_tta_input(
                row,
                shift,
            )


            inputs = inputs.to(
                "cuda"
            )


            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
            ):

                outputs = model(
                    **inputs
                )


            # ------------------------------------------------
            # 현재 prompt의 a/b/c/d logits
            # ------------------------------------------------

            next_logits = (
                outputs.logits[
                    0,
                    -1
                ]
            )


            raw_logits = (
                next_logits[
                    choice_id_tensor
                ]
                .float()
                .detach()
                .cpu()
                .numpy()
            )


            raw_logits_all.append(
                raw_logits
            )


            # ------------------------------------------------
            # probability
            # ------------------------------------------------

            raw_probs = np_softmax(
                raw_logits[
                    None,
                    :
                ]
            )[0]


            # ------------------------------------------------
            # 원래 선택지 위치로 복원
            #
            # 예:
            # shift1
            # prompt a = original b
            # prompt b = original c
            # prompt c = original d
            # prompt d = original a
            # ------------------------------------------------

            mapped_probs = np.zeros(
                4,
                dtype=np.float32,
            )


            perm = PERMS[
                shift
            ]


            for shown_idx in range(
                4
            ):

                original_idx = (
                    perm[
                        shown_idx
                    ]
                )

                mapped_probs[
                    original_idx
                ] = raw_probs[
                    shown_idx
                ]


            mapped_probs_all.append(
                mapped_probs
            )


            del outputs
            del inputs


            if (
                (i + 1) % 50
                == 0
                or i + 1
                == len(
                    ordered_valid
                )
            ):

                elapsed = (
                    time.time()
                    - start_time
                )


                print(
                    f"[SHIFT {shift}] "
                    f"{i + 1:03d}/672 "
                    f"time="
                    f"{elapsed/60:.1f}m"
                )


            if (
                (i + 1)
                % 50
                == 0
            ):

                gc.collect()
                torch.cuda.empty_cache()


    raw_logits_all = np.stack(
        raw_logits_all,
        axis=0,
    )


    mapped_probs_all = np.stack(
        mapped_probs_all,
        axis=0,
    )


    assert (
        raw_logits_all.shape
        == (672, 4)
    )

    assert (
        mapped_probs_all.shape
        == (672, 4)
    )


    shift_probs[
        shift
    ] = mapped_probs_all


    # raw prompt-space logits
    np.save(
        TTA_DIR
        / f"shift{shift}_raw_logits.npy",

        raw_logits_all,
    )


    # canonical/original option-space probs
    np.save(
        TTA_DIR
        / f"shift{shift}_mapped_probs.npy",

        mapped_probs_all,
    )


    print(
        f"✅ SHIFT {shift} SAVED"
    )


# ============================================================
# 9. 각 SHIFT 단독 정확도
# ============================================================

CHOICES = np.array(
    list("abcd")
)


shift_accuracy = {}


print(
    "\n===== INDIVIDUAL SHIFT ACCURACY ====="
)


for shift in range(4):

    pred_idx = np.argmax(
        shift_probs[shift],
        axis=1,
    )

    pred = CHOICES[
        pred_idx
    ]


    acc = float(
        np.mean(
            pred == answers
        )
    )


    shift_accuracy[
        shift
    ] = acc


    print(
        f"shift{shift}:",
        round(
            acc,
            6
        )
    )


# ============================================================
# 10. 4-WAY PROBABILITY AVERAGE
# ============================================================

tta_probs = np.mean(
    np.stack(
        [
            shift_probs[0],
            shift_probs[1],
            shift_probs[2],
            shift_probs[3],
        ],
        axis=0,
    ),

    axis=0,
)


assert tta_probs.shape == (
    672,
    4
)


tta_pred_idx = np.argmax(
    tta_probs,
    axis=1,
)


tta_pred = CHOICES[
    tta_pred_idx
]


baseline_pred = CHOICES[
    np.argmax(
        shift_probs[0],
        axis=1,
    )
]


baseline_correct = (
    baseline_pred
    == answers
)

tta_correct = (
    tta_pred
    == answers
)


baseline_acc = float(
    baseline_correct.mean()
)

tta_acc = float(
    tta_correct.mean()
)


# ============================================================
# 11. RESCUE / HARM
# ============================================================

rescue = (
    (~baseline_correct)
    & tta_correct
)

harm = (
    baseline_correct
    & (~tta_correct)
)


rescue_count = int(
    rescue.sum()
)

harm_count = int(
    harm.sum()
)

net = (
    rescue_count
    - harm_count
)


print(
    "\n========================================"
)

print(
    "4-WAY TTA RESULT"
)

print(
    "========================================"
)

print(
    "baseline:",
    round(
        baseline_acc,
        6
    )
)

print(
    "TTA     :",
    round(
        tta_acc,
        6
    )
)

print(
    "delta   :",
    round(
        tta_acc
        - baseline_acc,
        6
    )
)

print(
    "baseline correct:",
    int(
        baseline_correct.sum()
    )
)

print(
    "TTA correct     :",
    int(
        tta_correct.sum()
    )
)

print(
    "rescue:",
    rescue_count
)

print(
    "harm  :",
    harm_count
)

print(
    "net   :",
    net
)


# ============================================================
# 12. TTA CONFIDENCE / MARGIN
# ============================================================

sorted_tta = np.sort(
    tta_probs,
    axis=1,
)


tta_conf = (
    sorted_tta[:, -1]
)

tta_margin = (
    sorted_tta[:, -1]
    - sorted_tta[:, -2]
)


# ============================================================
# 13. DETAIL CSV
# ============================================================

result_df = pd.DataFrame({

    "id":
        valid_ids,

    "answer":
        answers,

    "baseline_pred":
        baseline_pred,

    "tta_pred":
        tta_pred,

    "baseline_correct":
        baseline_correct,

    "tta_correct":
        tta_correct,

    "rescue":
        rescue,

    "harm":
        harm,

    "tta_confidence":
        tta_conf,

    "tta_margin":
        tta_margin,
})


for i, c in enumerate(
    "abcd"
):

    result_df[
        f"tta_prob_{c}"
    ] = tta_probs[:, i]


# question도 추가
result_df["question"] = (
    ordered_valid["question"]
    .tolist()
)


RESULT_CSV = (
    TTA_DIR
    / "tta_valid_result.csv"
)


result_df.to_csv(
    RESULT_CSV,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 14. RESCUE / HARM 출력
# ============================================================

print(
    "\n===== RESCUED ====="
)

print(
    result_df[
        result_df["rescue"]
    ][
        [
            "id",
            "answer",
            "baseline_pred",
            "tta_pred",
            "tta_confidence",
            "tta_margin",
            "question",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\n===== HARMED ====="
)

print(
    result_df[
        result_df["harm"]
    ][
        [
            "id",
            "answer",
            "baseline_pred",
            "tta_pred",
            "tta_confidence",
            "tta_margin",
            "question",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 15. SAVE TTA OUTPUT
# ============================================================

np.save(
    TTA_DIR
    / "valid_tta_probs.npy",

    tta_probs,
)


(
    TTA_DIR
    / "valid_tta_ids.json"
).write_text(

    json.dumps(
        valid_ids,
        ensure_ascii=False,
        indent=2,
    ),

    encoding="utf-8",
)


manifest = {

    "base_run":
        str(
            RUN_DIR
        ),

    "checkpoint":
        str(
            FINAL_CKPT
        ),

    "n":
        672,

    "choice_order":
        [
            "a",
            "b",
            "c",
            "d",
        ],

    "permutations":
        {
            str(k):
                v
            for k, v
            in PERMS.items()
        },

    "aggregation":
        "mean mapped softmax probabilities",

    "thinking":
        False,

    "image_size":
        1024,

    "baseline_accuracy":
        baseline_acc,

    "tta_accuracy":
        tta_acc,

    "delta":
        (
            tta_acc
            - baseline_acc
        ),

    "rescue":
        rescue_count,

    "harm":
        harm_count,

    "net":
        net,

    "individual_shift_accuracy":
        {
            str(k):
                float(v)
            for k, v
            in shift_accuracy.items()
        },
}


MANIFEST_PATH = (
    TTA_DIR
    / "tta_manifest.json"
)


MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
    ),

    encoding="utf-8",
)


print(
    "\n========================================"
)

print(
    "✅ H-6B 4-WAY TTA SUCCESS"
)

print(
    "========================================"
)

print(
    "baseline:",
    round(
        baseline_acc,
        6
    )
)

print(
    "TTA:",
    round(
        tta_acc,
        6
    )
)

print(
    "rescue/harm/net:",
    rescue_count,
    "/",
    harm_count,
    "/",
    net
)

print(
    "TTA probs:",
    TTA_DIR
    / "valid_tta_probs.npy"
)

print(
    "manifest:",
    MANIFEST_PATH
)

===== FINAL ADAPTER RESTORE =====
adapter tensors: 800
unexpected keys: 0
VRAM: 26.94 GB
✅ checkpoint-final restored

SHIFT 0 loaded: (672, 4)

TTA SHIFT 1 START
perm: [1, 2, 3, 0]
[SHIFT 1] 050/672 time=0.7m
[SHIFT 1] 100/672 time=1.3m
[SHIFT 1] 150/672 time=2.0m
[SHIFT 1] 200/672 time=2.7m
[SHIFT 1] 250/672 time=3.4m
[SHIFT 1] 300/672 time=4.1m
[SHIFT 1] 350/672 time=4.7m
[SHIFT 1] 400/672 time=5.4m
[SHIFT 1] 450/672 time=6.1m
[SHIFT 1] 500/672 time=6.8m
[SHIFT 1] 550/672 time=7.5m
[SHIFT 1] 600/672 time=8.1m
[SHIFT 1] 650/672 time=8.8m
[SHIFT 1] 672/672 time=9.1m
✅ SHIFT 1 SAVED

TTA SHIFT 2 START
perm: [2, 3, 0, 1]
[SHIFT 2] 050/672 time=0.7m
[SHIFT 2] 100/672 time=1.4m
[SHIFT 2] 150/672 time=2.0m
[SHIFT 2] 200/672 time=2.7m
[SHIFT 2] 250/672 time=3.4m
[SHIFT 2] 300/672 time=4.1m
[SHIFT 2] 350/672 time=4.7m
[SHIFT 2] 400/672 time=5.4m
[SHIFT 2] 450/672 time=6.1m
[SHIFT 2] 500/672 time=6.8m
[SHIFT 2] 550/672 time=7.5m
[SHIFT 2] 600/672 time=8.1m
[SHIFT 2] 650/672 time=8.8m
[SHIFT 2]

In [ ]:
# ============================================================
# [H-6C] OFFLINE TTA COMBINATION SWEEP
#
# 새 inference 없음
# 저장된 shift0~3 mapped probabilities만 사용
#
# 검사:
# - 모든 1/2/3/4 shift 조합 (총 15개)
# - arithmetic probability mean
# - geometric probability mean
# - 정확도 / correct count
# - answer별 정확도
# ============================================================

import json
import itertools
import numpy as np
import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------

RUN_DIR = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38/"
    "runs/full_20260921_100843"
)

TTA_DIR = RUN_DIR / "tta_valid"

VALID_IDS_PATH = RUN_DIR / "valid_ids.json"

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38"
)


# ------------------------------------------------------------
# LOAD SHIFT PROBS
# ------------------------------------------------------------

shift_probs = {}

for s in range(4):

    path = (
        TTA_DIR
        / f"shift{s}_mapped_probs.npy"
    )

    assert path.exists(), path

    shift_probs[s] = np.load(
        path
    ).astype(np.float64)

    assert shift_probs[s].shape == (672, 4)


print("✅ shift probs loaded")


# ------------------------------------------------------------
# VALID IDS / ANSWERS
# ------------------------------------------------------------

valid_ids = json.loads(
    VALID_IDS_PATH.read_text(
        encoding="utf-8"
    )
)

split_df = pd.read_csv(
    PROJECT_ROOT / "fixed_split_seed42.csv"
)

work_df = (
    train_df
    .reset_index(drop=True)
    .copy()
)

work_df["row_idx"] = np.arange(
    len(work_df)
)

merged = work_df.merge(
    split_df,
    on="row_idx",
    how="inner",
    validate="one_to_one"
)

valid_df = (
    merged[
        merged["split"]
        .astype(str)
        .str.lower()
        .isin(["valid", "val", "validation"])
    ]
    .copy()
)

ordered_valid = (
    valid_df
    .set_index("id")
    .loc[valid_ids]
    .reset_index()
)

answers = (
    ordered_valid["answer"]
    .astype(str)
    .str.strip()
    .str.lower()
    .to_numpy()
)

CHOICES = np.array(
    list("abcd")
)


# ------------------------------------------------------------
# 평가 함수
# ------------------------------------------------------------

def evaluate_probs(probs):

    pred = CHOICES[
        np.argmax(
            probs,
            axis=1
        )
    ]

    correct = (
        pred == answers
    )

    return (
        float(correct.mean()),
        int(correct.sum()),
        pred,
        correct,
    )


# ------------------------------------------------------------
# 모든 subset 검사
# ------------------------------------------------------------

rows = []


for k in range(1, 5):

    for combo in itertools.combinations(
        range(4),
        k
    ):

        arr = np.stack(
            [
                shift_probs[s]
                for s in combo
            ],
            axis=0
        )


        # ================================================
        # 1. Arithmetic probability mean
        # ================================================

        mean_probs = arr.mean(
            axis=0
        )

        acc, n_correct, pred, correct = (
            evaluate_probs(
                mean_probs
            )
        )


        per_answer = {}

        for a in "abcd":

            mask = (
                answers == a
            )

            per_answer[a] = float(
                correct[mask].mean()
            )


        rows.append({
            "method":
                "arithmetic",

            "combo":
                "+".join(
                    map(str, combo)
                ),

            "n_shifts":
                len(combo),

            "correct":
                n_correct,

            "accuracy":
                acc,

            "acc_a":
                per_answer["a"],

            "acc_b":
                per_answer["b"],

            "acc_c":
                per_answer["c"],

            "acc_d":
                per_answer["d"],
        })


        # ================================================
        # 2. Geometric probability mean
        #
        # log probability 평균
        # ================================================

        eps = 1e-12

        log_mean = np.mean(
            np.log(
                arr + eps
            ),
            axis=0
        )

        # softmax
        log_mean = (
            log_mean
            - log_mean.max(
                axis=1,
                keepdims=True
            )
        )

        geo_probs = np.exp(
            log_mean
        )

        geo_probs /= (
            geo_probs.sum(
                axis=1,
                keepdims=True
            )
        )


        (
            geo_acc,
            geo_n_correct,
            geo_pred,
            geo_correct,
        ) = evaluate_probs(
            geo_probs
        )


        geo_per_answer = {}

        for a in "abcd":

            mask = (
                answers == a
            )

            geo_per_answer[a] = float(
                geo_correct[mask].mean()
            )


        rows.append({
            "method":
                "geometric",

            "combo":
                "+".join(
                    map(str, combo)
                ),

            "n_shifts":
                len(combo),

            "correct":
                geo_n_correct,

            "accuracy":
                geo_acc,

            "acc_a":
                geo_per_answer["a"],

            "acc_b":
                geo_per_answer["b"],

            "acc_c":
                geo_per_answer["c"],

            "acc_d":
                geo_per_answer["d"],
        })


# ------------------------------------------------------------
# 결과 정렬
# ------------------------------------------------------------

result = pd.DataFrame(
    rows
)

result = result.sort_values(
    [
        "correct",
        "accuracy",
    ],
    ascending=False
).reset_index(
    drop=True
)


print(
    "\n========================================"
)

print(
    "TOP TTA COMBINATIONS"
)

print(
    "========================================"
)


print(
    result[
        [
            "method",
            "combo",
            "n_shifts",
            "correct",
            "accuracy",
            "acc_a",
            "acc_b",
            "acc_c",
            "acc_d",
        ]
    ]
    .head(20)
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# baseline과 shift3 비교
# ------------------------------------------------------------

print(
    "\n===== REFERENCE ====="
)

for s in range(4):

    acc, n, _, _ = (
        evaluate_probs(
            shift_probs[s]
        )
    )

    print(
        f"shift{s}: "
        f"{n}/672 "
        f"({acc:.6f})"
    )


# ------------------------------------------------------------
# 최고 조합 저장
# ------------------------------------------------------------

BEST = result.iloc[0]

print(
    "\n========================================"
)

print(
    "BEST"
)

print(
    "========================================"
)

print(
    "method :",
    BEST["method"]
)

print(
    "combo  :",
    BEST["combo"]
)

print(
    "correct:",
    int(
        BEST["correct"]
    )
)

print(
    "accuracy:",
    float(
        BEST["accuracy"]
    )
)


RESULT_PATH = (
    TTA_DIR
    / "tta_combination_sweep.csv"
)

result.to_csv(
    RESULT_PATH,
    index=False,
    encoding="utf-8-sig"
)


print(
    "\n✅ saved:",
    RESULT_PATH
)

✅ shift probs loaded

TOP TTA COMBINATIONS
    method   combo  n_shifts  correct  accuracy    acc_a    acc_b    acc_c    acc_d
arithmetic       3         1      641  0.953869 0.941176 0.975610 0.953488 0.945783
 geometric       3         1      641  0.953869 0.941176 0.975610 0.953488 0.945783
arithmetic     1+3         2      640  0.952381 0.941176 0.975610 0.953488 0.939759
 geometric     1+3         2      640  0.952381 0.941176 0.975610 0.953488 0.939759
arithmetic     0+3         2      639  0.950893 0.935294 0.981707 0.947674 0.939759
 geometric     0+3         2      639  0.950893 0.935294 0.981707 0.947674 0.939759
arithmetic     2+3         2      639  0.950893 0.935294 0.969512 0.959302 0.939759
 geometric     2+3         2      639  0.950893 0.935294 0.969512 0.959302 0.939759
arithmetic   0+1+3         3      638  0.949405 0.935294 0.981707 0.947674 0.933735
 geometric   0+1+3         3      638  0.949405 0.935294 0.981707 0.947674 0.933735
 geometric   0+2+3         3     

In [ ]:
# ============================================================
# [H-7] TEST INFERENCE - SHIFT3 BEST
#
# valid best:
#   shift3 perm = [3,0,1,2]
#   shown [a,b,c,d] = original [d,a,b,c]
#
# 결과:
# - test_shift3_raw_logits.npy
# - test_shift3_mapped_probs.npy
# - test_shift3_ids.json
# - test_shift3_manifest.json
# - submission_shift3.csv
# ============================================================

import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from peft.utils.save_and_load import (
    load_peft_weights,
    set_peft_model_state_dict,
)


# ============================================================
# 0. PATH
# ============================================================

RUN_DIR = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38/"
    "runs/full_20260921_100843"
)

FINAL_CKPT = (
    RUN_DIR
    / "checkpoint-final"
)

TEST_DIR = (
    RUN_DIR
    / "test_shift3"
)

TEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 1. FINAL ADAPTER 다시 강제 복구
# ============================================================

print("===== FINAL ADAPTER RESTORE =====")

final_adapter = load_peft_weights(
    str(FINAL_CKPT),
    device="cpu",
)

assert len(final_adapter) == 800


load_result = set_peft_model_state_dict(
    model,
    final_adapter,
    adapter_name="default",
)

unexpected = getattr(
    load_result,
    "unexpected_keys",
    [],
)

assert len(unexpected) == 0


model.eval()

gc.collect()
torch.cuda.empty_cache()


print(
    "adapter tensors:",
    len(final_adapter)
)

print(
    "VRAM:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        2
    ),
    "GB"
)

print("✅ checkpoint-final restored")


# ============================================================
# 2. TEST / SUBMISSION LOAD
# ============================================================

test_df = pd.read_csv(
    DATA_ROOT / "test.csv"
)

sample_submission = pd.read_csv(
    DATA_ROOT / "sample_submission.csv"
)


def normalize_path(p):

    p = str(p)

    if p.startswith("/"):
        return p

    return str(
        DATA_ROOT / p
    )


test_df["path"] = (
    test_df["path"]
    .map(normalize_path)
)


print("\n===== TEST =====")
print("test:", len(test_df))
print("submission:", len(sample_submission))
print("test columns:", test_df.columns.tolist())
print("submission columns:", sample_submission.columns.tolist())


assert len(test_df) == 6714
assert len(sample_submission) == 6714


# ID alignment 확인
assert (
    test_df["id"]
    .astype(str)
    .tolist()
    ==
    sample_submission["id"]
    .astype(str)
    .tolist()
)

print("✅ test/submission ID alignment OK")


# ============================================================
# 3. CHOICE TOKEN IDS
# ============================================================

tokenizer = processor.tokenizer

CHOICE_IDS = {}

for letter in "abcd":

    ids = tokenizer.encode(
        letter,
        add_special_tokens=False,
    )

    assert len(ids) == 1

    CHOICE_IDS[letter] = ids[0]


choice_id_tensor = torch.tensor(
    [
        CHOICE_IDS["a"],
        CHOICE_IDS["b"],
        CHOICE_IDS["c"],
        CHOICE_IDS["d"],
    ],
    device="cuda",
    dtype=torch.long,
)


print(
    "CHOICE IDS:",
    CHOICE_IDS
)


# ============================================================
# 4. SHIFT3
#
# shown:
#   a = original d
#   b = original a
#   c = original b
#   d = original c
# ============================================================

SHIFT = 3

PERM = [
    3,
    0,
    1,
    2,
]


def softmax_np(x):

    x = (
        x
        - np.max(
            x,
            axis=-1,
            keepdims=True,
        )
    )

    e = np.exp(x)

    return (
        e
        / e.sum(
            axis=-1,
            keepdims=True,
        )
    )


# ============================================================
# 5. TEST INPUT
# ============================================================

def build_test_shift3_input(row):

    image = load_eval_image(
        row["path"]
    )


    ref = get_reference(
        row["path"]
    )


    system_text = make_system_prompt(
        ref
    )


    original_options = [
        row["a"],
        row["b"],
        row["c"],
        row["d"],
    ]


    shown_options = [
        original_options[i]
        for i in PERM
    ]


    user_text = build_mc_prompt(
        row["question"],
        shown_options[0],
        shown_options[1],
        shown_options[2],
        shown_options[3],
    )


    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": system_text,
                }
            ],
        },

        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image,
                },

                {
                    "type": "text",
                    "text": user_text,
                },
            ],
        },
    ]


    return processor.apply_chat_template(
        messages,

        tokenize=True,
        add_generation_prompt=True,

        enable_thinking=False,

        return_dict=True,
        return_tensors="pt",
    )


# ============================================================
# 6. INFERENCE
# ============================================================

raw_logits_all = []
mapped_probs_all = []
test_ids = []


start_time = time.time()


print(
    "\n========================================"
)

print(
    "H-7 TEST SHIFT3 START"
)

print(
    "PERM [shown -> original]:",
    PERM
)

print(
    "========================================"
)


with torch.inference_mode():

    for i in range(
        len(test_df)
    ):

        row = test_df.iloc[i]


        inputs = build_test_shift3_input(
            row
        )


        inputs = inputs.to(
            "cuda"
        )


        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):

            outputs = model(
                **inputs
            )


        # --------------------------------------------
        # shown-space a/b/c/d logits
        # --------------------------------------------

        next_logits = (
            outputs.logits[
                0,
                -1
            ]
        )


        raw_logits = (
            next_logits[
                choice_id_tensor
            ]
            .float()
            .detach()
            .cpu()
            .numpy()
        )


        raw_logits_all.append(
            raw_logits
        )


        raw_probs = softmax_np(
            raw_logits[
                None,
                :
            ]
        )[0]


        # --------------------------------------------
        # original-space로 역매핑
        # --------------------------------------------

        mapped_probs = np.zeros(
            4,
            dtype=np.float32,
        )


        for shown_idx in range(4):

            original_idx = (
                PERM[
                    shown_idx
                ]
            )

            mapped_probs[
                original_idx
            ] = raw_probs[
                shown_idx
            ]


        mapped_probs_all.append(
            mapped_probs
        )


        test_ids.append(
            str(
                row["id"]
            )
        )


        del outputs
        del inputs


        if (
            (i + 1) % 100
            == 0
            or i + 1
            == len(test_df)
        ):

            elapsed = (
                time.time()
                - start_time
            )


            speed = (
                (i + 1)
                / elapsed
            )


            remaining_sec = (
                (
                    len(test_df)
                    - (i + 1)
                )
                / max(
                    speed,
                    1e-9
                )
            )


            print(
                f"[TEST] "
                f"{i + 1:04d}"
                f"/{len(test_df)} "
                f"elapsed="
                f"{elapsed/60:.1f}m "
                f"ETA="
                f"{remaining_sec/60:.1f}m"
            )


        if (
            (i + 1) % 100
            == 0
        ):

            gc.collect()
            torch.cuda.empty_cache()


# ============================================================
# 7. STACK
# ============================================================

raw_logits_all = np.stack(
    raw_logits_all,
    axis=0,
)

mapped_probs_all = np.stack(
    mapped_probs_all,
    axis=0,
)


assert raw_logits_all.shape == (
    6714,
    4
)

assert mapped_probs_all.shape == (
    6714,
    4
)

assert len(test_ids) == 6714


# ============================================================
# 8. PREDICTION
# ============================================================

CHOICES = np.array(
    list("abcd")
)


pred_idx = np.argmax(
    mapped_probs_all,
    axis=1,
)


pred = CHOICES[
    pred_idx
]


print(
    "\n===== TEST PRED DISTRIBUTION ====="
)

print(
    pd.Series(
        pred
    ).value_counts()
    .sort_index()
)


# ============================================================
# 9. SAVE RAW / MAPPED / IDS
# ============================================================

RAW_PATH = (
    TEST_DIR
    / "test_shift3_raw_logits.npy"
)

PROBS_PATH = (
    TEST_DIR
    / "test_shift3_mapped_probs.npy"
)

IDS_PATH = (
    TEST_DIR
    / "test_shift3_ids.json"
)


np.save(
    RAW_PATH,
    raw_logits_all
)

np.save(
    PROBS_PATH,
    mapped_probs_all
)


IDS_PATH.write_text(
    json.dumps(
        test_ids,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 10. SUBMISSION
# ============================================================

submission = (
    sample_submission.copy()
)


# id 제외한 예측 column 자동 탐색
prediction_columns = [
    c
    for c in submission.columns
    if c != "id"
]


assert len(
    prediction_columns
) == 1, (
    f"예측 column 확인 필요: "
    f"{submission.columns.tolist()}"
)


pred_col = (
    prediction_columns[0]
)


submission[
    pred_col
] = pred


SUBMISSION_PATH = (
    TEST_DIR
    / "submission_shift3.csv"
)


submission.to_csv(
    SUBMISSION_PATH,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 11. 최종 ID 검증
# ============================================================

assert (
    submission["id"]
    .astype(str)
    .tolist()
    == test_ids
)

assert (
    set(
        submission[
            pred_col
        ].astype(str)
    )
    <= set(
        "abcd"
    )
)


# ============================================================
# 12. MANIFEST
# ============================================================

manifest = {

    "model":
        MODEL_ID,

    "checkpoint":
        str(
            FINAL_CKPT
        ),

    "source_valid_accuracy":
        0.9538690476190477,

    "selection":
        "best single cyclic validation permutation",

    "shift":
        3,

    "shown_to_original_perm":
        PERM,

    "shown_options":
        {
            "a": "original_d",
            "b": "original_a",
            "c": "original_b",
            "d": "original_c",
        },

    "n":
        len(test_ids),

    "choice_order_after_mapping":
        [
            "a",
            "b",
            "c",
            "d",
        ],

    "choice_token_ids":
        CHOICE_IDS,

    "thinking":
        False,

    "image_size":
        1024,

    "ocr":
        True,

    "dino":
        True,

    "aggregation":
        None,

    "raw_logits":
        str(
            RAW_PATH
        ),

    "mapped_probs":
        str(
            PROBS_PATH
        ),

    "ids":
        str(
            IDS_PATH
        ),

    "submission":
        str(
            SUBMISSION_PATH
        ),
}


MANIFEST_PATH = (
    TEST_DIR
    / "test_shift3_manifest.json"
)


MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 13. DONE
# ============================================================

elapsed = (
    time.time()
    - start_time
)


print(
    "\n========================================"
)

print(
    "✅ H-7 TEST SHIFT3 SUCCESS"
)

print(
    "========================================"
)

print(
    "samples:",
    len(test_ids)
)

print(
    "elapsed:",
    round(
        elapsed / 60,
        2
    ),
    "min"
)

print(
    "raw logits:",
    RAW_PATH
)

print(
    "mapped probs:",
    PROBS_PATH
)

print(
    "ids:",
    IDS_PATH
)

print(
    "manifest:",
    MANIFEST_PATH
)

print(
    "SUBMISSION:",
    SUBMISSION_PATH
)

print(
    "\nsubmission preview:"
)

print(
    submission.head(10)
)

===== FINAL ADAPTER RESTORE =====
adapter tensors: 800
VRAM: 26.94 GB
✅ checkpoint-final restored

===== TEST =====
test: 6714
submission: 6714
test columns: ['id', 'path', 'question', 'a', 'b', 'c', 'd']
submission columns: ['id', 'answer']
✅ test/submission ID alignment OK
CHOICE IDS: {'a': 64, 'b': 65, 'c': 66, 'd': 67}

H-7 TEST SHIFT3 START
PERM [shown -> original]: [3, 0, 1, 2]
[TEST] 0100/6714 elapsed=1.3m ETA=87.5m
[TEST] 0200/6714 elapsed=2.7m ETA=86.7m
[TEST] 0300/6714 elapsed=4.0m ETA=85.5m
[TEST] 0400/6714 elapsed=5.3m ETA=84.3m
[TEST] 0500/6714 elapsed=6.7m ETA=83.0m
[TEST] 0600/6714 elapsed=8.0m ETA=81.7m
[TEST] 0700/6714 elapsed=9.4m ETA=80.3m
[TEST] 0800/6714 elapsed=10.7m ETA=79.0m
[TEST] 0900/6714 elapsed=12.0m ETA=77.7m
[TEST] 1000/6714 elapsed=13.4m ETA=76.4m
[TEST] 1100/6714 elapsed=14.7m ETA=75.0m
[TEST] 1200/6714 elapsed=16.0m ETA=73.7m
[TEST] 1300/6714 elapsed=17.4m ETA=72.4m
[TEST] 1400/6714 elapsed=18.7m ETA=71.0m
[TEST] 1500/6714 elapsed=20.0m ETA=69.7m
[TEST

In [ ]:
# ============================================================
# [L] FINAL TEST RESULT CHECK
# - H-7 산출물 존재 확인
# - shape / NaN / probability sum
# - ID 정렬
# - submission과 probs prediction 일치
# - manifest 확인
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path


RUN_DIR = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38/"
    "runs/full_20260921_100843"
)

TEST_DIR = RUN_DIR / "test_shift3"

RAW_PATH = TEST_DIR / "test_shift3_raw_logits.npy"
PROBS_PATH = TEST_DIR / "test_shift3_mapped_probs.npy"
IDS_PATH = TEST_DIR / "test_shift3_ids.json"
MANIFEST_PATH = TEST_DIR / "test_shift3_manifest.json"
SUB_PATH = TEST_DIR / "submission_shift3.csv"


print("========================================")
print("[L] FINAL TEST RESULT CHECK")
print("========================================")


# ------------------------------------------------------------
# 1. 파일 존재 확인
# ------------------------------------------------------------

paths = {
    "raw_logits": RAW_PATH,
    "mapped_probs": PROBS_PATH,
    "ids": IDS_PATH,
    "manifest": MANIFEST_PATH,
    "submission": SUB_PATH,
}

for name, path in paths.items():

    assert path.exists(), f"❌ missing: {path}"

    print(
        f"{name:15s}",
        "✅",
        f"{path.stat().st_size / 1024**2:.2f} MB"
    )


# ------------------------------------------------------------
# 2. LOAD
# ------------------------------------------------------------

raw = np.load(RAW_PATH)

probs = np.load(PROBS_PATH)

ids = json.loads(
    IDS_PATH.read_text(
        encoding="utf-8"
    )
)

manifest = json.loads(
    MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

submission = pd.read_csv(
    SUB_PATH
)

sample_submission = pd.read_csv(
    DATA_ROOT / "sample_submission.csv"
)


print("\n===== SHAPES =====")

print("raw        :", raw.shape)
print("probs      :", probs.shape)
print("ids        :", len(ids))
print("submission :", len(submission))


assert raw.shape == (6714, 4)
assert probs.shape == (6714, 4)

assert len(ids) == 6714
assert len(submission) == 6714


# ------------------------------------------------------------
# 3. NUMERIC CHECK
# ------------------------------------------------------------

assert np.isfinite(raw).all()
assert np.isfinite(probs).all()

assert (
    probs >= 0
).all()

assert (
    probs <= 1
).all()


row_sums = probs.sum(
    axis=1
)


print(
    "\nprob sum min/max:",
    row_sums.min(),
    row_sums.max()
)


assert np.allclose(
    row_sums,
    1.0,
    atol=1e-5
)


print("✅ probability numeric check")


# ------------------------------------------------------------
# 4. ID ALIGNMENT
# ------------------------------------------------------------

test_ids = (
    test_df["id"]
    .astype(str)
    .tolist()
)

sample_ids = (
    sample_submission["id"]
    .astype(str)
    .tolist()
)

submission_ids = (
    submission["id"]
    .astype(str)
    .tolist()
)


assert ids == test_ids
assert ids == sample_ids
assert ids == submission_ids


print("✅ ID alignment")


# ------------------------------------------------------------
# 5. PROBS -> PRED
# ------------------------------------------------------------

LETTERS = np.array(
    list("abcd")
)


pred_from_probs = LETTERS[
    probs.argmax(
        axis=1
    )
]


prediction_columns = [
    c
    for c in submission.columns
    if c != "id"
]


assert len(
    prediction_columns
) == 1


pred_col = prediction_columns[0]


pred_from_csv = (
    submission[pred_col]
    .astype(str)
    .str.strip()
    .str.lower()
    .to_numpy()
)


assert np.array_equal(
    pred_from_probs,
    pred_from_csv
)


assert set(
    pred_from_csv
) <= set(
    "abcd"
)


print(
    "✅ submission prediction == mapped probs argmax"
)


# ------------------------------------------------------------
# 6. DISTRIBUTION
# ------------------------------------------------------------

dist = (
    pd.Series(
        pred_from_csv
    )
    .value_counts()
    .sort_index()
)


print(
    "\n===== PRED DISTRIBUTION ====="
)

print(dist)


print(
    "\npercentage:"
)

print(
    (
        dist
        / len(submission)
        * 100
    ).round(2)
)


# ------------------------------------------------------------
# 7. MANIFEST
# ------------------------------------------------------------

print(
    "\n===== MANIFEST ====="
)

print(
    "model:",
    manifest.get("model")
)

print(
    "checkpoint:",
    manifest.get("checkpoint")
)

print(
    "n:",
    manifest.get("n")
)

print(
    "shift:",
    manifest.get("shift")
)

print(
    "perm:",
    manifest.get(
        "shown_to_original_perm"
    )
)

print(
    "thinking:",
    manifest.get("thinking")
)

print(
    "valid source acc:",
    manifest.get(
        "source_valid_accuracy"
    )
)


assert manifest["n"] == 6714

assert manifest["shift"] == 3

assert (
    manifest[
        "shown_to_original_perm"
    ]
    == [3, 0, 1, 2]
)

assert (
    manifest["thinking"]
    is False
)


# ------------------------------------------------------------
# 8. SAMPLE PREVIEW
# ------------------------------------------------------------

print(
    "\n===== SUBMISSION PREVIEW ====="
)

print(
    submission.head(10)
)


# ------------------------------------------------------------
# 9. FINAL SUMMARY SAVE
# ------------------------------------------------------------

summary = {

    "status":
        "PASS",

    "n":
        6714,

    "raw_shape":
        list(raw.shape),

    "probs_shape":
        list(probs.shape),

    "id_alignment":
        True,

    "probability_check":
        True,

    "submission_matches_probs":
        True,

    "shift":
        3,

    "perm":
        [3, 0, 1, 2],

    "prediction_distribution":
        {
            str(k): int(v)
            for k, v
            in dist.items()
        },

    "submission":
        str(SUB_PATH),
}


SUMMARY_PATH = (
    TEST_DIR
    / "L_final_check.json"
)


SUMMARY_PATH.write_text(
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "\n========================================"
)

print(
    "✅ [L] FINAL CHECK PASS"
)

print(
    "========================================"
)

print(
    "submission:",
    SUB_PATH
)

print(
    "check report:",
    SUMMARY_PATH
)

[L] FINAL TEST RESULT CHECK
raw_logits      ✅ 0.10 MB
mapped_probs    ✅ 0.10 MB
ids             ✅ 0.12 MB
manifest        ✅ 0.00 MB
submission      ✅ 0.10 MB

===== SHAPES =====
raw        : (6714, 4)
probs      : (6714, 4)
ids        : 6714
submission : 6714

prob sum min/max: 0.99999976 1.0000001
✅ probability numeric check
✅ ID alignment
✅ submission prediction == mapped probs argmax

===== PRED DISTRIBUTION =====
a    1702
b    1650
c    1632
d    1730
Name: count, dtype: int64

percentage:
a    25.35
b    24.58
c    24.31
d    25.77
Name: count, dtype: float64

===== MANIFEST =====
model: btbtyler09/Qwen3.8-27B-GPTQ-4bit
checkpoint: /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/checkpoint-final
n: 6714
shift: 3
perm: [3, 0, 1, 2]
thinking: False
valid source acc: 0.9538690476190477

===== SUBMISSION PREVIEW =====
              id answer
0  test_0001.jpg      b
1  test_0002.jpg      c
2  test_0003.jpg      d
3  test_0004.jpg      d
4  test_0005.jpg      c
5  tes

## [교체 I] Dataset / Collator — assistant 정답 토큰만 loss

In [ ]:
class HybridVQADataset(Dataset):
    def __init__(self, df, train=True):
        self.df = df.reset_index(drop=True)
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        image = Image.open(row["path"]).convert("RGB")
        image = resize_with_padding(image, IMAGE_SIZE)

        if self.train:
            image = augment_pil(image)

        ref = get_reference(row["path"])
        return {
            "image": image,
            "system": make_system_prompt(ref),
            "user": build_mc_prompt(
                str(row["question"]),
                str(row["a"]), str(row["b"]),
                str(row["c"]), str(row["d"]),
            ),
            "answer": str(row["answer"]).strip().lower() if self.train else None,
        }

class AssistantOnlyCollator:
    def __init__(self, processor, train=True):
        self.processor = processor
        self.train = train

    def __call__(self, batch):
        assert len(batch) == 1
        x = batch[0]

        base_messages = [
            {"role":"system", "content":[{"type":"text", "text":x["system"]}]},
            {"role":"user", "content":[
                {"type":"image", "image":x["image"]},
                {"type":"text", "text":x["user"]},
            ]},
        ]

        prompt_text = self.processor.apply_chat_template(
            base_messages, tokenize=False, add_generation_prompt=True
        )

        if not self.train:
            return self.processor(
                text=[prompt_text], images=[x["image"]],
                padding=False, return_tensors="pt"
            )

        full_messages = base_messages + [
            {"role":"assistant", "content":[{"type":"text", "text":x["answer"]}]}
        ]
        full_text = self.processor.apply_chat_template(
            full_messages, tokenize=False, add_generation_prompt=False
        )

        prompt_enc = self.processor(
            text=[prompt_text], images=[x["image"]],
            padding=False, return_tensors="pt"
        )
        full_enc = self.processor(
            text=[full_text], images=[x["image"]],
            padding=False, return_tensors="pt"
        )

        labels = full_enc["input_ids"].clone()
        prompt_len = prompt_enc["input_ids"].shape[1]
        labels[:, :prompt_len] = -100
        labels[full_enc["attention_mask"] == 0] = -100
        full_enc["labels"] = labels
        return full_enc

train_ds = HybridVQADataset(train_subset, train=True)
train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    collate_fn=AssistantOnlyCollator(processor, train=True),
    num_workers=0,
    pin_memory=True,
)

print("train batches:", len(train_loader))

## [교체 J] LoRA 학습 — Cosine + Gradient Accumulation + 중간 저장

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=0.01,
)

opt_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_opt_steps = EPOCHS * opt_steps_per_epoch
warmup_steps = int(total_opt_steps * WARMUP_RATIO)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_opt_steps,
)

STATE_PATH = RUN_DIR/"training_state.pt"
global_step = 0

def save_checkpoint(epoch, global_step):
    ckpt = RUN_DIR/"checkpoints"/f"step_{global_step:06d}"
    ckpt.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(ckpt)
    processor.save_pretrained(ckpt)

    torch.save({
        "epoch": epoch,
        "global_step": global_step,
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "adapter_dir": str(ckpt),
    }, STATE_PATH)

    print(f"\n💾 checkpoint saved: {ckpt}")

model.train()
optimizer.zero_grad(set_to_none=True)

for epoch in range(EPOCHS):
    running = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for micro_step, batch in enumerate(pbar, start=1):
        batch = {
            k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v
            for k, v in batch.items()
        }

        with torch.autocast("cuda", dtype=torch.bfloat16):
            out = model(**batch)
            loss = out.loss / GRAD_ACCUM

        loss.backward()
        running += loss.item()

        do_update = (micro_step % GRAD_ACCUM == 0) or (micro_step == len(train_loader))
        if do_update:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            pbar.set_postfix({
                "loss": f"{running:.4f}",
                "lr": f"{scheduler.get_last_lr()[0]:.2e}",
                "opt_step": global_step,
            })
            running = 0.0

            if global_step % SAVE_EVERY_OPT_STEPS == 0:
                save_checkpoint(epoch, global_step)

FINAL_ADAPTER = RUN_DIR/"adapter_final"
model.save_pretrained(FINAL_ADAPTER)
processor.save_pretrained(FINAL_ADAPTER)
save_checkpoint(EPOCHS, global_step)

print("✅ final adapter:", FINAL_ADAPTER)

## [교체 K] ABCD Softmax + 4회 선지 순환 TTA + `.npy` 중간 저장

In [ ]:
model.eval()

LETTERS = ["a", "b", "c", "d"]

def choice_token_ids(tokenizer):
    ids = []
    for x in LETTERS:
        t = tokenizer.encode(x, add_special_tokens=False)
        if len(t) != 1:
            raise ValueError(f"{x!r}가 단일 토큰이 아닙니다: {t}")
        ids.append(t[0])
    return ids

CHOICE_IDS = choice_token_ids(processor.tokenizer)
print("choice ids:", dict(zip(LETTERS, CHOICE_IDS)))

def rotate_options(row, r):
    opts = [str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])]
    r = r % 4
    return opts[r:] + opts[:r]

def rotated_probs_to_original(p_rot, r):
    p_orig = np.zeros(4, dtype=np.float32)
    for j in range(4):
        orig = (j + r) % 4
        p_orig[orig] = p_rot[j]
    return p_orig

@torch.no_grad()
def score_one(row, rotation=0):
    image = Image.open(row["path"]).convert("RGB")
    image = resize_with_padding(image, IMAGE_SIZE)

    ref = get_reference(row["path"])
    sys = make_system_prompt(ref)
    opts = rotate_options(row, rotation)
    user = build_mc_prompt(str(row["question"]), *opts)

    messages = [
        {"role":"system", "content":[{"type":"text", "text":sys}]},
        {"role":"user", "content":[
            {"type":"image", "image":image},
            {"type":"text", "text":user},
        ]},
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inp = processor(
        text=[text], images=[image],
        return_tensors="pt", padding=False
    ).to(device)

    with torch.autocast("cuda", dtype=torch.bfloat16):
        out = model(**inp)
        logits = out.logits[0, -1, CHOICE_IDS].float()
        probs = torch.softmax(logits, dim=-1).cpu().numpy().astype(np.float32)

    return rotated_probs_to_original(probs, rotation)

def score_df_tta(df, prefix, rotations=4, save_every=100):
    n = len(df)
    rot_arrays = [np.zeros((n,4), dtype=np.float32) for _ in range(rotations)]

    for r in range(rotations):
        partial = RUN_DIR/"probs"/f"{prefix}_rot{r}.npy"
        done_path = RUN_DIR/"probs"/f"{prefix}_rot{r}_done.npy"
        done = np.zeros(n, dtype=bool)

        if partial.exists() and done_path.exists():
            rot_arrays[r] = np.load(partial)
            done = np.load(done_path).astype(bool)
            print(f"resume {prefix} rot{r}: {done.sum()}/{n}")

        for i in tqdm(range(n), desc=f"{prefix} rot={r}"):
            if done[i]:
                continue

            rot_arrays[r][i] = score_one(df.iloc[i], rotation=r)
            done[i] = True

            if (i+1) % save_every == 0:
                np.save(partial, rot_arrays[r])
                np.save(done_path, done)

        np.save(partial, rot_arrays[r])
        np.save(done_path, done)

    avg = np.mean(rot_arrays, axis=0)
    np.save(RUN_DIR/"probs"/f"{prefix}_probs.npy", avg)
    return avg

P_valid = score_df_tta(valid_subset, "valid", rotations=4)
gold = np.array([LETTERS.index(str(x).strip().lower()) for x in valid_subset["answer"]])
pred = P_valid.argmax(1)
print("VALID ACC:", (pred == gold).mean())

# dev.csv는 answer1~answer5의 의미가 확정 GT라고 단정할 수 없으므로
# 학습/정확도 계산에는 쓰지 않고, 앙상블용 확률 파일만 저장합니다.
P_dev = score_df_tta(dev_df, "dev", rotations=4)
dev_pred = [LETTERS[i] for i in P_dev.argmax(1)]
dev_out = dev_df[["id"]].copy()
dev_out["prediction"] = dev_pred
dev_out.to_csv(RUN_DIR/"dev_predictions.csv", index=False)

P_test = score_df_tta(test_df, "test", rotations=4)

test_pred = [LETTERS[i] for i in P_test.argmax(1)]
submission = sample_submission.copy()
submission["answer"] = test_pred
submission.to_csv(RUN_DIR/"submission_tta.csv", index=False)

print("saved:", RUN_DIR/"probs"/"valid_probs.npy")
print("saved:", RUN_DIR/"probs"/"dev_probs.npy")
print("saved:", RUN_DIR/"probs"/"test_probs.npy")
print("saved:", RUN_DIR/"dev_predictions.csv")
print("saved:", RUN_DIR/"submission_tta.csv")

## [교체 L] 저장 결과 점검

In [ ]:
for p in sorted((RUN_DIR/"probs").glob("*")):
    print(p.name, f"{p.stat().st_size/1024**2:.2f} MB")

print("\ncheckpoints:")
for p in sorted((RUN_DIR/"checkpoints").glob("*")):
    print(p.name)

print("\nRUN_DIR:", RUN_DIR)

## [교체 M] Smart Routing Ensemble

In [ ]:
# ============================================================
# [M-0A] 새 세션 Drive 복구 + 핵심 파일 확인
# ============================================================

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38"
)

RUN_DIR = (
    PROJECT_ROOT
    / "runs"
    / "full_20260921_100843"
)

check_paths = [
    RUN_DIR / "checkpoint-final",
    RUN_DIR / "valid_logits.npy",
    RUN_DIR / "valid_ids.json",

    RUN_DIR
    / "tta_valid"
    / "shift3_mapped_probs.npy",

    RUN_DIR
    / "tta_valid"
    / "valid_tta_probs.npy",

    RUN_DIR
    / "test_shift3"
    / "test_shift3_mapped_probs.npy",

    RUN_DIR
    / "test_shift3"
    / "submission_shift3.csv",

    RUN_DIR
    / "test_shift3"
    / "L_final_check.json",
]

print("\nPROJECT ROOT:", PROJECT_ROOT)
print("exists:", PROJECT_ROOT.exists())

print("\n===== CORE FILE CHECK =====")

for p in check_paths:
    print(
        "✅" if p.exists() else "❌",
        p
    )

Mounted at /content/drive

PROJECT ROOT: /content/drive/MyDrive/ssafy_vqa_qwen38
exists: True

===== CORE FILE CHECK =====
✅ /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/checkpoint-final
✅ /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/valid_logits.npy
✅ /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/valid_ids.json
✅ /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/tta_valid/shift3_mapped_probs.npy
✅ /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/tta_valid/valid_tta_probs.npy
✅ /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/test_shift3/test_shift3_mapped_probs.npy
✅ /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/test_shift3/submission_shift3.csv
✅ /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/test_shift3/L_final_check.json


In [ ]:
# ============================================================
# [M-0] SMART ROUTING - EXPERT 후보 탐색
#
# Drive에 저장된 .npy 중
#   valid 후보 : (672, 4)
#   test 후보  : (6714, 4)
#
# 를 전부 찾아서 보여줌
# ============================================================

import numpy as np
from pathlib import Path


PROJECT_ROOT = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38"
)

CURRENT_RUN = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38/"
    "runs/full_20260921_100843"
)


print("PROJECT ROOT:", PROJECT_ROOT)


valid_candidates = []
test_candidates = []
other_4col = []


for path in PROJECT_ROOT.rglob("*.npy"):

    try:

        arr = np.load(
            path,
            mmap_mode="r"
        )

    except Exception as e:

        continue


    if (
        arr.ndim == 2
        and arr.shape[1] == 4
    ):

        info = {
            "path": str(path),
            "shape": arr.shape,
            "size_mb":
                round(
                    path.stat().st_size
                    / 1024**2,
                    3
                ),
        }


        if arr.shape[0] == 672:

            valid_candidates.append(
                info
            )

        elif arr.shape[0] == 6714:

            test_candidates.append(
                info
            )

        else:

            other_4col.append(
                info
            )


# ============================================================
# PRINT
# ============================================================

print(
    "\n========================================"
)

print(
    "VALID CANDIDATES (672,4)"
)

print(
    "========================================"
)

for i, x in enumerate(
    valid_candidates
):

    print(
        f"[V{i:02d}]",
        x["shape"],
        x["path"]
    )


print(
    "\n========================================"
)

print(
    "TEST CANDIDATES (6714,4)"
)

print(
    "========================================"
)

for i, x in enumerate(
    test_candidates
):

    print(
        f"[T{i:02d}]",
        x["shape"],
        x["path"]
    )


print(
    "\n========================================"
)

print(
    "SUMMARY"
)

print(
    "========================================"
)

print(
    "valid candidates:",
    len(valid_candidates)
)

print(
    "test candidates :",
    len(test_candidates)
)

print(
    "other 4-col     :",
    len(other_4col)
)

PROJECT ROOT: /content/drive/MyDrive/ssafy_vqa_qwen38

VALID CANDIDATES (672,4)
[V00] (672, 4) /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/valid_logits.npy
[V01] (672, 4) /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/tta_valid/shift0_mapped_probs.npy
[V02] (672, 4) /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/tta_valid/shift1_raw_logits.npy
[V03] (672, 4) /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/tta_valid/shift1_mapped_probs.npy
[V04] (672, 4) /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/tta_valid/shift2_raw_logits.npy
[V05] (672, 4) /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/tta_valid/shift2_mapped_probs.npy
[V06] (672, 4) /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/tta_valid/shift3_raw_logits.npy
[V07] (672, 4) /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/tta_valid/shift3_mapped_probs.npy
[V08] (672, 4) /content

In [ ]:
# ============================================================
# [M-1] SMART ROUTING SEARCH - VALID ONLY
#
# MAIN   = shift3 (641/672, 0.953869)
# EXPERT = shift0 / shift1 / shift2 각각 검사
#
# routing rule:
#   main_conf <= MAIN_LOW
#   expert_conf >= EXPERT_HIGH
#   main_pred != expert_pred
#
# 새 inference 없음
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path


# ============================================================
# 0. PATH
# ============================================================

RUN_DIR = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38/"
    "runs/full_20260921_100843"
)

TTA_DIR = RUN_DIR / "tta_valid"

ANALYSIS_PATH = (
    RUN_DIR
    / "analysis"
    / "valid_analysis.csv"
)


# ============================================================
# 1. LOAD
# ============================================================

analysis_df = pd.read_csv(
    ANALYSIS_PATH
)

answers = (
    analysis_df["answer"]
    .astype(str)
    .str.strip()
    .str.lower()
    .to_numpy()
)


LETTERS = np.array(
    list("abcd")
)


shift_probs = {}

for s in range(4):

    path = (
        TTA_DIR
        / f"shift{s}_mapped_probs.npy"
    )

    assert path.exists(), path

    shift_probs[s] = np.load(
        path
    ).astype(np.float64)

    assert shift_probs[s].shape == (
        672,
        4
    )


# ============================================================
# 2. MAIN = SHIFT3
# ============================================================

MAIN_SHIFT = 3

main = shift_probs[
    MAIN_SHIFT
]


main_pred_idx = np.argmax(
    main,
    axis=1
)

main_pred = LETTERS[
    main_pred_idx
]

main_conf = np.max(
    main,
    axis=1
)

main_correct = (
    main_pred == answers
)


print(
    "MAIN shift3:",
    int(main_correct.sum()),
    "/672",
    round(
        main_correct.mean(),
        6
    )
)


# ============================================================
# 3. THRESHOLD GRID
#
# confidence가 높은 모델이라
# 0.40~0.99 범위를 촘촘하게 검사
# ============================================================

MAIN_THRESHOLDS = np.round(
    np.arange(
        0.40,
        1.001,
        0.02
    ),
    2
)

EXPERT_THRESHOLDS = np.round(
    np.arange(
        0.40,
        1.001,
        0.02
    ),
    2
)


rows = []


# ============================================================
# 4. EXPERT 0 / 1 / 2 각각 검사
# ============================================================

for expert_shift in [
    0,
    1,
    2
]:

    expert = shift_probs[
        expert_shift
    ]

    expert_pred_idx = np.argmax(
        expert,
        axis=1
    )

    expert_pred = LETTERS[
        expert_pred_idx
    ]

    expert_conf = np.max(
        expert,
        axis=1
    )


    for main_low in MAIN_THRESHOLDS:

        for expert_high in EXPERT_THRESHOLDS:

            # --------------------------------------------
            # 원본 M 스타일 routing
            # --------------------------------------------

            route = (
                (main_conf <= main_low)
                &
                (expert_conf >= expert_high)
                &
                (main_pred_idx != expert_pred_idx)
            )


            route_count = int(
                route.sum()
            )


            if route_count == 0:
                continue


            final_pred = (
                main_pred.copy()
            )

            final_pred[
                route
            ] = expert_pred[
                route
            ]


            final_correct = (
                final_pred
                == answers
            )


            # --------------------------------------------
            # rescue / harm
            # --------------------------------------------

            rescue = (
                (~main_correct)
                &
                final_correct
            )

            harm = (
                main_correct
                &
                (~final_correct)
            )


            rescue_count = int(
                rescue.sum()
            )

            harm_count = int(
                harm.sum()
            )

            net = (
                rescue_count
                - harm_count
            )


            correct_count = int(
                final_correct.sum()
            )

            accuracy = float(
                final_correct.mean()
            )


            # routed된 문제 중
            # expert가 실제로 맞은 비율
            routed_correct = int(
                (
                    expert_pred[route]
                    == answers[route]
                ).sum()
            )

            route_precision = (
                routed_correct
                / route_count
            )


            rows.append({

                "expert_shift":
                    expert_shift,

                "main_low":
                    float(main_low),

                "expert_high":
                    float(expert_high),

                "routed":
                    route_count,

                "rescue":
                    rescue_count,

                "harm":
                    harm_count,

                "net":
                    net,

                "route_precision":
                    route_precision,

                "correct":
                    correct_count,

                "accuracy":
                    accuracy,
            })


# ============================================================
# 5. RESULT
# ============================================================

result = pd.DataFrame(
    rows
)


result = result.sort_values(
    [
        "correct",
        "net",
        "route_precision",
        "routed",
    ],
    ascending=[
        False,
        False,
        False,
        True,
    ]
).reset_index(
    drop=True
)


print(
    "\n========================================"
)

print(
    "TOP SMART ROUTING RULES"
)

print(
    "========================================"
)


print(
    result.head(30)
    .to_string(
        index=False
    )
)


# ============================================================
# 6. EXPERT별 최고 결과
# ============================================================

print(
    "\n===== BEST PER EXPERT ====="
)


for s in [
    0,
    1,
    2
]:

    sub = result[
        result["expert_shift"]
        == s
    ]

    if len(sub) == 0:
        continue

    best = sub.iloc[0]

    print(
        f"\nEXPERT shift{s}"
    )

    print(
        "correct :",
        int(
            best["correct"]
        )
    )

    print(
        "accuracy:",
        round(
            float(
                best["accuracy"]
            ),
            6
        )
    )

    print(
        "main_low:",
        best["main_low"]
    )

    print(
        "expert_high:",
        best["expert_high"]
    )

    print(
        "routed:",
        int(
            best["routed"]
        )
    )

    print(
        "rescue/harm/net:",
        int(
            best["rescue"]
        ),
        "/",
        int(
            best["harm"]
        ),
        "/",
        int(
            best["net"]
        )
    )


# ============================================================
# 7. SAVE
# ============================================================

OUT_PATH = (
    TTA_DIR
    / "smart_routing_search.csv"
)


result.to_csv(
    OUT_PATH,
    index=False,
    encoding="utf-8-sig"
)


print(
    "\n✅ saved:",
    OUT_PATH
)

MAIN shift3: 641 /672 0.953869

TOP SMART ROUTING RULES
 expert_shift  main_low  expert_high  routed  rescue  harm  net  route_precision  correct  accuracy
            0      0.72         0.54       6       3     2    1         0.500000      642  0.955357
            0      0.72         0.56       6       3     2    1         0.500000      642  0.955357
            0      0.74         0.56       6       3     2    1         0.500000      642  0.955357
            0      0.76         0.56       6       3     2    1         0.500000      642  0.955357
            0      0.78         0.56       6       3     2    1         0.500000      642  0.955357
            0      0.72         0.48       9       4     3    1         0.444444      642  0.955357
            0      0.72         0.50       9       4     3    1         0.444444      642  0.955357
            1      0.72         0.54       5       2     1    1         0.400000      642  0.955357
            1      0.72         0.56       5

In [ ]:
# ============================================================
# [M-2] ROUTING DECISION + FINAL SUBMISSION
#
# Smart Routing 결과:
# MAIN shift3 = 641/672
# BEST routing = 642/672
#
# gain = +1 only
# -> threshold overfitting 위험 대비 이득이 작으므로 미채택
# -> shift3 단독을 FINAL로 확정
# ============================================================

import json
import shutil
import pandas as pd
from pathlib import Path


RUN_DIR = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38/"
    "runs/full_20260921_100843"
)

TEST_DIR = (
    RUN_DIR
    / "test_shift3"
)

TTA_DIR = (
    RUN_DIR
    / "tta_valid"
)


SOURCE_SUBMISSION = (
    TEST_DIR
    / "submission_shift3.csv"
)

FINAL_SUBMISSION = (
    RUN_DIR
    / "submission_final.csv"
)

ROUTING_SEARCH = (
    TTA_DIR
    / "smart_routing_search.csv"
)


# ============================================================
# 1. 확인
# ============================================================

assert SOURCE_SUBMISSION.exists()
assert ROUTING_SEARCH.exists()


sub = pd.read_csv(
    SOURCE_SUBMISSION
)

assert len(sub) == 6714

assert sub.columns.tolist() == [
    "id",
    "answer",
]

assert set(
    sub["answer"]
) <= set("abcd")


# ============================================================
# 2. FINAL submission 저장
#
# BOM 없는 일반 UTF-8
# ============================================================

sub.to_csv(
    FINAL_SUBMISSION,
    index=False,
    encoding="utf-8",
)


# ============================================================
# 3. 다시 읽어서 무결성 검사
# ============================================================

check = pd.read_csv(
    FINAL_SUBMISSION
)

assert check.equals(sub)


# ============================================================
# 4. ROUTING DECISION 기록
# ============================================================

decision = {

    "final_method":
        "shift3_single",

    "shift":
        3,

    "shown_to_original_perm":
        [3, 0, 1, 2],

    "valid_main_correct":
        641,

    "valid_main_total":
        672,

    "valid_main_accuracy":
        0.9538690476190477,

    "best_routing_correct":
        642,

    "best_routing_accuracy":
        0.9553571428571429,

    "best_routing_gain_correct":
        1,

    "best_routing_gain_accuracy":
        0.0014880952380952,

    "best_rule": {
        "expert_shift": 0,
        "main_low": 0.72,
        "expert_high": 0.54,
        "routed": 6,
        "rescue": 3,
        "harm": 2,
        "net": 1,
        "route_precision": 0.5,
    },

    "routing_adopted":
        False,

    "reason":
        (
            "Validation gain was only +1/672 after threshold search. "
            "Expert is another permutation of the same model, and "
            "additional test inference cost is high relative to the "
            "small and potentially overfit gain."
        ),

    "final_submission":
        str(FINAL_SUBMISSION),
}


DECISION_PATH = (
    RUN_DIR
    / "final_decision.json"
)


DECISION_PATH.write_text(
    json.dumps(
        decision,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 5. 출력
# ============================================================

print(
    "========================================"
)

print(
    "✅ FINAL MODEL / SUBMISSION LOCKED"
)

print(
    "========================================"
)

print(
    "method : shift3 single"
)

print(
    "valid  : 641/672 = 0.953869"
)

print(
    "routing: rejected (+1 only)"
)

print(
    "rows   :",
    len(sub)
)

print(
    "\nFINAL SUBMISSION:"
)

print(
    FINAL_SUBMISSION
)

print(
    "\nDECISION:"
)

print(
    DECISION_PATH
)

print(
    "\npreview:"
)

print(
    sub.head(10)
)

✅ FINAL MODEL / SUBMISSION LOCKED
method : shift3 single
valid  : 641/672 = 0.953869
routing: rejected (+1 only)
rows   : 6714

FINAL SUBMISSION:
/content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/submission_final.csv

DECISION:
/content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/final_decision.json

preview:
              id answer
0  test_0001.jpg      b
1  test_0002.jpg      c
2  test_0003.jpg      d
3  test_0004.jpg      d
4  test_0005.jpg      c
5  test_0006.jpg      d
6  test_0007.jpg      b
7  test_0008.jpg      c
8  test_0009.jpg      c
9  test_0010.jpg      a


N

In [2]:
# ============================================================
# [N-1A-FIX] torchao 충돌 제거 + 기존 BF16 모델에 LoRA 재부착
#
# base_model / processor는 이미 메모리에 있음
# 다시 다운로드 / 다시 로드 안 함
# ============================================================

import sys
import gc
import torch
import subprocess
import importlib
import importlib.metadata

from peft import PeftModel


# ------------------------------------------------------------
# 1. 현재 torchao 확인
# ------------------------------------------------------------

try:
    print(
        "torchao before:",
        importlib.metadata.version("torchao")
    )
except importlib.metadata.PackageNotFoundError:
    print("torchao before: not installed")


# ------------------------------------------------------------
# 2. torchao 제거
#
# 우리는 torchao quantization을 전혀 사용하지 않으므로
# 오래된 0.10.0을 제거하는 게 가장 안전함
# ------------------------------------------------------------

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torchao",
    ],
    check=True,
)

importlib.invalidate_caches()


try:
    print(
        "torchao after:",
        importlib.metadata.version("torchao")
    )
except importlib.metadata.PackageNotFoundError:
    print(
        "✅ torchao removed"
    )


# ------------------------------------------------------------
# 3. 기존 BF16 모델 살아있는지 확인
# ------------------------------------------------------------

assert "base_model" in globals(), (
    "base_model이 없습니다. "
    "세션이 재시작된 경우에만 BF16 LOAD를 다시 해야 합니다."
)

assert "processor" in globals()

assert "FINAL_CKPT" in globals()


print(
    "\nBF16 base still loaded"
)

print(
    "allocated:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        2
    ),
    "GB"
)


# ------------------------------------------------------------
# 4. LoRA ATTACH 재시도
# ------------------------------------------------------------

print(
    "\n===== LoRA ATTACH RETRY ====="
)

model = PeftModel.from_pretrained(
    base_model,
    FINAL_CKPT,
    is_trainable=False,
)

model.eval()

if hasattr(model, "config"):
    model.config.use_cache = False


# ------------------------------------------------------------
# 5. LoRA 확인
# ------------------------------------------------------------

lora_modules = sum(
    1
    for m in model.modules()
    if hasattr(m, "lora_A")
)

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    "LoRA modules:",
    lora_modules
)

print(
    "trainable params:",
    trainable
)


assert lora_modules == 400, (
    f"LoRA module mismatch: {lora_modules}"
)

assert trainable == 0


# ------------------------------------------------------------
# 6. VRAM
# ------------------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

allocated = (
    torch.cuda.memory_allocated()
    / 1024**3
)

peak = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

free, total = torch.cuda.mem_get_info()


print(
    "\n========================================"
)

print(
    "✅ N-1A BF16 + LoRA LOAD SUCCESS"
)

print(
    "========================================"
)

print(
    "LoRA modules:",
    lora_modules
)

print(
    "allocated:",
    round(allocated, 2),
    "GB"
)

print(
    "peak:",
    round(peak, 2),
    "GB"
)

print(
    "GPU free:",
    round(
        free / 1024**3,
        2
    ),
    "GB"
)

torchao before: 0.10.0
✅ torchao removed

BF16 base still loaded
allocated: 50.96 GB

===== LoRA ATTACH RETRY =====
LoRA modules: 400
trainable params: 0

✅ N-1A BF16 + LoRA LOAD SUCCESS
LoRA modules: 400
allocated: 51.37 GB
peak: 51.78 GB
GPU free: 27.38 GB


In [7]:
# ============================================================
# [N-1B-FIX] /content/data 복구
# ============================================================

import zipfile
from pathlib import Path

DATA_ZIP = Path(
    "/content/drive/MyDrive/data.zip"
)

DATA_ROOT = Path(
    "/content/data"
)


print("DATA_ZIP exists:", DATA_ZIP.exists())
print("DATA_ROOT exists:", DATA_ROOT.exists())


if not (DATA_ROOT / "train.csv").exists():

    assert DATA_ZIP.exists(), (
        f"❌ data.zip 없음: {DATA_ZIP}"
    )

    DATA_ROOT.mkdir(
        parents=True,
        exist_ok=True
    )

    print(
        "📦 data.zip -> /content/data 압축 해제 시작"
    )

    with zipfile.ZipFile(
        DATA_ZIP,
        "r"
    ) as zf:

        zf.extractall(
            DATA_ROOT
        )

    print(
        "✅ data unzip 완료"
    )

else:

    print(
        "✅ /content/data 이미 존재"
    )


# 핵심 확인
check = [
    DATA_ROOT / "train.csv",
    DATA_ROOT / "train",
]

print(
    "\n===== CHECK ====="
)

for p in check:
    print(
        "✅" if p.exists() else "❌",
        p
    )

assert all(
    p.exists()
    for p in check
)

print(
    "\n✅ N-1B 데이터 복구 완료"
)

DATA_ZIP exists: True
DATA_ROOT exists: False
📦 data.zip -> /content/data 압축 해제 시작
✅ data unzip 완료

===== CHECK =====
✅ /content/data/train.csv
✅ /content/data/train

✅ N-1B 데이터 복구 완료


In [8]:
# ============================================================
# [N-1B] Qwen3.8-27B BF16 + 기존 LoRA
#         FIXED VALID 672 / SHIFT3
#
# 기존 기준:
#   GPTQ 4bit shift3 = 641 / 672 = 0.953869
#
# 이번 실험:
#   동일 LoRA
#   동일 OCR / DINO
#   동일 1024 image
#   동일 prompt
#   동일 shift3 [d,a,b,c]
#   단, BASE만 BF16
# ============================================================

import gc
import json
import re
import time

import numpy as np
import pandas as pd
import torch

from pathlib import Path
from PIL import Image, ImageOps


# ============================================================
# 0. PATH
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/ssafy_vqa_qwen38"
)

DATA_ROOT = Path(
    "/content/data"
)

CACHE_ROOT = (
    PROJECT_ROOT
    / "cache"
)

OLD_RUN = (
    PROJECT_ROOT
    / "runs"
    / "full_20260921_100843"
)

OUT_DIR = (
    OLD_RUN
    / "bf16_valid_shift3"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 현재 모델 확인
# ------------------------------------------------------------

assert "model" in globals()
assert "processor" in globals()

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print(
    "VRAM allocated:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        2
    ),
    "GB"
)


# ============================================================
# 1. DATA CHECK
# ============================================================

TRAIN_CSV = (
    DATA_ROOT
    / "train.csv"
)

SPLIT_PATH = (
    PROJECT_ROOT
    / "fixed_split_seed42.csv"
)

OCR_PATH = (
    CACHE_ROOT
    / "ocr_cache.json"
)

DINO_PATH = (
    CACHE_ROOT
    / "dino_cache.json"
)

OLD_VALID_IDS_PATH = (
    OLD_RUN
    / "valid_ids.json"
)

OLD_SHIFT3_PATH = (
    OLD_RUN
    / "tta_valid"
    / "shift3_mapped_probs.npy"
)


required = [
    TRAIN_CSV,
    SPLIT_PATH,
    OCR_PATH,
    DINO_PATH,
    OLD_VALID_IDS_PATH,
    OLD_SHIFT3_PATH,
]

print(
    "\n===== FILE CHECK ====="
)

for p in required:

    print(
        "✅" if p.exists() else "❌",
        p
    )

assert all(
    p.exists()
    for p in required
), (
    "필수 데이터가 없습니다. "
    "/content/data가 유지됐는지 확인 필요."
)


# ============================================================
# 2. TRAIN / VALID 복구
# ============================================================

train_df = pd.read_csv(
    TRAIN_CSV
)


def normalize_path(p):

    p = str(p)

    if p.startswith("/"):
        return p

    return str(
        DATA_ROOT / p
    )


train_df["path"] = (
    train_df["path"]
    .map(normalize_path)
)


split_df = pd.read_csv(
    SPLIT_PATH
)


work_df = (
    train_df
    .reset_index(drop=True)
    .copy()
)

work_df["row_idx"] = np.arange(
    len(work_df),
    dtype=np.int64
)


merged = work_df.merge(
    split_df,
    on="row_idx",
    how="inner",
    validate="one_to_one"
)


valid_df = (
    merged[
        merged["split"]
        .astype(str)
        .str.lower()
        .isin(
            [
                "valid",
                "val",
                "validation",
            ]
        )
    ]
    .copy()
)


valid_ids = json.loads(
    OLD_VALID_IDS_PATH
    .read_text(
        encoding="utf-8"
    )
)


valid_lookup = (
    valid_df
    .set_index("id")
)


ordered_valid = (
    valid_lookup
    .loc[valid_ids]
    .reset_index()
)


assert len(
    ordered_valid
) == 672


assert (
    ordered_valid["id"]
    .astype(str)
    .tolist()
    == valid_ids
)


answers = (
    ordered_valid["answer"]
    .astype(str)
    .str.strip()
    .str.lower()
    .to_numpy()
)


print(
    "\nvalid:",
    len(ordered_valid)
)


# ============================================================
# 3. CACHE
# ============================================================

with open(
    OCR_PATH,
    "r",
    encoding="utf-8"
) as f:

    ocr_cache = json.load(f)


with open(
    DINO_PATH,
    "r",
    encoding="utf-8"
) as f:

    dino_cache = json.load(f)


print(
    "OCR :",
    len(ocr_cache)
)

print(
    "DINO:",
    len(dino_cache)
)


# ============================================================
# 4. REFERENCE 함수
# ============================================================

def clean_ocr_text(text):

    text = str(
        text
    ).strip()

    if not re.search(
        r"[A-Za-z0-9가-힣]",
        text
    ):
        return None

    if len(text) <= 1:
        return None

    return text


def get_reference(path):

    key = str(
        path
    )


    # OCR
    ocr_items = (
        ocr_cache
        .get(
            key,
            []
        )
    )

    ocr_texts = []

    for item in ocr_items:

        # 저장 구조:
        # [bbox, text, confidence]
        try:

            text = (
                item[1]
            )

            conf = float(
                item[2]
            )

        except Exception:
            continue

        if conf < 0.40:
            continue

        text = clean_ocr_text(
            text
        )

        if text is None:
            continue

        if text not in ocr_texts:
            ocr_texts.append(
                text
            )

        if len(
            ocr_texts
        ) >= 15:
            break


    # DINO
    dino_items = (
        dino_cache
        .get(
            key,
            []
        )
    )

    obj_labels = []

    for item in dino_items:

        try:

            label = str(
                item["label"]
            ).strip()

            score = float(
                item["score"]
            )

        except Exception:

            try:

                label = str(
                    item[0]
                ).strip()

                score = float(
                    item[1]
                )

            except Exception:
                continue

        if score < 0.30:
            continue

        if (
            label
            and label
            not in obj_labels
        ):

            obj_labels.append(
                label
            )

        if len(
            obj_labels
        ) >= 12:
            break


    return {
        "ocr": ocr_texts,
        "objects": obj_labels,
    }


# ============================================================
# 5. PROMPT
# ============================================================

def make_system_prompt(ref):

    ocr_line = (
        ", ".join(
            ref["ocr"]
        )
        if ref["ocr"]
        else "없음"
    )

    obj_line = (
        ", ".join(
            ref["objects"]
        )
        if ref["objects"]
        else "없음"
    )

    return (
        "당신은 이미지 기반 4지선다 VQA 전문가입니다.\n"
        "이미지를 가장 중요한 근거로 사용하고 질문과 선택지를 비교하세요.\n\n"

        "[보조 참고자료]\n"

        f"- OCR 추출 텍스트: {ocr_line}\n"
        f"- 객체 탐지 후보: {obj_line}\n\n"

        "참고자료는 오검출될 수 있으므로 이미지와 충돌하면 무시하세요.\n"

        "최종 답은 반드시 a, b, c, d 중 소문자 한 글자로 선택하세요."
    )


def build_mc_prompt(
    question,
    a,
    b,
    c,
    d,
):

    return (
        f"질문: {question}\n"

        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"

        "정답:"
    )


# ============================================================
# 6. IMAGE
# ============================================================

IMAGE_SIZE = 1024


def letterbox_image(
    image,
    size
):

    image = image.convert(
        "RGB"
    )

    w, h = image.size

    scale = min(
        size / w,
        size / h
    )

    nw = max(
        1,
        int(
            round(
                w * scale
            )
        )
    )

    nh = max(
        1,
        int(
            round(
                h * scale
            )
        )
    )

    image = image.resize(
        (
            nw,
            nh
        ),
        Image.Resampling.LANCZOS
    )

    canvas = Image.new(
        "RGB",
        (
            size,
            size
        ),
        (
            0,
            0,
            0
        )
    )

    canvas.paste(
        image,
        (
            (size - nw) // 2,
            (size - nh) // 2
        )
    )

    return canvas


def load_eval_image(path):

    image = Image.open(
        path
    )

    return letterbox_image(
        image,
        IMAGE_SIZE
    )


# ============================================================
# 7. ABCD TOKEN IDS
# ============================================================

tokenizer = (
    processor.tokenizer
)


CHOICE_IDS = {}

for letter in "abcd":

    ids = tokenizer.encode(
        letter,
        add_special_tokens=False
    )

    assert len(
        ids
    ) == 1

    CHOICE_IDS[
        letter
    ] = ids[0]


print(
    "\nCHOICE IDS:",
    CHOICE_IDS
)


choice_id_tensor = torch.tensor(
    [
        CHOICE_IDS["a"],
        CHOICE_IDS["b"],
        CHOICE_IDS["c"],
        CHOICE_IDS["d"],
    ],
    device="cuda",
    dtype=torch.long
)


# ============================================================
# 8. SHIFT3
#
# shown:
#   a = original d
#   b = original a
#   c = original b
#   d = original c
# ============================================================

SHIFT = 3

PERM = [
    3,
    0,
    1,
    2
]


def np_softmax(x):

    x = (
        x
        - np.max(
            x,
            axis=-1,
            keepdims=True
        )
    )

    e = np.exp(
        x
    )

    return (
        e
        / e.sum(
            axis=-1,
            keepdims=True
        )
    )


def build_shift3_input(
    row
):

    image = load_eval_image(
        row["path"]
    )

    ref = get_reference(
        row["path"]
    )

    system_text = (
        make_system_prompt(
            ref
        )
    )


    original_options = [
        row["a"],
        row["b"],
        row["c"],
        row["d"],
    ]


    shown_options = [
        original_options[i]
        for i in PERM
    ]


    user_text = build_mc_prompt(
        row["question"],
        shown_options[0],
        shown_options[1],
        shown_options[2],
        shown_options[3],
    )


    messages = [

        {
            "role": "system",

            "content": [
                {
                    "type": "text",
                    "text": system_text,
                }
            ],
        },

        {
            "role": "user",

            "content": [

                {
                    "type": "image",
                    "image": image,
                },

                {
                    "type": "text",
                    "text": user_text,
                },

            ],
        },

    ]


    inputs = (
        processor
        .apply_chat_template(

            messages,

            tokenize=True,

            add_generation_prompt=True,

            enable_thinking=False,

            return_dict=True,

            return_tensors="pt",
        )
    )

    return inputs


# ============================================================
# 9. RESUME FILE
# ============================================================

PARTIAL_PATH = (
    OUT_DIR
    / "bf16_shift3_partial.npy"
)

DONE_PATH = (
    OUT_DIR
    / "bf16_shift3_done.json"
)


if (
    PARTIAL_PATH.exists()
    and DONE_PATH.exists()
):

    mapped_probs_all = np.load(
        PARTIAL_PATH
    ).tolist()

    done = json.loads(
        DONE_PATH
        .read_text(
            encoding="utf-8"
        )
    )

    start_idx = int(
        done["n_done"]
    )

    print(
        f"\n♻️ resume: {start_idx}/672"
    )

else:

    mapped_probs_all = []

    start_idx = 0


# ============================================================
# 10. BF16 INFERENCE
# ============================================================

model.eval()

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


start_time = time.time()


with torch.inference_mode():

    for i in range(
        start_idx,
        len(ordered_valid)
    ):

        row = (
            ordered_valid
            .iloc[i]
        )


        inputs = build_shift3_input(
            row
        )

        inputs = inputs.to(
            "cuda"
        )


        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16
        ):

            outputs = model(
                **inputs
            )


        next_logits = (
            outputs.logits[
                0,
                -1
            ]
        )


        raw_logits = (
            next_logits[
                choice_id_tensor
            ]
            .float()
            .cpu()
            .numpy()
        )


        raw_probs = np_softmax(
            raw_logits[
                None,
                :
            ]
        )[0]


        mapped_probs = np.zeros(
            4,
            dtype=np.float32
        )


        for shown_idx in range(
            4
        ):

            original_idx = (
                PERM[
                    shown_idx
                ]
            )

            mapped_probs[
                original_idx
            ] = raw_probs[
                shown_idx
            ]


        mapped_probs_all.append(
            mapped_probs
        )


        del outputs
        del inputs


        # ----------------------------------------
        # Drive 중간 저장
        # ----------------------------------------

        if (
            (i + 1) % 50 == 0
            or
            i + 1 == 672
        ):

            current = np.stack(
                mapped_probs_all,
                axis=0
            )

            np.save(
                PARTIAL_PATH,
                current
            )

            DONE_PATH.write_text(

                json.dumps(
                    {
                        "n_done":
                            i + 1
                    },
                    indent=2
                ),

                encoding="utf-8"
            )


            elapsed = (
                time.time()
                - start_time
            )


            print(
                f"[BF16 SHIFT3] "
                f"{i+1:03d}/672 "
                f"time={elapsed/60:.1f}m "
                f"peak={torch.cuda.max_memory_allocated()/1024**3:.1f}GB"
            )


            gc.collect()
            torch.cuda.empty_cache()


# ============================================================
# 11. RESULT
# ============================================================

bf16_probs = np.stack(
    mapped_probs_all,
    axis=0
)


assert bf16_probs.shape == (
    672,
    4
)


CHOICES = np.array(
    list(
        "abcd"
    )
)


bf16_pred = CHOICES[
    np.argmax(
        bf16_probs,
        axis=1
    )
]


bf16_correct = (
    bf16_pred
    == answers
)


bf16_n = int(
    bf16_correct.sum()
)

bf16_acc = float(
    bf16_correct.mean()
)


# ============================================================
# 12. OLD 4BIT SHIFT3 비교
# ============================================================

old_probs = np.load(
    OLD_SHIFT3_PATH
)

old_pred = CHOICES[
    np.argmax(
        old_probs,
        axis=1
    )
]

old_correct = (
    old_pred
    == answers
)


rescue = (
    (~old_correct)
    &
    bf16_correct
)

harm = (
    old_correct
    &
    (~bf16_correct)
)


changed = (
    old_pred
    != bf16_pred
)


# ============================================================
# 13. SAVE
# ============================================================

FINAL_PROB_PATH = (
    OUT_DIR
    / "bf16_shift3_mapped_probs.npy"
)

np.save(
    FINAL_PROB_PATH,
    bf16_probs
)


result_df = pd.DataFrame({

    "id":
        valid_ids,

    "answer":
        answers,

    "old_4bit_pred":
        old_pred,

    "bf16_pred":
        bf16_pred,

    "changed":
        changed,

    "rescue":
        rescue,

    "harm":
        harm,

})


result_df.to_csv(
    OUT_DIR
    / "bf16_vs_4bit_valid.csv",

    index=False,
    encoding="utf-8-sig"
)


summary = {

    "old_4bit_correct":
        int(
            old_correct.sum()
        ),

    "old_4bit_accuracy":
        float(
            old_correct.mean()
        ),

    "bf16_correct":
        bf16_n,

    "bf16_accuracy":
        bf16_acc,

    "changed_predictions":
        int(
            changed.sum()
        ),

    "rescue":
        int(
            rescue.sum()
        ),

    "harm":
        int(
            harm.sum()
        ),

    "net":
        int(
            rescue.sum()
            - harm.sum()
        ),

    "shift":
        3,

    "perm":
        PERM,

    "base":
        "Qwen/Qwen3.8-27B BF16",

    "adapter":
        str(
            FINAL_CKPT
        ),
}


(
    OUT_DIR
    / "bf16_valid_summary.json"
).write_text(

    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2
    ),

    encoding="utf-8"
)


# ============================================================
# 14. PRINT
# ============================================================

print(
    "\n========================================"
)

print(
    "BF16 vs GPTQ4 VALID RESULT"
)

print(
    "========================================"
)

print(
    "4bit :",
    int(
        old_correct.sum()
    ),
    "/672 =",
    round(
        float(
            old_correct.mean()
        ),
        6
    )
)

print(
    "BF16 :",
    bf16_n,
    "/672 =",
    round(
        bf16_acc,
        6
    )
)

print(
    "changed:",
    int(
        changed.sum()
    )
)

print(
    "rescue :",
    int(
        rescue.sum()
    )
)

print(
    "harm   :",
    int(
        harm.sum()
    )
)

print(
    "net    :",
    int(
        rescue.sum()
        - harm.sum()
    )
)

print(
    "\n✅ saved:",
    OUT_DIR
)

GPU: NVIDIA A100-SXM4-80GB
VRAM allocated: 51.37 GB

===== FILE CHECK =====
✅ /content/data/train.csv
✅ /content/drive/MyDrive/ssafy_vqa_qwen38/fixed_split_seed42.csv
✅ /content/drive/MyDrive/ssafy_vqa_qwen38/cache/ocr_cache.json
✅ /content/drive/MyDrive/ssafy_vqa_qwen38/cache/dino_cache.json
✅ /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/valid_ids.json
✅ /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/tta_valid/shift3_mapped_probs.npy

valid: 672
OCR : 16111
DINO: 16111

CHOICE IDS: {'a': 64, 'b': 65, 'c': 66, 'd': 67}


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[BF16 SHIFT3] 050/672 time=1.3m peak=52.8GB
[BF16 SHIFT3] 100/672 time=1.8m peak=52.8GB
[BF16 SHIFT3] 150/672 time=2.3m peak=52.8GB
[BF16 SHIFT3] 200/672 time=2.8m peak=52.8GB
[BF16 SHIFT3] 250/672 time=3.2m peak=52.8GB
[BF16 SHIFT3] 300/672 time=3.7m peak=52.8GB
[BF16 SHIFT3] 350/672 time=4.2m peak=52.8GB
[BF16 SHIFT3] 400/672 time=4.7m peak=52.8GB
[BF16 SHIFT3] 450/672 time=5.2m peak=52.8GB
[BF16 SHIFT3] 500/672 time=5.7m peak=52.8GB
[BF16 SHIFT3] 550/672 time=6.2m peak=52.8GB
[BF16 SHIFT3] 600/672 time=6.6m peak=52.8GB
[BF16 SHIFT3] 650/672 time=7.1m peak=52.8GB
[BF16 SHIFT3] 672/672 time=7.3m peak=52.8GB

BF16 vs GPTQ4 VALID RESULT
4bit : 641 /672 = 0.953869
BF16 : 640 /672 = 0.952381
changed: 11
rescue : 4
harm   : 5
net    : -1

✅ saved: /content/drive/MyDrive/ssafy_vqa_qwen38/runs/full_20260921_100843/bf16_valid_shift3


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


다른 모델들 확인

In [15]:
# ============================================================
# [AGNES-0 LOAD + KOREAN TEXT SANITY]
#
# 1. 현재 Q36 완전 unload
# 2. Agnes-AI/Agnes-3.0-Flash BF16 load
# 3. 한국어 텍스트 이해/MCQ sanity
#
# 아직 이미지 추론 X
# 아직 학습 X
# ============================================================

import gc
import time
import torch
import transformers

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)


AGNES_MODEL_ID = (
    "Agnes-AI/Agnes-3.0-Flash"
)


# ============================================================
# 0. ENV
# ============================================================

print(
    "Transformers:",
    transformers.__version__
)

print(
    "Torch:",
    torch.__version__
)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print(
    "GPU total:",
    round(
        torch.cuda.get_device_properties(0)
        .total_memory
        / 1024**3,
        2,
    ),
    "GB"
)


# ============================================================
# 1. Q36 / PEFT / TMP OBJECT UNLOAD
# ============================================================

print(
    "\nVRAM before unload:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        2,
    ),
    "GB"
)


REMOVE_NAMES = [
    # Q36 model
    "model",
    "core_model",
    "gptq_wrapper",

    # processor / tokenizer
    "processor",
    "tokenizer",

    # previous forward objects
    "outputs",
    "loss",
    "inputs",
    "labels",
    "batch",

    # temporary tensors
    "inp0",
    "inp1",
    "lab0",
    "lab1",

    # optimizer etc. 혹시 존재하면 제거
    "optimizer",
    "scheduler",
]


for name in REMOVE_NAMES:

    if name in globals():

        try:
            del globals()[name]
        except Exception:
            pass


gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


print(
    "VRAM after unload:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        2,
    ),
    "GB"
)

print(
    "reserved:",
    round(
        torch.cuda.memory_reserved()
        / 1024**3,
        2,
    ),
    "GB"
)


# Agnes 66GB급이라
# 기존 모델 reference가 남아 있으면 진행하지 않음
assert (
    torch.cuda.memory_allocated()
    / 1024**3
) < 3.0, (
    "❌ Q36 GPU 메모리가 충분히 해제되지 않았습니다."
)


# ============================================================
# 2. TOKENIZER
# ============================================================

print(
    "\n===== AGNES TOKENIZER LOAD ====="
)


agnes_tokenizer = (
    AutoTokenizer
    .from_pretrained(
        AGNES_MODEL_ID,

        trust_remote_code=True,
    )
)


print(
    "tokenizer:",
    type(agnes_tokenizer)
)

print(
    "vocab:",
    len(agnes_tokenizer)
)


# ============================================================
# 3. MODEL LOAD
#
# 공식 권장 방식:
# AutoModelForCausalLM
# BF16
# trust_remote_code=True
# device_map="auto"
# ============================================================

print(
    "\n===== AGNES MODEL LOAD ====="
)

load_start = time.time()


torch.cuda.reset_peak_memory_stats()


agnes_model = (
    AutoModelForCausalLM
    .from_pretrained(
        AGNES_MODEL_ID,

        torch_dtype=torch.bfloat16,

        device_map="auto",

        trust_remote_code=True,

        low_cpu_mem_usage=True,
    )
)


agnes_model.eval()


torch.cuda.synchronize()


print(
    "\n✅ Agnes loaded"
)

print(
    "model class:",
    type(agnes_model)
)

print(
    "load time:",
    round(
        (
            time.time()
            - load_start
        ) / 60,
        2,
    ),
    "min"
)


print(
    "allocated:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        2,
    ),
    "GB"
)

print(
    "reserved:",
    round(
        torch.cuda.memory_reserved()
        / 1024**3,
        2,
    ),
    "GB"
)

print(
    "peak:",
    round(
        torch.cuda.max_memory_allocated()
        / 1024**3,
        2,
    ),
    "GB"
)


# ============================================================
# 4. DEVICE MAP 확인
# ============================================================

if hasattr(
    agnes_model,
    "hf_device_map"
):

    device_values = list(
        agnes_model
        .hf_device_map
        .values()
    )

    print(
        "\n===== DEVICE MAP ====="
    )

    print(
        "unique devices:",
        sorted(
            set(
                str(x)
                for x in device_values
            )
        )
    )

    cpu_modules = sum(
        str(x) == "cpu"
        for x in device_values
    )

    disk_modules = sum(
        str(x) == "disk"
        for x in device_values
    )

    print(
        "CPU modules:",
        cpu_modules
    )

    print(
        "disk modules:",
        disk_modules
    )


# ============================================================
# 5. INPUT DEVICE
# ============================================================

input_device = next(
    p.device
    for p in agnes_model.parameters()
    if p.device.type != "meta"
)


print(
    "\ninput device:",
    input_device
)


# ============================================================
# 6. KOREAN TEXT SANITY
#
# 공식 카드에는 ko 지원 명시가 없으므로
# 실제로 3종류 확인
#
# 1) 한국 상식
# 2) 한국어 4지선다
# 3) 한국어 지식 질의
# ============================================================

KOREAN_TESTS = [

    (
        "CAPITAL",
        (
            "대한민국의 수도는 어디인가요? "
            "설명하지 말고 도시 이름만 한국어로 답하세요."
        ),
        "서울",
    ),

    (
        "MCQ",
        (
            "다음 중 과일은 무엇인가요?\n"
            "(a) 자동차\n"
            "(b) 사과\n"
            "(c) 의자\n"
            "(d) 신발\n\n"
            "정답은 반드시 a, b, c, d 중 "
            "소문자 한 글자로만 답하세요."
        ),
        "b",
    ),

    (
        "KOREAN_KNOWLEDGE",
        (
            "훈민정음을 창제한 조선의 왕은 누구인가요? "
            "설명하지 말고 왕의 이름만 한국어로 답하세요."
        ),
        "세종대왕",
    ),
]


# ============================================================
# 7. GENERATION
# ============================================================

print(
    "\n========================================"
)

print(
    "AGNES KOREAN TEXT SANITY"
)

print(
    "========================================"
)


for (
    test_name,
    prompt,
    expected,
) in KOREAN_TESTS:

    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]


    # Agnes 공식 chat template에서
    # thinking OFF 지원
    encoded = (
        agnes_tokenizer
        .apply_chat_template(
            messages,

            tokenize=True,

            add_generation_prompt=True,

            enable_thinking=False,

            return_tensors="pt",
        )
    )


    encoded = encoded.to(
        input_device
    )


    prompt_len = (
        encoded.shape[1]
    )


    torch.cuda.reset_peak_memory_stats()

    t0 = time.time()


    with torch.inference_mode():

        generated = (
            agnes_model.generate(
                encoded,

                max_new_tokens=32,

                do_sample=False,

                use_cache=True,
            )
        )


    torch.cuda.synchronize()


    new_tokens = (
        generated[
            0,
            prompt_len:
        ]
    )


    answer = (
        agnes_tokenizer
        .decode(
            new_tokens,

            skip_special_tokens=True,
        )
        .strip()
    )


    elapsed = (
        time.time()
        - t0
    )


    print(
        f"\n[{test_name}]"
    )

    print(
        "EXPECTED:",
        repr(expected)
    )

    print(
        "OUTPUT  :",
        repr(answer)
    )

    print(
        "time    :",
        round(
            elapsed,
            2
        ),
        "sec"
    )

    print(
        "peak    :",
        round(
            torch.cuda.max_memory_allocated()
            / 1024**3,
            2,
        ),
        "GB"
    )


    del encoded
    del generated
    del new_tokens


# ============================================================
# 8. END
# ============================================================

gc.collect()
torch.cuda.empty_cache()


print(
    "\n========================================"
)

print(
    "AGNES-0 COMPLETE"
)

print(
    "allocated:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        2,
    ),
    "GB"
)

print(
    "========================================"
)

Transformers: 5.17.0
Torch: 2.11.0+cu128
GPU: NVIDIA A100-SXM4-80GB
GPU total: 79.25 GB

VRAM before unload: 21.87 GB
VRAM after unload: 21.87 GB
reserved: 21.96 GB


AssertionError: ❌ Q36 GPU 메모리가 충분히 해제되지 않았습니다.

In [ ]:
# ============================================================
# [122B-2 FORCE GPTQ_TRITON LOAD]
#
# 목표:
# - Marlin 강제 회피
# - GPTQModel native GPTQ_TRITON으로 로드
# - A100 80GB 실제 VRAM 적재 가능 여부만 확인
#
# 아직 generation X
# ============================================================

import gc
import time
import torch

from collections import Counter
from gptqmodel import (
    GPTQModel,
    BACKEND,
)


MODEL_ID_122B = (
    "Qwen/Qwen3.5-122B-A10B-GPTQ-Int4"
)


# ============================================================
# 0. 이전 실패 흔적 CLEAN
# ============================================================

for name in [
    "model_122b",
    "gptq_122b",
    "wrapper_122b",
]:
    if name in globals():
        try:
            del globals()[name]
        except Exception:
            pass


gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


print(
    "VRAM before:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        2
    ),
    "GB"
)

free_b, total_b = torch.cuda.mem_get_info()

print(
    "free before:",
    round(
        free_b / 1024**3,
        2
    ),
    "GB"
)


# ============================================================
# 1. FORCE GPTQ_TRITON
# ============================================================

print(
    "\n===== 122B GPTQ_TRITON LOAD ====="
)

print(
    "backend:",
    BACKEND.GPTQ_TRITON
)


torch.cuda.reset_peak_memory_stats()

t0 = time.time()


gptq_122b = GPTQModel.load(
    MODEL_ID_122B,

    backend=BACKEND.GPTQ_TRITON,

    device="cuda:0",

    dtype=torch.bfloat16,

    trust_remote_code=True,
)


# ============================================================
# 2. 실제 HF MODEL 꺼내기
# ============================================================

model_122b = (
    gptq_122b.model

    if hasattr(
        gptq_122b,
        "model"
    )

    else gptq_122b
)


model_122b.eval()

torch.cuda.synchronize()


# ============================================================
# 3. RESULT
# ============================================================

print(
    "\n✅ 122B GPTQ_TRITON LOAD SUCCESS"
)

print(
    "wrapper:",
    type(gptq_122b)
)

print(
    "model:",
    type(model_122b)
)

print(
    "load time:",
    round(
        (
            time.time()
            - t0
        ) / 60,
        2
    ),
    "min"
)


allocated = (
    torch.cuda.memory_allocated()
    / 1024**3
)

reserved = (
    torch.cuda.memory_reserved()
    / 1024**3
)

peak = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)


free_b, total_b = torch.cuda.mem_get_info()


print(
    "\n===== VRAM ====="
)

print(
    "allocated:",
    round(
        allocated,
        2
    ),
    "GB"
)

print(
    "reserved:",
    round(
        reserved,
        2
    ),
    "GB"
)

print(
    "peak:",
    round(
        peak,
        2
    ),
    "GB"
)

print(
    "free:",
    round(
        free_b / 1024**3,
        2
    ),
    "GB"
)


# ============================================================
# 4. QUANT MODULE 구조 확인
# ============================================================

quant_classes = Counter()


for name, module in (
    model_122b.named_modules()
):

    cls = (
        module
        .__class__
        .__name__
    )

    if (
        "Linear" in cls
        or "Quant" in cls
    ):

        quant_classes[
            cls
        ] += 1


print(
    "\n===== LINEAR / QUANT CLASSES ====="
)

for cls, count in (
    quant_classes
    .most_common(20)
):

    print(
        f"{cls:30s}",
        count
    )


# ============================================================
# 5. 특히 routed expert / visual 확인
# ============================================================

triton_total = 0
triton_expert = 0
triton_visual = 0
triton_shared = 0


for name, module in (
    model_122b.named_modules()
):

    if (
        module
        .__class__
        .__name__
        == "TritonV2Linear"
    ):

        triton_total += 1

        if ".experts." in name:
            triton_expert += 1

        if ".visual." in name:
            triton_visual += 1

        if "shared_expert" in name:
            triton_shared += 1


print(
    "\n===== TRITON CHECK ====="
)

print(
    "TritonV2Linear total:",
    triton_total
)

print(
    "routed expert:",
    triton_expert
)

print(
    "visual:",
    triton_visual
)

print(
    "shared expert:",
    triton_shared
)


print(
    "\n========================================"
)

print(
    "122B-2 COMPLETE"
)

print(
    "========================================"
)

VRAM before: 0.0 GB
free before: 78.83 GB

===== 122B GPTQ_TRITON LOAD =====
backend: BACKEND.GPTQ_TRITON


HF: overriding trust_remote_code=True to False for `Qwen/Qwen3.5-122B-A10B-GPTQ-Int4` because model_type `qwen3_5_moe` is integrated in installed transformers as `Qwen3_5MoeForCausalLM`.


from_quantized: adapter: None


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 53 files:   0%|          | 0/53 [00:00<?, ?it/s]

INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: modules_to_not_convert.
INFO  QuantizeConfig: `format` is missing from the quantization configuration and is automatically inferred to FORMAT.GPTQ
INFO  QuantizeConfig: offload_to_disk_path auto set to temporary dir `/tmp/gptqmodel_yj4lno35`
INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]
INFO  Experts (before replacement) [model.language_model.layers.0.mlp.experts] (Qwen3_5MoeExperts):
Qwen3_5MoeExperts(
  (act_fn): SiLUActivation()
)
DEBUG Skipping unfuse for Conv1d: does not support @use_experts_implementation  
DEBUG Skipping unfuse for Conv1d: does not support @use_experts_implementation  
DEBUG Skipping unfuse for Conv1d: does not support @use_experts_implementation  
DEBUG Skipping unfuse for Conv1d: does not support @use_experts_implementation  
DEBUG Skipping unfuse for Conv1d: does not support @use_experts_implementation  
DEBUG Skipping unfu

## 추천 실행 순서

`A → B → C → D → E(OCR/DINO 1회) → F → G → H → I → J(선택 학습) → K → L → M`

### 저장되는 핵심 파일
- `config.json`
- `ocr_cache.json`
- `dino_cache.json`
- `checkpoints/step_xxxxxx/`
- `adapter_final/`
- `valid_rot0~3.npy`
- `valid_probs.npy`
- `test_rot0~3.npy`
- `test_probs.npy`
- `test_probs_routed.npy`
- `submission_tta.csv`
- `submission_routed.csv`

중간 확률과 checkpoint가 Drive에 남기 때문에 Colab 런타임이 끊겨도 재시작 비용을 줄일 수 있습니다.